# 18 — Equilíbrio das fontes: 12/4 versus 8/8

O 16 não aprovou o ajuste de supervisão de armários; o 17 não encontrou perfil
elegível entre os 49 limiares. Agora a hipótese é se a proporção de treino
favorece demais Home e reduz a retenção externa.

| Propriedade | Controle C12_4 | Candidato E8_8 |
| --- | --- | --- |
| Imagens por batch | 12 Home + 4 Open Images | 8 Home + 8 Open Images |
| Ponto inicial | Candidato 10 autenticado | Mesmo candidato 10 |
| Dados disponíveis | 1.820 Home + 2.000 OI históricos | Os mesmos IDs e bytes |
| Épocas curtas | 3 × 114 batches | 3 × 114 batches |
| Atualizações | 342 | 342 |
| Exposições Home / OI | 4.104 / 1.368 | 2.736 / 2.736 |
| Confiança para alertas | 0,25 | 0,25 |
| Rótulos Home | Originais, todas as classes | Originais, todas as classes |

Ambos são treinados nesta execução. A proporção muda a exposição e os IDs de cada
batch; os fluxos aleatórios por fonte têm a mesma semente e prefixos comuns.
O desfecho é **E8_8_e03 contra C12_4_e03**. E01/e02 são diagnósticos e não podem
substituir o candidato final. Mantêm-se os seis critérios numéricos históricos.

As validações já foram usadas no desenvolvimento, inclusive no 17: esta etapa é
**exploratória**, sem alegação de teste independente. TEST e reserva continuam fechados.


## 1. Executar como novo notebook no Kaggle

Importe este arquivo como **novo notebook**. Ative **Internet**, selecione **GPU T4**
e habilite o Secret existente **GDRIVE_RCLONE_CONFIG_B64**. Use
**Save Version → Save & Run All**. Não precisa alterar código nem preencher fichas.

**Estimativa em T4: 20–40 minutos no total.** Cada braço do 16 levou cerca de
6 min 16 s com treino, avaliações e backups. Downloads dos cinco lotes e dos ZIPs,
instalação e envio ao Drive tornam o total variável; com lentidão, reserve até
cerca de uma hora. CPU não tem essa estimativa e pode levar várias horas.


In [ ]:
%pip install -q ultralytics==8.4.150 pyyaml pandas pillow scipy matplotlib


In [ ]:
import os
os.environ['YOLO_OFFLINE']='true'
import sys,json,hashlib,shutil,platform,random,zipfile,gc,time
from pathlib import Path
from datetime import datetime,timezone
from collections import Counter
import numpy as np
import torch,torchvision,ultralytics,PIL,scipy,cv2
from ultralytics import YOLO,settings
from IPython.display import display,FileLink
SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(2);torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
cv2.setNumThreads(0)
DEVICE=0 if torch.cuda.is_available() else 'cpu'
if ultralytics.__version__!='8.4.150':raise RuntimeError('Reinicie a sessão e execute desde o início após instalar ultralytics==8.4.150.')
settings.update({k:False for k in ['sync','hub','platform','wandb','comet','mlflow','neptune','clearml','dvc','tensorboard'] if k in settings})
BASE_ID='86a8aef1a5b4a13c64304684'
RUN18='preparacao18_'+BASE_ID
INICIO_EXECUCAO=time.monotonic();ULTIMA_ETAPA=INICIO_EXECUCAO;TEMPOS=[]
DRIVE=None
PASTA_LOCAL=Path('/tmp/tcc_fontes_18')/RUN18
PASTA_BOOT=Path('/kaggle/working/resultados_tcc')/RUN18
PASTA_LOCAL.mkdir(parents=True,exist_ok=True);PASTA_BOOT.mkdir(parents=True,exist_ok=True)
PASTA_RESULTADOS=PASTA_BOOT;NOME_EXPERIMENTO=RUN18
AMBIENTE={'python':platform.python_version(),'torch':torch.__version__,'torchvision':torchvision.__version__,
    'ultralytics':ultralytics.__version__,'numpy':np.__version__,'pillow':PIL.__version__,
    'scipy':scipy.__version__,'opencv':cv2.__version__,'device':str(DEVICE),
    'gpu':torch.cuda.get_device_name(0) if DEVICE==0 else None,
    'cuda':torch.version.cuda,'cudnn':torch.backends.cudnn.version(),'threads':torch.get_num_threads()}
print('Dispositivo:',AMBIENTE['gpu'] or 'CPU')
print('Dois braços; 342 atualizações por braço; sem fichas manuais.')

print('Estimativa com T4: 20–40 min no total; depende dos downloads e backups.')


## 2. Conferir código, API e perda parcial

O notebook incorpora os módulos autenticados. O autoteste usa a perda YOLO real
com dados sintéticos: Home mantém cobertura completa e classes desconhecidas de
Open Images não recebem gradiente negativo. A versão da biblioteca é fixada.


In [ ]:
EXPECTED={'baseline': '4170653e0c0a604f85b6747e42b3fd2ff5ddb765ee4762915dfc9f3d33d322d9', 'candidato': '97040f91876d8835a67a7db287d3e1c5f129a4068e36dd4483d947f62b260c5f', 'politica': '1aa6b13cb68d8f30a2f388c3577a61a12dcac4c604581877091d3fbdc6042845', 'home_zip': 'ada7fbee37b83f673b1b76bc57643d1fe4648941e0e8a7b895282216aed47676', 'test_home': 'f48be0a2b3a93e59259b1ae0d3655028951bf270ef7f4e7916a9424530951a89', 'preparacao': '9865042654c51e83955c2aeb186a1b5e27e1028a5c56f1f163e2891abc0bec49', 'divisao_10': 'a5dd35ff484dda77e8988aefdc961f05d07ea409d1336cbe90c4a5420632bab9', 'manifesto_pacotes': '655094dea28755a7bddc817ae66660d06fab28ccb8f9d140d3ffebb07f03a0d4', 'pacote_11': 'a3f3fb500cef83fc4890f6bbec6725bbf5e080503fd0478af3dd84d6f5ef6e93', 'dataset_novo': '695b445e15c905642e2b276801b71132ced28e5eaee92d9fbcdd608f582f15a0', 'zip_novo': '5af596546e2ad734c7b663f4567de2087d7fb42b41527d4852affdb4e7d2b96c', 'codigo_11': {'diag11_core.py': '29e7c52fced44ab4898edfbc46477176587f290d2fbdacad4de5f595d4187670', 'diag11_data.py': 'c9f137d4dcc8c8f5fb156aaf25cddf6707d24e74567c05237332c2144e6e2f98', 'diag11_reports.py': '0068af8fd70b616cb642e9fb8a75d503cdc003f18fe79d64fd487bd7822d85e2'}, 'pacote_12': '1c3a6016099c2bb23aca29396286497c7973f00136c1afbb0bbc62e45aa20650', 'codigo_12': {'diag11_core.py': '29e7c52fced44ab4898edfbc46477176587f290d2fbdacad4de5f595d4187670', 'diag11_data.py': 'c9f137d4dcc8c8f5fb156aaf25cddf6707d24e74567c05237332c2144e6e2f98', 'diag11_reports.py': '0068af8fd70b616cb642e9fb8a75d503cdc003f18fe79d64fd487bd7822d85e2', 'exp10_partial.py': '86a36f93aa643853f1268dcc133bed7e7344dea269b9d1d58cf54095538fffae', 'exp12_protocol.py': '257d09b0b2aafe94da28bdcb59859778ffb31e25601db75d1058092b943b5759', 'exp12_data.py': '0a9de7e99d302a2690f18f51924f4dc80b32861c9d2149ccfe0572ec75499f30', 'exp12_training.py': '6c51c9965336b2f00bd9b282e2159c0b63f07c5c16859da78da2818bd4202fdc', 'exp12_reports.py': '120609247939189f895c01cdfd1411e4d2b416d1e293cb26851d63d04835b49a'}, 'B_e03': 'b683e1ef5fac4f8ddb4fad2f610e10dcb33bc658ea72593b8088652dbdd3c21c', 'regra_12_sha256': '04c6287c2c4adb8c43baf28c10acbaa1a599198b905d48808a8f5f67f39a4909', 'pacote_15': '266c0b4a0b8e7ac7fa940a009e0a009c22ab1cd17f403a4917d4ab3abe514487', 'pacote_16': 'cd9acf4eba29bb28dee20465ec2503b72cc65302f3d7c8377dd442455a4d16fa', 'pacote_17': '0ef3ce66a70216c641176a8c682e74b7a08c758f5d98ccb1ebd3db31943fde93', 'contexto_16': '44a814e0a728085209af51a3da632ef7568722115204a93e6a223ef49fde399b', 'pasta_16': 'resultados/controle16_44a814e0a728085209af51a3', 'pasta_17': 'resultados/limiares17_6f4aa31baf57035ce481a469', 'associador': '1da5dff21a6c6f6ec53d041247ce18d012008c4b26023b52c3fd2745f4c4105d'}
MODULOS_EMBUTIDOS={'conf14_data.py': '"""Entradas autenticadas do diagnóstico 14; extrai somente Home VAL."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport csv\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\nfrom PIL import Image\nimport diag11_core as core\n\nCOHORTS = (\'home_val_historica\', \'openimages_val_nova\')\nBOOLS = {\'True\':True, \'False\':False}\n\ndef rows(raw):\n    return list(csv.DictReader(io.StringIO(raw.decode(\'utf-8-sig\'))))\n\ndef parse_events(raw):\n    result = rows(raw)\n    for r in result:\n        r[\'classe_id\'] = int(r[\'classe_id\'])\n        for field in [\'bbox_ref\',\'bbox_pred\']:\n            r[field] = json.loads(r[field]) if r[field] else None\n        for field in [\'ha_alerta_ref\',\'ha_alerta_pred\']:\n            r[field] = BOOLS[r[field]]\n        r[\'confianca\'] = float(r[\'confianca\']) if r[\'confianca\'] else None\n    return result\n\ndef gt_key(r):\n    return (r[\'id\'],int(r[\'classe_id\']),tuple(round(float(v),6) for v in r[\'bbox_ref\']))\n\ndef gt_map(events):\n    result = {}\n    for r in events:\n        if r[\'bbox_ref\'] is None: continue\n        key = gt_key(r)\n        if key in result: raise ValueError(\'Objeto GT repetido nos registros.\')\n        result[key] = r\n    return result\n\ndef load_source12(path, expected, output=None):\n    if core.sha256_file(path)!=expected[\'pacote_12\']:\n        raise RuntimeError(\'O pacote 12 mudou em relação ao recebido.\')\n    bundle = {\'evaluations\':{}}\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist())!=len(set(z.namelist())):\n            raise RuntimeError(\'Pacote 12 com membros repetidos.\')\n        for name in z.namelist(): core.safe_member(name)\n        if z.testzip() is not None: raise RuntimeError(\'Falha de CRC no pacote 12.\')\n        cfg = json.loads(z.read(\'configuracao_experimento_12.json\'))\n        summary = json.loads(z.read(\'resumo_experimento_12.json\'))\n        if not summary[\'concluido\'] or summary[\'reserva_futura_inferida\'] or summary[\'test_home_inferido\'] or summary[\'test_oi20k_inferido\']:\n            raise RuntimeError(\'Execução 12 incompatível com o protocolo.\')\n        if cfg[\'expected\'][\'candidato\']!=expected[\'candidato\'] or cfg[\'expected\'][\'politica\']!=expected[\'politica\'] or summary[\'dataset_sha256\']!=expected[\'dataset_novo\']:\n            raise RuntimeError(\'Referências ou política diferentes do experimento 12.\')\n        if cfg[\'codigo_embutido_sha256\']!=expected[\'codigo_12\']:\n            raise RuntimeError(\'Código do experimento 12 diferente do recebido.\')\n        for name, digest in expected[\'codigo_12\'].items():\n            if hashlib.sha256(z.read(\'codigo/\'+name)).hexdigest()!=digest:\n                raise RuntimeError(\'Código alterado no pacote 12: \'+name)\n        rule_raw = z.read(\'regra_selecao_congelada.json\')\n        if hashlib.sha256(rule_raw).hexdigest()!=cfg[\'regra_sha256\']:\n            raise RuntimeError(\'A regra original do experimento 12 foi alterada.\')\n        selection = json.loads(z.read(\'selecao_checkpoints.json\'))\n        checkpoint = next(e for e in selection[\'assessed\'] if e[\'id\']==\'B_home_replay_e03\')\n        if checkpoint[\'weight_sha256\']!=expected[\'B_e03\']:\n            raise RuntimeError(\'Checkpoint B_e03 diferente do esperado.\')\n        for model,prefix in [(\'baseline\',\'referencias/baseline\'),(\'candidato_10\',\'referencias/candidato_10\'),(\'B_e03\',\'avaliacoes/B_home_replay_e03\')]:\n            bundle[\'evaluations\'][model] = {}\n            for cohort in COHORTS:\n                base=prefix+\'/\'+cohort+\'/\'\n                members={name:z.read(base+name) for name in [\'metricas.json\',\'alertas_por_imagem.csv\',\'avaliacao_objetos.csv\']}\n                metrics=json.loads(members[\'metricas.json\'])\n                images=rows(members[\'alertas_por_imagem.csv\'])\n                for r in images:\n                    for key in [\'tp_alerta\',\'fn_alerta\',\'fp_alerta\',\'alta_ref\',\'alta_sem_alerta\',\'tp_detector\',\'fn_detector\',\'fp_detector\',\'gt_avaliaveis\',\'gt_ignorados\',\'predicoes_ignoradas_conf_025\']:\n                        r[key]=int(r[key])\n                if len({r[\'id\'] for r in images})!=len(images) or len(images)!=metrics[\'imagens\']:\n                    raise RuntimeError(\'IDs ou quantidades inconsistentes nos registros 12.\')\n                for field, n in [(\'tp_alerta\',metrics[\'alertas_por_objeto\'][\'tp\']),(\'fn_alerta\',metrics[\'alertas_por_objeto\'][\'fn\']),(\'fp_alerta\',metrics[\'alertas_por_objeto\'][\'fp\']),(\'alta_ref\',metrics[\'alta_ref\']),(\'alta_sem_alerta\',metrics[\'alta_sem_alerta\'])]:\n                    if sum(r[field] for r in images)!=n: raise RuntimeError(\'Contagens inconsistentes no pacote 12.\')\n                events=parse_events(members[\'avaliacao_objetos.csv\'])\n                gt_map(events)\n                bundle[\'evaluations\'][model][cohort]={\'summary\':metrics,\'images\':images,\'events\':events}\n                if output is not None:\n                    dest=Path(output)/\'referencias_12\'/model/cohort\n                    dest.mkdir(parents=True,exist_ok=True)\n                    for name, raw in members.items(): (dest/name).write_bytes(raw)\n        for cohort in COHORTS:\n            target=gt_map(bundle[\'evaluations\'][\'B_e03\'][cohort][\'events\'])\n            for model in [\'baseline\',\'candidato_10\']:\n                source=gt_map(bundle[\'evaluations\'][model][cohort][\'events\'])\n                if source.keys()!=target.keys(): raise RuntimeError(\'Modelos do 12 usam referências GT diferentes.\')\n                if any(source[k][\'ha_alerta_ref\']!=target[k][\'ha_alerta_ref\'] or source[k][\'prioridade_ref\']!=target[k][\'prioridade_ref\'] for k in target):\n                    raise RuntimeError(\'Referências de prioridade diferentes entre modelos.\')\n        bundle.update(config=cfg,summary=summary,rule=json.loads(rule_raw),\n                      audit=json.loads(z.read(\'auditoria_dados.json\')))\n    return bundle\n\ndef extract_home_val(path, destination, expected_zip, expected_test, expected_policy):\n    if core.sha256_file(path)!=expected_zip: raise RuntimeError(\'ZIP Home alterado.\')\n    destination=Path(destination)\n    records=[];frozen=[]\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist())!=len(set(z.namelist())): raise RuntimeError(\'Membros repetidos no Home.\')\n        for name in z.namelist():core.safe_member(name)\n        manifest=json.loads(z.read(\'verificacao_preparacao.json\'))\n        if (manifest[\'test\'][\'sha256\']!=expected_test or manifest[\'test\'][\'imagens\']!=403 or\n            {int(k):v for k,v in manifest[\'classes\'].items()}!=dict(enumerate(core.NAMES))):\n            raise RuntimeError(\'Manifesto Home incompatível.\')\n        root=manifest[\'dataset_dir\']\n        items={r[\'arquivo\']:r for r in manifest[\'arquivos\']}\n        if len(items)!=len(manifest[\'arquivos\']): raise RuntimeError(\'Manifesto Home repetido.\')\n        train_hash=set();val_hash=set();all_hash=set();counts=Counter()\n        for name,item in items.items():\n            is_image=name.startswith(root+\'/images/\') and Path(name).suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.webp\',\'.bmp\'}\n            if not is_image:continue\n            all_hash.add(item[\'sha256\'])\n            if name.startswith(root+\'/images/train/raw_train/\'):\n                counts[\'train\']+=1;train_hash.add(item[\'sha256\'])\n            elif name.startswith(root+\'/images/val/raw_train/\'):\n                counts[\'val\']+=1;val_hash.add(item[\'sha256\'])\n        if counts!={\'train\':1824,\'val\':456} or train_hash&val_hash:\n            raise RuntimeError(\'A divisão atual de Home mudou ou tem bytes repetidos entre TRAIN e VAL.\')\n        for name in z.namelist():\n            if not(name.startswith(\'codigo/\') and name.endswith(\'.py\')):continue\n            raw=z.read(name)\n            if name in items and hashlib.sha256(raw).hexdigest()!=items[name][\'sha256\']:\n                raise RuntimeError(\'Código Home alterado.\')\n            out=destination/name;out.parent.mkdir(parents=True,exist_ok=True);out.write_bytes(raw)\n            frozen.append((out,hashlib.sha256(raw).hexdigest()))\n        for name,item in sorted(items.items()):\n            if not(name.startswith(root+\'/images/val/raw_train/\') and Path(name).suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.webp\',\'.bmp\'}):continue\n            label=name.replace(\'/images/\',\'/labels/\').rsplit(\'.\',1)[0]+\'.txt\'\n            if label not in items:raise RuntimeError(\'Label Home ausente.\')\n            paths={}\n            for field,member in [(\'image\',name),(\'label\',label)]:\n                raw=z.read(member)\n                if hashlib.sha256(raw).hexdigest()!=items[member][\'sha256\']:raise RuntimeError(\'Home alterado: \'+member)\n                out=destination/member;out.parent.mkdir(parents=True,exist_ok=True);out.write_bytes(raw)\n                paths[field]=out;frozen.append((out,items[member][\'sha256\']))\n            with Image.open(paths[\'image\']) as im:\n                width,height=im.size;im.verify()\n            records.append({\'id\':\'home:\'+name,\'name\':name.split(\'/raw_train/\')[-1],**paths,\n                \'role\':\'home_val\',\'coorte\':\'home_val_historica\',\'width\':width,\'height\':height,\n                \'known_mask\':core.ALL_KNOWN,\'sha256\':item[\'sha256\'],\'label_sha256\':items[label][\'sha256\'],\n                \'gt\':core.parse_yolo(paths[\'label\'].read_bytes(),width,height,source=label)})\n    if len(records)!=456 or core.sha256_file(destination/\'codigo/src/politica_alertas.py\')!=expected_policy:\n        raise RuntimeError(\'Quantidade Home VAL ou política V2 incompatível.\')\n    audit={\'home_train_manifesto\':1824,\'home_val_extraida\':456,\'home_train_extraida\':False,\n           \'home_test_extraido\':False,\'test_home_manifesto\':403,\'intersecao_bytes_train_val\':0,\n           \'historico_original_baseline_nao_comprovado_pela_separacao_atual\':True}\n    return records,all_hash,frozen,audit\n\ndef validate_cohorts(cohorts,bundle,reserved_ids):\n    if set(cohorts)!=set(COHORTS) or {k:len(v) for k,v in cohorts.items()}!={\'home_val_historica\':456,\'openimages_val_nova\':1000}:\n        raise ValueError(\'Coortes diferentes das duas validações congeladas.\')\n    for cohort,records in cohorts.items():\n        ids={r[\'id\'] for r in records}\n        if len(ids)!=len(records) or ids&set(reserved_ids):raise ValueError(\'ID repetido ou reservado na avaliação.\')\n        expected_role=\'home_val\' if cohort==\'home_val_historica\' else \'val_nova\'\n        if any(r[\'role\']!=expected_role for r in records):raise ValueError(\'Papel incompatível com a coorte de validação.\')\n        old=bundle[\'evaluations\'][\'B_e03\'][cohort]\n        if ids!={r[\'id\'] for r in old[\'images\']}:raise ValueError(\'IDs diferentes dos avaliados no 12.\')\n        actual={}\n        for r in records:\n            for g in r[\'gt\']:\n                if r[\'known_mask\']&(1<<g[\'classe_id\']):\n                    k=(r[\'id\'],g[\'classe_id\'],tuple(round(float(x),6) for x in g[\'bbox\']))\n                    if k in actual:raise ValueError(\'GT repetido na entrada atual.\')\n                    actual[k]=True\n        if actual.keys()!=gt_map(old[\'events\']).keys():raise ValueError(\'Caixas avaliáveis diferentes das referências originais.\')\n    return True\n', 'conf14_evaluation.py': '"""Varredura de confiança: uma inferência por imagem, política V2 fixa."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport gzip\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport diag11_core as core\nimport conf14_data as data\nfrom exp12_protocol import judge\n\nTHRESHOLDS=(.05,.10,.15,.20,.25,.30,.40)\nPREDICT_ARGS={\'conf\':.001,\'imgsz\':640,\'rect\':True,\'iou\':.70,\'max_det\':300,\'half\':False,\'batch\':1}\nVERSION=\'14-confianca-automatica-1.0\'\n\ndef point_name(conf):return \'conf_\'+f\'{conf:.2f}\'.replace(\'.\',\'\')\n\n\ndef cache_request(records,weight_sha,environment):\n    if not records or len({r[\'id\'] for r in records})!=len(records):raise ValueError(\'Cache exige IDs únicos.\')\n    if any(r[\'role\'] not in {\'home_val\',\'val_nova\'} for r in records):raise ValueError(\'Cache somente para validação.\')\n    signature=[{k:r[k] for k in [\'id\',\'sha256\',\'label_sha256\',\'width\',\'height\',\'known_mask\',\'role\']} for r in records]\n    return {\'versao\':VERSION,\'peso_sha256\':weight_sha,\'parametros\':PREDICT_ARGS,\n            \'ambiente\':environment,\'coorte\':records[0][\'coorte\'],\'imagens\':len(records),\n            \'entradas_sha256\':hashlib.sha256(core.canonical(signature).encode()).hexdigest()}\n\ndef read_cache(path,records):\n    expected={r[\'id\']:r for r in records};result={}\n    with gzip.open(path,\'rt\',encoding=\'utf-8\') as f:\n        for line in f:\n            row=json.loads(line);iid=row[\'id\']\n            if iid not in expected or iid in result:raise ValueError(\'Cache com ID desconhecido ou repetido.\')\n            r=expected[iid]\n            if row[\'image_sha256\']!=r[\'sha256\'] or row[\'label_sha256\']!=r[\'label_sha256\']:\n                raise ValueError(\'Cache pertence a outros bytes de imagem/label.\')\n            validate_predictions(row[\'predicoes\'],r);result[iid]=row[\'predicoes\']\n    if result.keys()!=expected.keys():raise ValueError(\'Cache incompleto.\')\n    return result\n\n\ndef event(record,kind,g,p,overlap,pred_low,conf):\n    row=core.record_event(record,\'B_e03\',kind,g,p,overlap,pred_low)\n    row[\'limiar_confianca\']=conf\n    row[\'melhor_confianca_mesma_classe\']=None\n    if kind==\'objeto_nao_detectado\':\n        flags=[v for v in row[\'indicios\'].split(\'|\') if v and v not in {\'deteccao_abaixo_da_confianca_025\',\'sobreposicao_com_previsao_de_outra_classe\',\'omissao_sem_indicio_automatico\'}]\n        matrix=core.iou_matrix([g],pred_low)[0]\n        same=[i for i,p0 in enumerate(pred_low) if p0[\'classe_id\']==g[\'classe_id\']]\n        if same:\n            best=max(same,key=lambda i:matrix[i]);row[\'melhor_confianca_mesma_classe\']=pred_low[best][\'confianca\']\n            if matrix[best]>=.5 and pred_low[best][\'confianca\']<conf:flags.append(\'deteccao_abaixo_do_limiar_avaliado\')\n        if any(matrix[i]>=.5 and p0[\'classe_id\']!=g[\'classe_id\'] and p0[\'confianca\']>=conf for i,p0 in enumerate(pred_low)):\n            flags.append(\'sobreposicao_com_previsao_de_outra_classe\')\n        row[\'indicios\']=\'|\'.join(flags or [\'omissao_sem_indicio_automatico\'])\n    return row\n\ndef analyse(record,pred_low,conf,policy,matcher):\n    if record[\'role\'] not in {\'home_val\',\'val_nova\'} or conf not in THRESHOLDS:\n        raise ValueError(\'Papel ou limiar fora do protocolo.\')\n    known=int(record[\'known_mask\'])\n    gt=core.with_policy([g for g in record[\'gt\'] if known&(1<<g[\'classe_id\'])],record[\'width\'],record[\'height\'],policy)\n    low=[p for p in pred_low if known&(1<<p[\'classe_id\'])]\n    all_pred=core.with_policy([p for p in pred_low if p[\'confianca\']>=conf],record[\'width\'],record[\'height\'],policy)\n    pred=[p for p in all_pred if known&(1<<p[\'classe_id\'])]\n    pairs,missing,extras=matcher(gt,pred)\n    events=[event(record,\'correspondencia\',gt[gi],pred[pi],float(iou),low,conf) for gi,pi,iou in pairs]\n    events += [event(record,\'objeto_nao_detectado\',gt[gi],None,None,low,conf) for gi in missing]\n    events += [event(record,\'previsao_nao_associada\',None,pred[pi],None,low,conf) for pi in extras]\n    messages=[policy.gerar_mensagem(p,\'medio\') for p in all_pred if core.alert_on(p)]\n    if any(not isinstance(s,str) or not s.strip() for s in messages):raise RuntimeError(\'Alerta sem mensagem.\')\n    im={\'coorte\':record[\'coorte\'],\'imagem\':record[\'name\'],\'id\':record[\'id\'],\'modelo\':\'B_e03\',\'limiar_confianca\':conf,\n        \'tp_alerta\':sum(e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in events),\n        \'fn_alerta\':sum(e[\'ha_alerta_ref\'] and not e[\'ha_alerta_pred\'] for e in events),\n        \'fp_alerta\':sum(not e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in events),\n        \'alta_ref\':sum(g[\'prioridade\']==\'alto\' for g in gt),\n        \'alta_sem_alerta\':sum(e[\'prioridade_ref\']==\'alto\' and not e[\'ha_alerta_pred\'] for e in events),\n        \'tp_detector\':len(pairs),\'fn_detector\':len(missing),\'fp_detector\':len(extras),\n        \'gt_avaliaveis\':len(gt),\'gt_ignorados\':len(record[\'gt\'])-len(gt),\n        \'predicoes_ignoradas_conf\':len(all_pred)-len(pred),\n        \'cobertura_completa_alertas\':known&core.ALERT_MASK==core.ALERT_MASK,\n        \'ha_alerta_ref\':any(core.alert_on(g) for g in gt),\'ha_alerta_pred\':any(core.alert_on(p) for p in pred),\n        \'mensagem_alerta\':\'\\n\'.join(messages)}\n    return im,events\n\ndef summarise(images,events,conf):\n    if not images:raise ValueError(\'Validação vazia.\')\n    total=lambda k:sum(r[k] for r in images)\n    complete=[r for r in images if r[\'cobertura_completa_alertas\']]\n    summary={\'modelo\':\'B_e03\',\'coorte\':images[0][\'coorte\'],\'imagens\':len(images),\'conf_alertas\':conf,\n        \'alertas_por_objeto\':core.alert_metrics(total(\'tp_alerta\'),total(\'fn_alerta\'),total(\'fp_alerta\')),\n        \'alta_ref\':total(\'alta_ref\'),\'alta_sem_alerta\':total(\'alta_sem_alerta\'),\n        \'taxa_alta_sem_alerta\':core.ratio(total(\'alta_sem_alerta\'),total(\'alta_ref\')),\n        \'detector_no_limiar\':core.alert_metrics(total(\'tp_detector\'),total(\'fn_detector\'),total(\'fp_detector\')),\n        \'gt_ignorados_sem_cobertura\':total(\'gt_ignorados\'),\'predicoes_ignoradas_conf\':total(\'predicoes_ignoradas_conf\'),\n        \'imagens_com_cobertura_completa_alertas\':len(complete),\n        \'alertas_por_imagem_cobertura_completa\':core.alert_metrics(\n            sum(r[\'ha_alerta_ref\'] and r[\'ha_alerta_pred\'] for r in complete),\n            sum(r[\'ha_alerta_ref\'] and not r[\'ha_alerta_pred\'] for r in complete),\n            sum(not r[\'ha_alerta_ref\'] and r[\'ha_alerta_pred\'] for r in complete)),\n        \'referencia\':\'Concordância com política V2 nas caixas; não valida risco físico.\',\n        \'nao_e_openimages_challenge\':True,\'ap_recalculada\':False}\n    classes=[]\n    for cid,name in enumerate(core.NAMES):\n        part=[e for e in events if e[\'classe_id\']==cid];n=sum(e[\'bbox_ref\']!=\'\' for e in part)\n        classes.append({\'classe_id\':cid,\'classe\':name,\'gt_avaliaveis\':n,\'suporte_reduzido\':n<30,\'conf_alertas\':conf,\n            **core.alert_metrics(sum(e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in part),\n                                 sum(e[\'ha_alerta_ref\'] and not e[\'ha_alerta_pred\'] for e in part),\n                                 sum(not e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in part)),\n            \'alta_ref\':sum(e[\'prioridade_ref\']==\'alto\' for e in part),\n            \'alta_sem_alerta\':sum(e[\'prioridade_ref\']==\'alto\' and not e[\'ha_alerta_pred\'] for e in part)})\n    return summary,classes\n\ndef evaluate_cache(records,cache,conf,policy,matcher,output=None):\n    if cache.keys()!={r[\'id\'] for r in records}:raise ValueError(\'Cache não corresponde aos IDs da validação.\')\n    images=[];events=[]\n    for r in records:\n        im,ev=analyse(r,cache[r[\'id\']],conf,policy,matcher);images.append(im);events.extend(ev)\n    summary,classes=summarise(images,events,conf)\n    if output is not None:\n        output=Path(output);output.mkdir(parents=True,exist_ok=True)\n        core.save_json(output/\'metricas.json\',summary)\n        core.save_csv(output/\'metricas_por_classe.csv\',classes)\n        core.save_csv(output/\'alertas_por_imagem.csv\',images)\n        core.save_csv(output/\'avaliacao_objetos.csv\',events)\n    return {\'summary\':summary,\'classes\':classes,\'images\':images,\'events\':events}\n\ndef parse_current_events(events):\n    result=[]\n    for r in events:\n        v=dict(r)\n        for key in [\'bbox_ref\',\'bbox_pred\']:v[key]=json.loads(v[key]) if v[key] else None\n        result.append(v)\n    return result\n\ndef check_reproduction(result,original,cohort):\n    old={r[\'id\']:r for r in original[\'images\']};new={r[\'id\']:r for r in result[\'images\']}\n    if old.keys()!=new.keys():raise ValueError(\'Reprodução exige os mesmos IDs.\')\n    rows=[]\n    fields=[\'tp_alerta\',\'fn_alerta\',\'fp_alerta\',\'alta_ref\',\'alta_sem_alerta\',\'tp_detector\',\'fn_detector\',\'fp_detector\',\'gt_avaliaveis\',\'gt_ignorados\',\'predicoes_ignoradas_conf_025\']\n    for iid in sorted(old):\n        for key in fields:\n            key_new=\'predicoes_ignoradas_conf\' if key==\'predicoes_ignoradas_conf_025\' else key\n            a,b=old[iid][key],new[iid][key_new]\n            rows.append({\'coorte\':cohort,\'id\':iid,\'metrica\':key,\'notebook12\':a,\'notebook14\':b,\'igual\':a==b})\n    old_gt=data.gt_map(original[\'events\']);new_gt=data.gt_map(parse_current_events(result[\'events\']))\n    if old_gt.keys()!=new_gt.keys():raise ValueError(\'Referências GT mudaram na reprodução.\')\n    object_rows=[]\n    for key,a in old_gt.items():\n        b=new_gt[key]\n        fields_gt=[\'evento\',\'prioridade_ref\',\'prioridade_pred\',\'ha_alerta_ref\',\'ha_alerta_pred\']\n        diffs=[field for field in fields_gt if a[field]!=b[field]]\n        object_rows.append({\'coorte\':cohort,\'id\':key[0],\'classe\':a[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                            \'campos_diferentes\':\'|\'.join(diffs),\'igual\':not diffs})\n    return {\'coorte\':cohort,\'checagens_imagem\':len(rows),\'divergencias_imagem\':sum(not r[\'igual\'] for r in rows),\n            \'objetos_GT\':len(object_rows),\'divergencias_objetos\':sum(not r[\'igual\'] for r in object_rows),\n            \'reproduziu\':all(r[\'igual\'] for r in rows+object_rows)},rows,object_rows\n\ndef select_threshold(results,rule,reproduction_ok):\n    assessed=[]\n    for conf in THRESHOLDS:\n        home=results[\'home_val_historica\'][conf][\'summary\'];ext=results[\'openimages_val_nova\'][conf][\'summary\']\n        supported=all(s[\'alertas_por_objeto\'][k] is not None and math.isfinite(s[\'alertas_por_objeto\'][k])\n                      for s in [home,ext] for k in [\'precision\',\'recall\',\'f1\'])\n        if supported:\n            checks={**judge(home,ext,rule),\'metricas_com_suporte\':True}\n        else:\n            # Precisão sem nenhum alerta é indefinida; não vira zero ou aprovação.\n            checks={k:None for k in [\'home_precision_superior\',\'home_recall_preservado\',\'home_altas_preservadas\',\n                                    \'externa_precision_preservada\',\'externa_recall_preservado\',\'externa_altas_preservadas\']}\n            checks.update(elegivel=False,retencao_externa=False,metricas_com_suporte=False)\n        assessed.append({\'conf\':conf,\'home\':home,\'external\':ext,\'checks\':checks,\n                         \'comparavel_historico\':reproduction_ok,\'elegivel\':checks[\'elegivel\'] and reproduction_ok})\n    def ranking(r):\n        h=r[\'home\'][\'alertas_por_objeto\'];e=r[\'external\'][\'alertas_por_objeto\']\n        return (-h[\'precision\'],-h[\'recall\'],r[\'home\'][\'alta_sem_alerta\'],-e[\'f1\'],abs(r[\'conf\']-.25),-r[\'conf\'])\n    good=[r for r in assessed if r[\'elegivel\']]\n    chosen=min(good,key=ranking) if good else None\n    status=\'limiar_candidato_em_validacao\' if chosen else \'nenhum_limiar_elegivel\'\n    if not reproduction_ok:status=\'reproducao_divergente_sem_sugestao_de_limiar\'\n    return {\'status\':status,\'limiar_sugerido\':chosen[\'conf\'] if chosen else None,\n            \'comparabilidade_confirmada\':bool(reproduction_ok),\'assessed\':assessed,\n            \'limiar_galeria\':chosen[\'conf\'] if chosen else .10,\n            \'criterio_galeria_sem_elegivel\':\'0,10 fixado antes da inferência; somente diagnóstico.\',\n            \'modelo_em_uso_continua_baseline\':True,\'substituicao_automatica\':False}\n\ndef effects(results,bundle,cohort,cache):\n    old=data.gt_map(bundle[\'evaluations\'][\'candidato_10\'][cohort][\'events\'])\n    anchor=data.gt_map(parse_current_events(results[cohort][.25][\'events\']))\n    lost={k for k in anchor if old[k][\'ha_alerta_ref\'] and old[k][\'ha_alerta_pred\'] and not anchor[k][\'ha_alerta_pred\']}\n    changed=[];losses=[];priority=[]\n    for key in sorted(lost):\n        a,b=old[key],anchor[key]\n        if b[\'evento\']==\'correspondencia\':\n            priority.append({\'coorte\':cohort,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                \'prioridade_candidato10\':a[\'prioridade_pred\'],\'prioridade_B_025\':b[\'prioridade_pred\'],\n                \'confianca_candidato10\':a[\'confianca\'],\'confianca_B_025\':b[\'confianca\'],\n                \'observacao\':\'Objeto ainda associado; alerta pode mudar pela geometria/prioridade.\'})\n    for conf in THRESHOLDS:\n        current=data.gt_map(parse_current_events(results[cohort][conf][\'events\']))\n        for key,b in current.items():\n            a=anchor[key]\n            if a[\'ha_alerta_ref\'] and a[\'ha_alerta_pred\']!=b[\'ha_alerta_pred\']:\n                changed.append({\'coorte\':cohort,\'conf\':conf,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                    \'alerta_B_025\':a[\'ha_alerta_pred\'],\'alerta_no_limiar\':b[\'ha_alerta_pred\'],\n                    \'prioridade_ref\':b[\'prioridade_ref\'],\'confianca_associada\':b[\'confianca\']})\n            if key not in lost:continue\n            same=[p for p in cache[key[0]] if p[\'classe_id\']==key[1]]\n            ious=core.iou_matrix([{\'bbox\':list(key[2])}],same)[0]\n            best=max(range(len(same)),key=lambda i:ious[i]) if same else None\n            losses.append({\'coorte\':cohort,\'conf\':conf,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                \'prioridade_ref\':b[\'prioridade_ref\'],\'recuperado\':b[\'ha_alerta_pred\'],\n                \'evento_B_025\':a[\'evento\'],\'evento_no_limiar\':b[\'evento\'],\n                \'melhor_iou_cache_mesma_classe\':float(ious[best]) if best is not None else None,\n                \'confianca_dessa_proposta\':same[best][\'confianca\'] if best is not None else None,\n                \'limite\':\'Melhor IoU não garante associação nem alerta; coluna recuperado usa avaliação completa.\'})\n    return changed,losses,priority\n\n\ndef validate_predictions(pred,record):\n    """Valida estrutura; preserva coordenadas numéricas como no avaliador 11/12.\n\n    Área zero e limites da imagem são diagnósticos, não filtros do cache.\n    Nenhuma proposta é removida, recortada ou corrigida aqui.\n    """\n    totals={\'predicoes\':len(pred),\'area_zero\':0,\'fora_da_imagem\':0,\n            \'area_zero_na_grade\':0,\'fora_da_imagem_na_grade\':0,\'amostras\':[]}\n    for index,p in enumerate(pred):\n        reason=None\n        if not isinstance(p,dict):\n            raise ValueError(f"Previsão não é um dicionário: {record[\'id\']}; índice={index}; valor={p!r}")\n        cid=p.get(\'classe_id\');box=p.get(\'bbox\');score=p.get(\'confianca\')\n        if type(cid)!=int or not 0<=cid<len(core.NAMES):reason=\'classe_id inválida\'\n        elif p.get(\'classe\')!=core.NAMES[cid]:reason=\'nome da classe incompatível\'\n        elif type(score) not in (int,float) or not math.isfinite(score) or not .001-1e-8<=score<=1:\n            reason=\'confiança não finita ou fora da faixa de captura\'\n        elif not isinstance(box,(list,tuple)) or len(box)!=4:reason=\'bbox deve ter quatro coordenadas\'\n        elif not all(type(v) in (int,float) and math.isfinite(v) for v in box):reason=\'coordenada não numérica ou não finita\'\n        elif box[2]<box[0] or box[3]<box[1]:reason=\'coordenadas xyxy invertidas\'\n        if reason:\n            raise ValueError(f"Previsão incompatível com o cache: {record[\'id\']}; motivo={reason}; "\n                f"índice={index}; classe_id={cid!r}; confiança={score!r}; bbox={box!r}; "\n                f"dimensões=(w={record[\'width\']}, h={record[\'height\']})")\n        zero=box[2]==box[0] or box[3]==box[1]\n        outside=(min(box)<-.01 or box[2]>record[\'width\']+.01 or box[3]>record[\'height\']+.01)\n        totals[\'area_zero\']+=int(zero);totals[\'fora_da_imagem\']+=int(outside)\n        totals[\'area_zero_na_grade\']+=int(zero and score>=min(THRESHOLDS))\n        totals[\'fora_da_imagem_na_grade\']+=int(outside and score>=min(THRESHOLDS))\n        if (zero or outside) and len(totals[\'amostras\'])<5:\n            totals[\'amostras\'].append({\'indice\':index,\'classe\':p[\'classe\'],\'confianca\':score,\n                \'bbox\':list(box),\'area_zero\':zero,\'fora_da_imagem\':outside,\n                \'width\':record[\'width\'],\'height\':record[\'height\']})\n    return totals\n\ndef save_geometry_diagnostics(directory,items):\n    totals={key:sum(item[\'diagnostico\'][key] for item in items) for key in\n            [\'predicoes\',\'area_zero\',\'fora_da_imagem\',\'area_zero_na_grade\',\'fora_da_imagem_na_grade\']}\n    unusual=[item for item in items if item[\'diagnostico\'][\'area_zero\'] or item[\'diagnostico\'][\'fora_da_imagem\']]\n    report={\'correcao\':\'14-cache-validacao-1.0.1\',\'imagens\':len(items),\n        \'imagens_com_geometria_incomum\':len(unusual),\'totais\':totals,\n        \'propostas_removidas\':0,\'coordenadas_alteradas\':False,\n        \'observacao\':\'Geometria incomum preservada no cache e no protocolo; não constitui objeto aprovado. Reprodução em 0,25 continua obrigatória.\',\n        \'casos\':unusual}\n    core.save_json(Path(directory)/\'diagnostico_geometria_previsoes.json\',report)\n    return {k:v for k,v in report.items() if k!=\'casos\'}\n\ndef obtain_cache(model,records,directory,weight_sha,environment,device,predict_fn=None):\n    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)\n    path=directory/\'predicoes_conf_0p001.jsonl.gz\';meta_path=directory/\'cache_manifesto.json\'\n    request=cache_request(records,weight_sha,environment)\n    if path.is_file() and meta_path.is_file():\n        meta=json.loads(meta_path.read_text())\n        if meta[\'solicitacao\']!=request or meta[\'cache_sha256\']!=core.sha256_file(path):\n            raise RuntimeError(\'Cache existente não corresponde aos pesos/entradas/ambiente. Não será reutilizado.\')\n        cached=read_cache(path,records)\n        diagnostics=[{\'id\':r[\'id\'],\'diagnostico\':validate_predictions(cached[r[\'id\']],r)} for r in records]\n        geometry=save_geometry_diagnostics(directory,diagnostics)\n        print(\'Cache conferido e reutilizado:\',records[0][\'coorte\'],flush=True)\n        return path,{**meta,\'geometria_resumo\':geometry,\'reutilizado_nesta_execucao\':True}\n    predict=predict_fn or core.model_predictions\n    tmp=path.with_name(path.name+\'.tmp\')\n    saturated=0;diagnostics=[]\n    with gzip.open(tmp,\'wt\',encoding=\'utf-8\',compresslevel=4) as f:\n        for i,r in enumerate(records,1):\n            if r[\'role\'] not in {\'home_val\',\'val_nova\'}:raise ValueError(\'Papel proibido na inferência.\')\n            pred=predict(model,r,device)\n            try:diagnostic=validate_predictions(pred,r)\n            except ValueError as exc:\n                core.save_json(directory/\'previsao_rejeitada.json\',{\'id\':r[\'id\'],\'mensagem\':str(exc),\n                    \'width\':r[\'width\'],\'height\':r[\'height\'],\'propostas\':pred})\n                raise\n            diagnostics.append({\'id\':r[\'id\'],\'diagnostico\':diagnostic})\n            if len(pred)==PREDICT_ARGS[\'max_det\']:saturated+=1\n            f.write(json.dumps({\'id\':r[\'id\'],\'imagem\':r[\'name\'],\'image_sha256\':r[\'sha256\'],\n                                \'label_sha256\':r[\'label_sha256\'],\'predicoes\':pred},ensure_ascii=False)+\'\\n\')\n            if i%100==0 or i==len(records):print(f"Checkpoint / {r[\'coorte\']}: {i}/{len(records)}",flush=True)\n    tmp.replace(path)\n    geometry=save_geometry_diagnostics(directory,diagnostics)\n    meta={\'solicitacao\':request,\'cache_sha256\':core.sha256_file(path),\'concluido\':True,\n          \'validacao_previsoes\':\'14-cache-validacao-1.0.1\',\'geometria_resumo\':geometry,\n          \'imagens_atingiram_max_det\':saturated,\'limite\':\'Propostas pós-NMS em confiança 0,001; max_det=300, sem acesso aos logits brutos.\'}\n    core.save_json(meta_path,meta)\n    read_cache(path,records)\n    print(\'Diagnóstico geométrico do cache:\',geometry[\'totais\'],flush=True)\n    return path,{**meta,\'reutilizado_nesta_execucao\':False}\n', 'control16_data.py': '"""Autenticação da auditoria, replay histórico e projeções locais por braço."""\nfrom __future__ import annotations\nimport gzip,hashlib,json,shutil,zipfile\nfrom pathlib import Path\nfrom collections import Counter,defaultdict\nimport numpy as np\nfrom PIL import Image\nimport yaml\nimport diag11_core as core\nimport diag11_data as d11\nimport control16_protocol as p16\n\nQUARANTINE_IDS={\n \'home:homeobjects_yolo/images/train/raw_train/living_room_1p (200).jpg\',\n \'home:homeobjects_yolo/images/train/raw_train/living_room_1p (203).jpg\',\n \'home:homeobjects_yolo/images/train/raw_train/living_room_590.jpg\',\n \'home:homeobjects_yolo/images/train/raw_train/living_room_718.jpg\'}\n\ndef read_audit15(path,expected_sha):\n    if core.sha256_file(path)!=expected_sha:raise ValueError(\'Pacote 15 diferente do analisado.\')\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist())!=len(set(z.namelist())):raise ValueError(\'Pacote 15 com nomes repetidos.\')\n        for n in z.namelist():core.safe_member(n)\n        if z.testzip():raise ValueError(\'CRC inválido no pacote 15.\')\n        summary=json.loads(z.read(\'resumo_auditoria_15.json\'));cfg=json.loads(z.read(\'configuracao_auditoria_15.json\'))\n        records=json.loads(gzip.decompress(z.read(\'manifesto_imagens_auditadas.json.gz\')))\n    if not summary[\'concluido\'] or len(records)!=6824 or p16.digest(records)!=cfg[\'dados_sha256\']:\n        raise ValueError(\'Auditoria 15 incompleta ou alterada.\')\n    if Counter(r[\'source\'] for r in records)!={\'home\':1824,\'openimages\':5000} or any(r[\'role\']!=\'train\' for r in records):\n        raise ValueError(\'Papel/origem inesperados no pacote 15.\')\n    by_id={r[\'id\']:r for r in records}\n    if len(by_id)!=6824 or not QUARANTINE_IDS<=by_id.keys():raise ValueError(\'Identidades da auditoria incompatíveis.\')\n    groups=defaultdict(list)\n    for r in records:groups[r[\'sha256\']].append(r)\n    duplicates=[rs for rs in groups.values() if len(rs)>1]\n    exact_ids={r[\'id\'] for rs in duplicates for r in rs}\n    if len(duplicates)!=2 or exact_ids!=QUARANTINE_IDS or any(len({r[\'label_sha256\'] for r in rs})!=len(rs) for rs in duplicates):\n        raise ValueError(\'Os pares exatos não correspondem à quarentena pré-definida.\')\n    return {\'records\':by_id,\'summary\':summary,\'config\':cfg,\n            \'quarantine\':[{\'id\':r[\'id\'],\'image_sha256\':r[\'sha256\'],\'label_sha256\':r[\'label_sha256\']}\n                          for rs in duplicates for r in rs]}\n\ndef read_replay12(package_path,expected_sha,audit,expected_count=2000):\n    if core.sha256_file(package_path)!=expected_sha:raise ValueError(\'Pacote 12 alterado.\')\n    with zipfile.ZipFile(package_path) as z:\n        obj=json.loads(z.read(\'manifesto_replay.json\'))\n    rows=obj[\'imagens\']\n    if len(rows)!=expected_count or len({r[\'id\'] for r in rows})!=expected_count:raise ValueError(\'Replay histórico incompleto.\')\n    for r in rows:\n        a=audit[\'records\'][\'oi:\'+r[\'id\']]\n        if (r[\'sha256\']!=a[\'sha256\'] or r[\'label_sha256\']!=a[\'label_sha256\'] or r[\'known_mask\']!=a[\'known_mask\']\n                or r[\'coverage_sha256\']!=a[\'coverage_sha256\'] or r[\'lote\']!=a[\'lote\']):\n            raise ValueError(\'Replay 12 incompatível com a auditoria 15.\')\n    return rows\n\ndef extract_replay_exact(lots,rows,history_rows,local,fetch,cfg_sha,protected_ids,protected_hashes,near,audit):\n    wanted={r[\'id\']:r for r in rows};history={r[\'id\']:r for r in history_rows};local=Path(local)\n    records=[];seen=set();receipts=[]\n    for item in lots:\n        path=local/item[\'arquivo\'];fetch(item,path)\n        if core.sha256_file(path)!=item[\'zip_sha256\'] or path.stat().st_size!=item[\'zip_bytes\']:raise ValueError(\'Lote incompleto/alterado.\')\n        with zipfile.ZipFile(path) as z:\n            if len(z.namelist())!=len(set(z.namelist())):raise ValueError(\'Lote com nomes repetidos.\')\n            for n in z.namelist():core.safe_member(n)\n            manifest=json.loads(z.read(\'manifesto_lote.json\'))\n            if manifest[\'config_sha256\']!=cfg_sha or manifest[\'origem_split\']!=\'train\':raise ValueError(\'Lote fora do TRAIN autenticado.\')\n            for r in manifest[\'aceitas\']:\n                if r[\'id\'] not in wanted:continue\n                if r[\'id\'] in seen:raise ValueError(\'ID duplicado no replay.\')\n                seen.add(r[\'id\']);prior=wanted[r[\'id\']];h=history[r[\'id\']];a=audit[\'records\'][\'oi:\'+r[\'id\']]\n                if (h[\'split\']!=\'train\' or h[\'lote\']!=item[\'arquivo\'] or prior[\'lote\']!=item[\'arquivo\']\n                    or h[\'imagem_sha256\']!=r[\'sha256\'] or int(h[\'known_mask\'])!=r[\'known_mask\']):raise ValueError(\'Replay fora da divisão histórica.\')\n                if r[\'id\'] in protected_ids or r[\'sha256\'] in protected_hashes:raise ValueError(\'Replay coincide com conjunto protegido.\')\n                mapping=[(\'imagem\',\'sha256\'),(\'label\',\'label_sha256\'),(\'cobertura\',\'cobertura_sha256\')]\n                blobs={}\n                for field,key in mapping:\n                    raw=z.read(r[field]);sha=hashlib.sha256(raw).hexdigest()\n                    expected=prior[\'coverage_sha256\'] if key==\'cobertura_sha256\' else prior[key]\n                    if sha!=r[key] or sha!=expected:raise ValueError(\'Bytes do replay divergentes.\')\n                    blobs[field]=raw\n                cov=json.loads(blobs[\'cobertura\'])\n                positives=set(cov[\'classes_positivas_verificadas_exaustivas\']);negatives=set(cov[\'classes_negativas_verificadas\'])\n                if (positives&negatives or sum(1<<c for c in positives|negatives)!=r[\'known_mask\']\n                    or cov[\'known_mask\']!=r[\'known_mask\']):raise ValueError(\'Cobertura parcial inválida.\')\n                image=local/\'images/train\'/(\'oi_\'+r[\'id\']+\'.jpg\');label=local/\'labels/train\'/(\'oi_\'+r[\'id\']+\'.txt\')\n                for target,blob in [(image,blobs[\'imagem\']),(label,blobs[\'label\'])]:target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(blob)\n                with Image.open(image) as im:w,hgt=im.size;im.verify()\n                if (w,hgt)!=(r[\'width\'],r[\'height\']):raise ValueError(\'Dimensões do replay alteradas.\')\n                if near.find(d11.phash(image)):raise ValueError(\'Replay histórico passou a coincidir perceptualmente com conjunto protegido; não haverá substituição silenciosa de IDs.\')\n                gt=core.parse_yolo(blobs[\'label\'],w,hgt)\n                if {g[\'classe_id\'] for g in gt}&negatives or not positives<={g[\'classe_id\'] for g in gt}:raise ValueError(\'Cobertura e caixas divergem.\')\n                if hashlib.sha256(blobs[\'label\']).hexdigest()!=a[\'label_sha256\']:raise ValueError(\'Label diverge da auditoria 15.\')\n                records.append({\'id\':\'oi:\'+r[\'id\'],\'oi_id\':r[\'id\'],\'source\':\'openimages\',\'role\':\'train\',\n                    \'image\':image,\'label\':label,\'width\':w,\'height\':hgt,\'known_mask\':r[\'known_mask\'],\n                    \'sha256\':r[\'sha256\'],\'label_sha256\':r[\'label_sha256\'],\'coverage_sha256\':r[\'cobertura_sha256\'],\n                    \'lote\':prior[\'lote\'],\'gt\':gt,\'name\':image.name})\n        receipts.append(item);path.unlink()\n    if seen!=wanted.keys():raise ValueError(\'Não foram recuperadas todas as imagens do replay 12.\')\n    return records,receipts\n\ndef training_home(home,audit):\n    selected=[];quarantined=[]\n    for r in home:\n        if r[\'role\']!=\'home_train\':continue\n        a=audit[\'records\'][r[\'id\']]\n        if r[\'sha256\']!=a[\'sha256\'] or r[\'label_sha256\']!=a[\'label_sha256\']:raise ValueError(\'Home diverge da auditoria 15.\')\n        if r[\'id\'] in QUARANTINE_IDS:quarantined.append(r[\'id\']);continue\n        selected.append({**r,\'role\':\'train\',\'origin_role\':\'home_train\',\'source\':\'home\',\'known_mask\':4095})\n    if len(selected)!=1820 or set(quarantined)!=QUARANTINE_IDS:raise ValueError(\'Quarentena Home diferente da planejada.\')\n    return selected\n\ndef make_arms(records,home_val,destination,output):\n    destination=Path(destination);output=Path(output);destination.mkdir(parents=True,exist_ok=True)\n    if any(r[\'role\']!=\'train\' for r in records):raise ValueError(\'Somente TRAIN pode receber projeção de supervisão.\')\n    ids={r[\'id\'] for r in records}\n    if len(ids)!=len(records):raise ValueError(\'Treino com ID duplicado.\')\n    arms={};manifest=[];frozen=[]\n    required=sum(Path(r[\'image\']).stat().st_size for r in records)*2+256*1024**2\n    if shutil.disk_usage(destination).free<required:raise RuntimeError(\'Espaço insuficiente para as cópias isoladas dos braços.\')\n    val_file=destination/\'home_val.txt\';val_file.write_text(\'\\n\'.join(str(Path(r[\'image\']).resolve()) for r in home_val)+\'\\n\')\n    for arm in p16.ARMS:\n        root=destination/arm;mapping={};rows=[];ignored=0\n        for r in p16.logical_order(records):\n            image=root/\'images/train\'/p16.filename(r);label=root/\'labels/train\'/Path(p16.filename(r)).with_suffix(\'.txt\')\n            image.parent.mkdir(parents=True,exist_ok=True);label.parent.mkdir(parents=True,exist_ok=True)\n            shutil.copy2(r[\'image\'],image)\n            original=Path(r[\'label\']).read_bytes()\n            if hashlib.sha256(original).hexdigest()!=r[\'label_sha256\']:raise ValueError(\'Label original modificado antes da projeção.\')\n            projected,mask,removed=p16.project_label(original,r[\'source\'],arm,r[\'known_mask\']);ignored+=removed\n            label.write_bytes(projected)\n            if core.sha256_file(image)!=r[\'sha256\']:raise ValueError(\'A cópia de imagem diverge do original.\')\n            if not p16.same_other_classes(original,projected):raise ValueError(\'Outras classes foram alteradas pela projeção.\')\n            gt=core.parse_yolo(projected,r[\'width\'],r[\'height\'])\n            obj={**r,\'image\':image,\'label\':label,\'gt\':gt,\'known_mask\':mask,\'original_known_mask\':r[\'known_mask\'],\n                 \'original_label_sha256\':r[\'label_sha256\'],\'label_sha256\':hashlib.sha256(projected).hexdigest(),\n                 \'ignored_home7_objects\':removed,\'arm\':arm}\n            if arm==p16.ARMS[1] and r[\'source\']==\'home\' and (mask&(1<<7) or any(g[\'classe_id\']==7 for g in gt)):\n                raise ValueError(\'Wardrobe Home permaneceu supervisionada no braço experimental.\')\n            key=image.resolve().as_posix();mapping[key]=obj\n            row={k:obj.get(k) for k in [\'id\',\'source\',\'role\',\'sha256\',\'label_sha256\',\'original_label_sha256\',\n                    \'known_mask\',\'original_known_mask\',\'ignored_home7_objects\']}\n            row.update(arm=arm,relative_image=str(image.relative_to(destination)),relative_label=str(label.relative_to(destination)),\n                       gt_per_class=dict(Counter(g[\'classe_id\'] for g in gt)))\n            rows.append(row);frozen.extend([(image,r[\'sha256\']),(label,obj[\'label_sha256\'])])\n        train=root/\'train.txt\';train.write_text(\'\\n\'.join(sorted(mapping))+\'\\n\')\n        data_file=root/\'data.yaml\';data_file.write_text(yaml.safe_dump({\'path\':str(root.resolve()),\'train\':str(train.resolve()),\n            \'val\':str(val_file.resolve()),\'names\':dict(enumerate(core.NAMES))},sort_keys=False,allow_unicode=True))\n        arms[arm]={\'records\':mapping,\'yaml\':data_file,\'manifest\':rows,\'ignored_home7_objects\':ignored}\n        manifest.extend(rows)\n    a,b=[arms[k] for k in p16.ARMS]\n    if sorted(r[\'id\'] for r in a[\'records\'].values())!=sorted(r[\'id\'] for r in b[\'records\'].values()):raise ValueError(\'Braços com imagens distintas.\')\n    core.save_json(output/\'manifesto_supervisao_16.json\',manifest)\n    core.save_json(output/\'quarentena_duplicacoes_exatas.json\',sorted(QUARANTINE_IDS))\n    return arms,frozen\n', 'control16_protocol.py': '"""Protocolo pareado: somente a supervisão Home/wardrobe muda entre braços."""\nfrom __future__ import annotations\nimport gzip,hashlib,json,math\nfrom pathlib import Path\nfrom collections import Counter\nimport exp12_protocol as historical\n\nVERSION=\'16-home-unknown7-pareado-1.0.0\'\nARMS=(\'C_original\',\'M_home_unknown7\')\nWARDROBE=7\nSPEC={\'epochs\':3,\'batches\':114,\'batch\':16,\'oi_per_batch\':4,\'lr0\':3e-6,\n      \'home_unique\':1820,\'oi_unique\':2000,\'seed\':42}\nAUGMENTATION_ZERO=[\'mosaic\',\'mixup\',\'cutmix\',\'copy_paste\',\'scale\',\'translate\',\'fliplr\',\'flipud\',\n                   \'degrees\',\'shear\',\'perspective\',\'hsv_h\',\'hsv_s\',\'hsv_v\',\'multi_scale\']\n\ndef normalize(x):\n    if isinstance(x,dict):return {str(k):normalize(v) for k,v in x.items()}\n    if isinstance(x,(list,tuple)):return [normalize(v) for v in x]\n    return x\ndef canonical(x):return json.dumps(normalize(x),sort_keys=True,ensure_ascii=False,separators=(\',\',\':\'),allow_nan=False)\ndef digest(x):return hashlib.sha256(canonical(x).encode()).hexdigest()\ndef save_gz(path,obj):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_name(path.name+\'.tmp\')\n    with tmp.open(\'wb\') as f,gzip.GzipFile(fileobj=f,mode=\'wb\',mtime=0) as g:g.write(canonical(obj).encode())\n    tmp.replace(path)\ndef load_gz(path):\n    with gzip.open(path,\'rt\',encoding=\'utf-8\') as f:return json.load(f)\n\ndef filename(record):\n    prefix=\'home\' if record[\'source\']==\'home\' else \'oi\'\n    return prefix+\'_\'+hashlib.sha256(record[\'id\'].encode()).hexdigest()+\'.jpg\'\n\ndef project_label(raw,source,arm,known):\n    if arm not in ARMS or source not in {\'home\',\'openimages\'}:raise ValueError(\'Braço/fonte desconhecido.\')\n    if not 0<=known<4096:raise ValueError(\'Máscara inválida.\')\n    lines=raw.decode(\'utf-8-sig\').splitlines(keepends=True)\n    removed=[];kept=[]\n    for line in lines:\n        if not line.strip():kept.append(line);continue\n        parts=line.split()\n        if len(parts)!=5:raise ValueError(\'Label deve conter cinco campos por linha.\')\n        values=[float(v) for v in parts]\n        if not all(map(math.isfinite,values)) or not values[0].is_integer() or not 0<=values[0]<12:\n            raise ValueError(\'Classe ou valor inválido no label.\')\n        cid=int(values[0])\n        if arm==ARMS[1] and source==\'home\' and cid==WARDROBE:removed.append(line)\n        else:kept.append(line)\n    if arm==ARMS[1] and source==\'home\':\n        return \'\'.join(kept).encode(\'utf-8\'),known&~(1<<WARDROBE),len(removed)\n    return raw,known,0\n\ndef logical_order(records):return sorted(records,key=filename)\n\ndef schedule(records,spec=SPEC):\n    ordered=logical_order(records)\n    home=[i for i,r in enumerate(ordered) if r[\'source\']==\'home\']\n    oi=[i for i,r in enumerate(ordered) if r[\'source\']==\'openimages\']\n    sampler=historical.BudgetSampler(home,oi,spec[\'batch\'],spec[\'batches\'],spec[\'oi_per_batch\'],spec[\'seed\'])\n    rows=[]\n    for epoch in range(spec[\'epochs\']):\n        sampler.set_epoch(epoch);seq=list(sampler)\n        for offset in range(0,len(seq),spec[\'batch\']):\n            chosen=[ordered[i] for i in seq[offset:offset+spec[\'batch\']]]\n            ids=[r[\'id\'] for r in chosen]\n            rows.append({\'step\':len(rows)+1,\'epoch\':epoch+1,\'ids\':ids,\n                         \'input_signature\':digest([(r[\'id\'],r[\'sha256\']) for r in chosen])})\n    return rows\n\ndef same_other_classes(raw_a,raw_b):\n    def rows(raw):\n        return [tuple(map(float,l.split())) for l in raw.decode(\'utf-8-sig\').splitlines() if l.strip() and int(float(l.split()[0]))!=WARDROBE]\n    return rows(raw_a)==rows(raw_b)\n\ndef safe_judge(home,external,rule):\n    valid=all(s[\'alertas_por_objeto\'][k] is not None and math.isfinite(s[\'alertas_por_objeto\'][k])\n              for s in [home,external] for k in [\'precision\',\'recall\',\'f1\'])\n    if valid:return {**historical.judge(home,external,rule),\'metricas_com_suporte\':True}\n    return {\'elegivel\':False,\'retencao_externa\':False,\'metricas_com_suporte\':False}\n\ndef select_final(entries,rule):\n    assessed=[{**e,\'checks\':safe_judge(e[\'home\'],e[\'external\'],rule)} for e in entries]\n    primary=[e for e in assessed if e[\'arm\'] in ARMS and e[\'epoch\']==SPEC[\'epochs\']]\n    if len(primary)!=2:raise ValueError(\'Exigidos os dois checkpoints finais pré-fixados.\')\n    good=[e for e in primary if e[\'checks\'][\'elegivel\']]\n    def key(e):\n        h=e[\'home\'][\'alertas_por_objeto\'];x=e[\'external\'][\'alertas_por_objeto\']\n        return (-h[\'precision\'],-h[\'recall\'],e[\'home\'][\'alta_sem_alerta\'],-x[\'f1\'],e[\'id\'])\n    chosen=min(good,key=key) if good else None\n    return {\'assessed\':assessed,\'primary_ids\':[e[\'id\'] for e in primary],\n            \'eligible\':chosen,\'status\':\'checkpoint_final_elegivel_em_validacao\' if chosen else \'nenhum_checkpoint_final_elegivel\',\n            \'checkpoints_intermediarios\':\'somente diagnóstico, sem substituir o desfecho principal e03\',\n            \'modelo_em_uso_continua_baseline\':True,\'modelo_substituido\':False}\n\ndef compare_pair(log_c,log_m,expected_schedule):\n    if not(len(log_c)==len(log_m)==len(expected_schedule)):raise ValueError(\'Quantidade de batches diferente entre braços.\')\n    home7_ignored=0;oi7_targets=0\n    for a,b,p in zip(log_c,log_m,expected_schedule):\n        for k in [\'step\',\'epoch\',\'ids\',\'input_signature\']:\n            if a[k]!=b[k] or a[k]!=p[k]:raise ValueError(\'Amostragem pareada divergente: \'+k)\n        if a[\'tensor_amostra_sha256\']!=b[\'tensor_amostra_sha256\']:raise ValueError(\'As amostras de pixels dos batches divergem.\')\n        if a[\'source_counts\']!=b[\'source_counts\']:raise ValueError(\'Exposições divergentes.\')\n        if a[\'gradient_audit\'][\'unknown_negative_max\']!=0 or b[\'gradient_audit\'][\'unknown_negative_max\']!=0:\n            raise ValueError(\'Gradiente negativo vazou em classe desconhecida.\')\n        if b[\'gradient_audit\'][\'home7_max\']!=0:raise ValueError(\'Gradiente de wardrobe Home não foi zerado.\')\n        for i,iid in enumerate(a[\'ids\']):\n            ca={str(k):v for k,v in a[\'gt_per_image_by_class\'][i].items()}\n            cb={str(k):v for k,v in b[\'gt_per_image_by_class\'][i].items()}\n            if iid.startswith(\'home:\'):\n                home7_ignored+=ca.get(\'7\',0)\n                if cb.get(\'7\',0) or a[\'known_masks\'][i]!=4095 or b[\'known_masks\'][i]!=3967:\n                    raise ValueError(\'Supervisão Home7 diferente do plano.\')\n                ca.pop(\'7\',None);cb.pop(\'7\',None)\n            else:\n                oi7_targets+=ca.get(\'7\',0)\n                if a[\'known_masks\'][i]!=b[\'known_masks\'][i]:raise ValueError(\'Máscara Open Images alterada.\')\n            if ca!=cb:raise ValueError(\'Outros alvos diferem entre braços no batch observado.\')\n    if oi7_targets<=0:raise ValueError(\'Nenhum alvo wardrobe Open Images foi exposto ao treino.\')\n    return {\'pareamento_conferido\':True,\'batches_por_braco\':len(log_c),\n            \'mesmos_ids_na_mesma_ordem\':True,\'mesmos_hashes_imagem\':True,\n            \'amostras_tensor_iguais\':True,\'gradiente_negativo_desconhecido_zero\':True,\n            \'gradiente_home7_experimental_zero\':True,\n            \'exposicoes_home7_ignoradas_somente_M\':home7_ignored,\n            \'exposicoes_oi7_preservadas_em_ambos\':oi7_targets,\n            \'outros_alvos_e_mascaras_preservados\':True,\n            \'tensor_limite\':\'SHA de amostras espaciais de pixels, combinado a identidade das imagens e ausência de augmentações; não é um hash de cada pixel do tensor completo.\'}\n', 'diag11_core.py': '"""Núcleo do diagnóstico 11. Não contém treinamento nem seleção de pesos."""\nfrom __future__ import annotations\n\nimport csv\nimport gzip\nimport hashlib\nimport io\nimport json\nimport math\nfrom collections import Counter, defaultdict\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nfrom PIL import Image\n\nNAMES = ["bed", "sofa", "chair", "table", "lamp", "tv", "laptop",\n         "wardrobe", "window", "door", "potted plant", "photo frame"]\nALL_KNOWN = (1 << len(NAMES)) - 1\nALERT_IDS = {0, 1, 2, 3, 7, 10}\nALERT_MASK = sum(1 << c for c in ALERT_IDS)\nLEVEL = {"sem_prioridade": 0, "baixo": 1, "medio": 2, "alto": 3}\n\n\ndef canonical(x):\n    return json.dumps(x, sort_keys=True, ensure_ascii=False, separators=(",", ":"))\n\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for block in iter(lambda: f.read(2**20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef json_clean(value):\n    if isinstance(value, dict):\n        return {str(k): json_clean(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [json_clean(v) for v in value]\n    if isinstance(value, np.generic):\n        return json_clean(value.item())\n    if isinstance(value, Path):\n        return str(value)\n    if isinstance(value, float) and not math.isfinite(value):\n        return None\n    return value\n\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    tmp.write_text(json.dumps(json_clean(value), ensure_ascii=False, indent=2,\n                              allow_nan=False), encoding="utf-8")\n    tmp.replace(path)\n\n\ndef save_csv(path, rows, fields=None):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    rows = list(rows)\n    if fields is None:\n        fields = list(dict.fromkeys(k for row in rows for k in row))\n    with path.open("w", newline="", encoding="utf-8-sig") as f:\n        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="raise")\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef safe_member(name):\n    p = PurePosixPath(name)\n    if p.is_absolute() or ".." in p.parts or "\\\\" in name:\n        raise ValueError(f"Membro inseguro no ZIP: {name}")\n    return p\n\n\ndef parse_yolo(raw, width, height, names=NAMES, source="label"):\n    objects = []\n    for line_no, line in enumerate(raw.decode("utf-8-sig").splitlines(), 1):\n        if not line.strip():\n            continue\n        parts = line.split()\n        if len(parts) != 5:\n            raise ValueError(f"{source}:{line_no}: esperados cinco campos.")\n        vals = [float(s) for s in parts]\n        if not all(math.isfinite(v) for v in vals):\n            raise ValueError(f"{source}:{line_no}: valor não finito.")\n        c, x, y, w, h = vals\n        if int(c) != c or not 0 <= c < len(names) or min(w, h) <= 0:\n            raise ValueError(f"{source}:{line_no}: classe ou dimensão inválida.")\n        box = [x-w/2, y-h/2, x+w/2, y+h/2]\n        if min(box) < -0.002 or max(box) > 1.002:\n            raise ValueError(f"{source}:{line_no}: caixa fora da imagem.")\n        box = [max(0, min(1, v)) for v in box]\n        objects.append({"classe_id": int(c), "classe": names[int(c)],\n                        "bbox": [box[0]*width, box[1]*height,\n                                 box[2]*width, box[3]*height],\n                        "ocluido": None, "truncado": None})\n    return objects\n\n\ndef hierarchy_mapping(cfg, class_rows, hierarchy):\n    if cfg["nomes"] != NAMES or len(cfg["mids_raizes"]) != len(NAMES):\n        raise ValueError("Classes da preparação incompatíveis com o projeto.")\n    descriptions = {r[0]: r[1] for r in class_rows if len(r) >= 2}\n    roots = cfg["mids_raizes"]\n    if any(mid not in descriptions for mid in roots):\n        raise ValueError("A descrição oficial não contém todas as raízes.")\n    descendants, ancestors = defaultdict(set), defaultdict(set)\n\n    def walk(node, parents=()):\n        mid = node["LabelName"]\n        ancestors[mid].update(parents)\n        descendants[mid].add(mid)\n        for parent in parents:\n            descendants[parent].add(mid)\n        for child in node.get("Subcategory", []):\n            walk(child, parents + (mid,))\n\n    walk(hierarchy)\n    excluded_names = set(cfg.get("ambiguas_excluidas", []))\n    excluded = {mid for mid, name in descriptions.items() if name in excluded_names}\n    candidates = defaultdict(set)\n    root_desc = {}\n    for cid, mid in enumerate(roots):\n        if mid not in descendants:\n            raise ValueError(f"Raiz ausente da hierarquia: {mid}")\n        root_desc[cid] = set(descendants[mid])\n        for sub in root_desc[cid]:\n            candidates[sub].add(cid)\n    ambiguous = excluded | {mid for mid, ids in candidates.items() if len(ids) > 1}\n    mapping = {mid: next(iter(ids)) for mid, ids in candidates.items()\n               if mid not in ambiguous}\n    return {"roots": roots, "descendants": root_desc, "ancestors": ancestors,\n            "mapping": mapping, "ambiguous": ambiguous,\n            "descriptions": descriptions,\n            "relevant_labels": set().union(*root_desc.values(),\n                                            *(ancestors[r] for r in roots))}\n\n\ndef prepare_oi_annotations(box_rows, label_rows, info):\n    """Cobertura conservadora: positivo na raiz ou negativo na raiz/ancestral.\n\n    Uma positiva apenas numa subclasse não prova cobertura exaustiva da classe\n    agregada. Grupos, representações, atributos desconhecidos ou caixas inválidas\n    tornam essa classe/imagem não avaliável, em vez de criarem falsos negativos.\n    """\n    labels = defaultdict(lambda: {"positive": set(), "negative": set()})\n    for row in label_rows:\n        if row.get("Source") not in {"verification", "crowdsource-verification"}:\n            continue\n        mid, confidence = row["LabelName"], float(row["Confidence"])\n        if mid not in info["relevant_labels"]:\n            continue\n        if confidence in (0, 1):\n            key = "positive" if confidence == 1 else "negative"\n            labels[row["ImageID"]][key].add(mid)\n    boxes, blocked = defaultdict(list), defaultdict(lambda: defaultdict(set))\n    audited = Counter()\n    for row in box_rows:\n        mid, iid = row["LabelName"], row["ImageID"]\n        affected = [cid for cid, mids in info["descendants"].items() if mid in mids]\n        if not affected:\n            continue\n        if mid in info["ambiguous"]:\n            for cid in affected:\n                blocked[iid][cid].add("subclasse_ambigua_excluida")\n            audited["caixas_subclasse_ambigua"] += 1\n            continue\n        cid = info["mapping"].get(mid)\n        if cid is None:\n            continue\n        flags = {k: int(float(row.get(k, -1))) for k in\n                 ["IsGroupOf", "IsDepiction", "IsOccluded", "IsTruncated"]}\n        reasons = []\n        if row.get("Source") not in {"xclick", "activemil"}:\n            reasons.append("origem_caixa_nao_reconhecida")\n        if flags["IsGroupOf"] != 0:\n            reasons.append("grupo_ou_atributo_desconhecido")\n        if flags["IsDepiction"] != 0:\n            reasons.append("representacao_ou_atributo_desconhecido")\n        xyxy = [float(row[k]) for k in ["XMin", "YMin", "XMax", "YMax"]]\n        if (not all(math.isfinite(v) for v in xyxy) or min(xyxy) < 0\n                or max(xyxy) > 1 or xyxy[2] <= xyxy[0] or xyxy[3] <= xyxy[1]):\n            reasons.append("coordenadas_invalidas")\n        if reasons:\n            blocked[iid][cid].update(reasons)\n            audited.update(reasons)\n            continue\n        boxes[iid].append({"classe_id": cid, "classe": NAMES[cid],\n                           "bbox_norm": xyxy, "mid_original": mid,\n                           "ocluido": flags["IsOccluded"] == 1 if flags["IsOccluded"] >= 0 else None,\n                           "truncado": flags["IsTruncated"] == 1 if flags["IsTruncated"] >= 0 else None})\n    records = {}\n    for iid in sorted(set(labels) | set(boxes)):\n        gt = []\n        seen = set()\n        for obj in boxes[iid]:\n            key = (obj["classe_id"], *(round(x, 8) for x in obj["bbox_norm"]))\n            if key not in seen:\n                seen.add(key)\n                gt.append(obj)\n        known, positive, negative, reasons = 0, 0, 0, {}\n        present = {o["classe_id"] for o in gt}\n        for cid, root in enumerate(info["roots"]):\n            pos = root in labels[iid]["positive"]\n            neg = bool(({root} | info["ancestors"][root]) & labels[iid]["negative"])\n            has_positive_label = bool(info["descendants"][cid] & labels[iid]["positive"])\n            reason = set(blocked[iid].get(cid, set()))\n            if neg and (cid in present or has_positive_label):\n                reason.add("contradicao_positivo_negativo")\n            if pos and cid not in present:\n                reason.add("positivo_sem_caixa_utilizavel")\n            if reason:\n                reasons[str(cid)] = sorted(reason)\n                continue\n            if neg:\n                known |= 1 << cid\n                negative |= 1 << cid\n            elif pos and cid in present:\n                known |= 1 << cid\n                positive |= 1 << cid\n        if positive or known & ALERT_MASK == ALERT_MASK:\n            records[iid] = {"id": iid, "source_split": "validation",\n                            "known_mask": known, "positive_mask": positive,\n                            "negative_mask": negative, "gt_normalizado": gt,\n                            "classes_positivas_avaliaveis": sorted(c for c in range(12)\n                                                                 if positive & (1 << c)),\n                            "restricoes_cobertura": reasons}\n    return records, dict(audited)\n\n\ndef iou_matrix(gt, pred):\n    if not gt or not pred:\n        return np.zeros((len(gt), len(pred)))\n    a = np.asarray([o["bbox"] for o in gt], dtype=float)\n    b = np.asarray([o["bbox"] for o in pred], dtype=float)\n    lt = np.maximum(a[:, None, :2], b[None, :, :2])\n    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])\n    intersection = np.maximum(rb-lt, 0).prod(axis=2)\n    aa = np.maximum(a[:, 2:]-a[:, :2], 0).prod(axis=1)\n    ba = np.maximum(b[:, 2:]-b[:, :2], 0).prod(axis=1)\n    return intersection / np.maximum(aa[:, None] + ba[None, :] - intersection, 1e-12)\n\n\ndef ap_matches(gt, pred, thresholds=np.linspace(.5, .95, 10)):\n    correct = np.zeros((len(pred), len(thresholds)), dtype=bool)\n    matrix = iou_matrix(gt, pred)\n    order = sorted(range(len(pred)), key=lambda pi: (-pred[pi]["confianca"], pi))\n    for ti, threshold in enumerate(thresholds):\n        used = set()\n        for pi in order:\n            options = [gi for gi, g in enumerate(gt) if gi not in used\n                       and g["classe_id"] == pred[pi]["classe_id"]\n                       and matrix[gi, pi] >= threshold]\n            if options:\n                gi = max(options, key=lambda k: (matrix[k, pi], -k))\n                used.add(gi)\n                correct[pi, ti] = True\n    return correct\n\n\ndef ratio(n, d):\n    return n/d if d else None\n\n\ndef alert_metrics(tp, fn, fp):\n    return {"tp": int(tp), "fn": int(fn), "fp": int(fp),\n            "precision": ratio(tp, tp+fp), "recall": ratio(tp, tp+fn),\n            "f1": ratio(2*tp, 2*tp+fp+fn)}\n\n\ndef alert_on(obj):\n    return bool(obj and LEVEL[obj["prioridade"]] >= LEVEL["medio"])\n\n\ndef with_policy(objects, width, height, policy):\n    return [{**o, **policy.avaliar_bbox(o["classe"], o["bbox"], width, height,\n                                       versao="v2")} for o in objects]\n\n\ndef record_event(record, model_name, event, g=None, p=None, iou=None, all_pred=()):\n    gbox = g["bbox"] if g else None\n    pbox = p["bbox"] if p else None\n    area = ((gbox[2]-gbox[0])*(gbox[3]-gbox[1]) /\n            (record["width"]*record["height"])) if gbox else None\n    side_at_640 = (min(gbox[2]-gbox[0], gbox[3]-gbox[1]) *\n                   640/max(record["width"], record["height"])) if gbox else None\n    flags = []\n    near_best = None\n    if event == "objeto_nao_detectado":\n        if side_at_640 < 32:\n            flags.append("objeto_pequeno_na_entrada_640")\n        if g.get("ocluido") is True:\n            flags.append("oclusao_anotada")\n        if g.get("truncado") is True:\n            flags.append("truncamento_anotado")\n        overlaps = iou_matrix([g], all_pred)[0] if all_pred else []\n        same = [i for i, o in enumerate(all_pred) if o["classe_id"] == g["classe_id"]]\n        if same:\n            best = max(same, key=lambda i: overlaps[i])\n            near_best = float(overlaps[best])\n            if overlaps[best] >= .5 and all_pred[best]["confianca"] < .25:\n                flags.append("deteccao_abaixo_da_confianca_025")\n            elif .1 <= overlaps[best] < .5:\n                flags.append("caixa_mesma_classe_iou_010_a_050")\n        if any(overlaps[i] >= .5 and o["classe_id"] != g["classe_id"]\n               and o["confianca"] >= .25 for i, o in enumerate(all_pred)):\n            flags.append("sobreposicao_com_previsao_de_outra_classe")\n        if not flags:\n            flags.append("omissao_sem_indicio_automatico")\n    if event == "previsao_nao_associada":\n        flags.append("previsao_sem_correspondencia_na_classe_avaliavel")\n    return {"coorte": record["coorte"], "imagem": record["name"], "id": record["id"],\n            "modelo": model_name, "evento": event,\n            "classe_id": g["classe_id"] if g else p["classe_id"],\n            "classe": g["classe"] if g else p["classe"],\n            "bbox_ref": json.dumps(gbox) if g else "",\n            "bbox_pred": json.dumps(pbox) if p else "",\n            "iou": iou, "confianca": p["confianca"] if p else None,\n            "prioridade_ref": g["prioridade"] if g else "sem_prioridade",\n            "prioridade_pred": p["prioridade"] if p else "sem_prioridade",\n            "ha_alerta_ref": alert_on(g), "ha_alerta_pred": alert_on(p),\n            "area_relativa_gt": area, "menor_lado_gt_em_640": side_at_640,\n            "oclusao_anotada": g.get("ocluido") if g else None,\n            "truncamento_anotado": g.get("truncado") if g else None,\n            "melhor_iou_mesma_classe": near_best,\n            "indicios": "|".join(flags)}\n\n\ndef analyse_prediction(record, pred_low, model_name, policy, matcher):\n    if record.get("role") not in {"home_val", "val_nova"}:\n        raise ValueError("Somente validação pode entrar na inferência do notebook 11.")\n    known = int(record["known_mask"])\n    gt = with_policy([g for g in record["gt"] if known & (1 << g["classe_id"])],\n                     record["width"], record["height"], policy)\n    pred_low_known = [p for p in pred_low if known & (1 << p["classe_id"])]\n    pred_all = with_policy([p for p in pred_low if p["confianca"] >= .25],\n                           record["width"], record["height"], policy)\n    pred = [p for p in pred_all if known & (1 << p["classe_id"])]\n    pairs, missing, extras = matcher(gt, pred)\n    events = []\n    for gi, pi, iou in pairs:\n        events.append(record_event(record, model_name, "correspondencia",\n                                   gt[gi], pred[pi], float(iou)))\n    for gi in missing:\n        events.append(record_event(record, model_name, "objeto_nao_detectado",\n                                   g=gt[gi], all_pred=pred_low_known))\n    for pi in extras:\n        events.append(record_event(record, model_name, "previsao_nao_associada", p=pred[pi]))\n    tp = sum(e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in events)\n    fn = sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in events)\n    fp = sum(not e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in events)\n    high_n = sum(g["prioridade"] == "alto" for g in gt)\n    high_missing = sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in events)\n    messages = [policy.gerar_mensagem(p, "medio") for p in pred_all if alert_on(p)]\n    if any(not isinstance(m, str) or not m.strip() for m in messages):\n        raise RuntimeError("A política produziu alerta sem texto.")\n    image = {"coorte": record["coorte"], "imagem": record["name"], "id": record["id"],\n             "modelo": model_name, "tp_alerta": tp, "fn_alerta": fn, "fp_alerta": fp,\n             "alta_ref": high_n, "alta_sem_alerta": high_missing,\n             "tp_detector": len(pairs), "fn_detector": len(missing), "fp_detector": len(extras),\n             "gt_avaliaveis": len(gt), "gt_ignorados": len(record["gt"])-len(gt),\n             "predicoes_ignoradas_conf_025": len(pred_all)-len(pred),\n             "cobertura_completa_alertas": known & ALERT_MASK == ALERT_MASK,\n             "ha_alerta_ref": any(alert_on(g) for g in gt),\n             "ha_alerta_pred": any(alert_on(p) for p in pred),\n             "mensagem_alerta": "\\n".join(messages)}\n    stats = (ap_matches(gt, pred_low_known),\n             np.array([p["confianca"] for p in pred_low_known], dtype=float),\n             np.array([p["classe_id"] for p in pred_low_known], dtype=int),\n             np.array([g["classe_id"] for g in gt], dtype=int))\n    return image, events, stats\n\n\ndef summarise(images, events, stats, ap_function=None):\n    if not images:\n        raise ValueError("Coorte de avaliação vazia.")\n    total = lambda k: sum(int(r[k]) for r in images)\n    output = {"imagens": len(images), "conf_alertas": .25,\n              "alertas_por_objeto": alert_metrics(total("tp_alerta"), total("fn_alerta"), total("fp_alerta")),\n              "alta_ref": total("alta_ref"), "alta_sem_alerta": total("alta_sem_alerta"),\n              "taxa_alta_sem_alerta": ratio(total("alta_sem_alerta"), total("alta_ref")),\n              "detector_conf_025": alert_metrics(total("tp_detector"), total("fn_detector"), total("fp_detector")),\n              "gt_ignorados_sem_cobertura": total("gt_ignorados"),\n              "predicoes_ignoradas_conf_025": total("predicoes_ignoradas_conf_025")}\n    complete = [r for r in images if r["cobertura_completa_alertas"]]\n    output["imagens_com_cobertura_completa_alertas"] = len(complete)\n    output["alertas_por_imagem_cobertura_completa"] = alert_metrics(\n        sum(r["ha_alerta_ref"] and r["ha_alerta_pred"] for r in complete),\n        sum(r["ha_alerta_ref"] and not r["ha_alerta_pred"] for r in complete),\n        sum(not r["ha_alerta_ref"] and r["ha_alerta_pred"] for r in complete))\n    correct = np.concatenate([s[0] for s in stats], axis=0)\n    scores = np.concatenate([s[1] for s in stats])\n    pred_cls = np.concatenate([s[2] for s in stats])\n    target_cls = np.concatenate([s[3] for s in stats])\n    ap = {}\n    if ap_function is not None and len(target_cls) and len(pred_cls):\n        result = ap_function(correct, scores, pred_cls, target_cls, plot=False,\n                             names=dict(enumerate(NAMES)))\n        values, ids = np.asarray(result[5]), np.asarray(result[6], dtype=int)\n        if values.shape != (len(ids), 10):\n            raise RuntimeError("Retorno de AP incompatível com a versão fixada.")\n        ap = {int(c): a for c, a in zip(ids, values)}\n    elif ap_function is not None and len(target_cls):\n        ap = {int(c): np.zeros(10) for c in np.unique(target_cls)}\n    output["map50_recorte"] = float(np.mean([a[0] for a in ap.values()])) if ap else None\n    output["map50_95_recorte"] = float(np.mean(list(ap.values()))) if ap else None\n    class_rows = []\n    for cid, name in enumerate(NAMES):\n        part = [e for e in events if e["classe_id"] == cid]\n        a = ap.get(cid)\n        n_gt = int((target_cls == cid).sum())\n        class_rows.append({"classe_id": cid, "classe": name, "gt_avaliaveis": n_gt,\n                           "ap50": float(a[0]) if a is not None else None,\n                           "ap50_95": float(a.mean()) if a is not None else None,\n                           "suporte_reduzido": n_gt < 30,\n                           **alert_metrics(sum(e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in part),\n                                           sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in part),\n                                           sum(not e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in part)),\n                           "alta_ref": sum(e["prioridade_ref"] == "alto" for e in part),\n                           "alta_sem_alerta": sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in part)})\n    return output, class_rows\n\n\ndef model_predictions(model, record, device):\n    if record["role"] not in {"home_val", "val_nova"}:\n        raise ValueError("Tentativa de inferência em reserva ou conjunto não autorizado.")\n    result = model.predict(str(record["image"]), conf=.001, imgsz=640, rect=True,\n                           iou=.70, max_det=300, device=device, half=False,\n                           verbose=False, save=False)[0]\n    if Path(result.path).resolve() != Path(record["image"]).resolve():\n        raise RuntimeError("A inferência retornou uma imagem diferente.")\n    if result.boxes is None:\n        return []\n    return [{"classe_id": int(c), "classe": NAMES[int(c)],\n             "bbox": [float(v) for v in box], "confianca": float(score)}\n            for c, box, score in zip(result.boxes.cls.cpu().numpy(),\n                                      result.boxes.xyxy.cpu().numpy(),\n                                      result.boxes.conf.cpu().numpy())]\n\n\ndef evaluate_model(model, records, model_name, output_dir, policy, matcher, device,\n                   ap_function):\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    images, events, stats = [], [], []\n    cache_file = output_dir / "predicoes_conf_0001.jsonl.gz"\n    with gzip.open(cache_file, "wt", encoding="utf-8") as cache:\n        for i, record in enumerate(records, 1):\n            pred = model_predictions(model, record, device)\n            cache.write(json.dumps({"id": record["id"], "imagem": record["name"],\n                                    "predicoes": pred}, ensure_ascii=False) + "\\n")\n            im, ev, st = analyse_prediction(record, pred, model_name, policy, matcher)\n            images.append(im)\n            events.extend(ev)\n            stats.append(st)\n            if i % 100 == 0 or i == len(records):\n                print(f"{record[\'coorte\']} / {model_name}: {i}/{len(records)}", flush=True)\n    summary, classes = summarise(images, events, stats, ap_function)\n    summary.update(modelo=model_name, coorte=records[0]["coorte"],\n                   protocolo="12 classes; AP por confiança; alertas por IoU; cobertura conservadora",\n                   nao_e_openimages_challenge=True, selecao_automatica_de_modelo=False)\n    save_json(output_dir / "metricas.json", summary)\n    save_csv(output_dir / "metricas_por_classe.csv", classes)\n    save_csv(output_dir / "alertas_por_imagem.csv", images)\n    save_csv(output_dir / "avaliacao_objetos.csv", events)\n    (output_dir / "alertas_texto.txt").write_text(\n        "\\n\\n".join(r["imagem"]+"\\n"+r["mensagem_alerta"] for r in images if r["mensagem_alerta"]),\n        encoding="utf-8")\n    return {"summary": summary, "classes": classes, "images": images, "events": events,\n            "cache_file": cache_file}\n', 'diag11_data.py': '"""Entradas, auditoria documental e nova validação, sem consulta a previsões."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport time\nimport urllib.request\nimport zipfile\nfrom collections import Counter, defaultdict\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nfrom scipy.fft import dctn\n\nfrom diag11_core import (ALL_KNOWN, ALERT_MASK, NAMES, canonical, parse_yolo,\n                         safe_member, save_csv, save_json, sha256_file)\n\nSOURCE_URLS = {\n    "classes": "https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions-boxable.csv",\n    "hierarquia": "https://storage.googleapis.com/openimages/2018_04/bbox_labels_600_hierarchy.json",\n    "caixas": "https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv",\n    "rotulos": "https://storage.googleapis.com/openimages/v5/validation-annotations-human-imagelabels-boxable.csv",\n    "metadata": "https://storage.googleapis.com/openimages/2018_04/validation/validation-images-with-rotation.csv",\n}\n\n\ndef download(url, path, retries=3):\n    """Download atômico, com recibo local e validação de tamanho quando informado."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    receipt = path.with_name(path.name + ".download.json")\n    if path.is_file() and receipt.is_file():\n        prior = json.loads(receipt.read_text())\n        if prior.get("url") == url and prior.get("sha256") == sha256_file(path):\n            return path\n    last = None\n    for attempt in range(retries):\n        tmp = path.with_name(path.name + ".tmp")\n        try:\n            req = urllib.request.Request(url, headers={"User-Agent": "TCC-Diagnostico-11/1.0"})\n            with urllib.request.urlopen(req, timeout=45) as response, tmp.open("wb") as out:\n                expected = response.headers.get("Content-Length")\n                for block in iter(lambda: response.read(2**20), b""):\n                    out.write(block)\n            if expected and tmp.stat().st_size != int(expected):\n                raise IOError("Tamanho recebido difere do Content-Length.")\n            if not tmp.stat().st_size:\n                raise IOError("Arquivo recebido vazio.")\n            tmp.replace(path)\n            save_json(receipt, {"url": url, "sha256": sha256_file(path),\n                                "bytes": path.stat().st_size})\n            return path\n        except Exception as exc:\n            last = type(exc).__name__ + ": " + str(exc)[:180]\n            tmp.unlink(missing_ok=True)\n            if attempt + 1 < retries:\n                time.sleep(min(2**attempt, 4))\n    raise IOError(f"Download não concluído: {url}; {last}")\n\n\ndef csv_rows(path):\n    with Path(path).open(encoding="utf-8-sig", newline="") as f:\n        yield from csv.DictReader(f)\n\n\ndef load_history(csv_path, json_path, expected_hash):\n    split = json.loads(Path(json_path).read_text())\n    stated = split.pop("divisao_sha256")\n    computed = hashlib.sha256(canonical(split).encode()).hexdigest()\n    if stated != expected_hash or computed != stated:\n        raise RuntimeError("Divisão histórica não corresponde à execução 10.")\n    rows = list(csv_rows(csv_path))\n    if len(rows) != 100000 or len({r["id"] for r in rows}) != 100000:\n        raise RuntimeError("O manifesto histórico precisa de 100.000 IDs distintos.")\n    if Counter(r["split"] for r in rows) != {"train": 80000, "test": 20000}:\n        raise RuntimeError("Divisão histórica diferente de 80/20.")\n    for label, key in [("train", "ids_treino"), ("test", "ids_teste")]:\n        if {r["id"] for r in rows if r["split"] == label} != set(split[key]):\n            raise RuntimeError("CSV e JSON históricos divergem.")\n    hashes = {r["imagem_sha256"] for r in rows}\n    if len(hashes) != len(rows):\n        raise RuntimeError("O histórico contém imagens repetidas em bytes.")\n    return {"ids": {r["id"] for r in rows}, "hashes": hashes, "rows": rows,\n            "split_hash": stated}\n\n\ndef phash(path):\n    with Image.open(path) as image:\n        small = np.asarray(image.convert("L").resize((32, 32), Image.Resampling.LANCZOS),\n                           dtype=float)\n    low = dctn(small, norm="ortho")[:8, :8]\n    bits = low > np.median(low.flatten()[1:])\n    bits[0, 0] = False\n    return int.from_bytes(np.packbits(bits.flatten()).tobytes(), "big")\n\n\nclass NearIndex:\n    """pHash de 64 bits, distância <=4; sinal de semelhança, não prova semântica."""\n    def __init__(self):\n        self.tables = [defaultdict(list) for _ in range(8)]\n\n    def add(self, value, name):\n        for band in range(8):\n            self.tables[band][(value >> (8*band)) & 255].append((value, name))\n\n    def find(self, value, max_distance=4):\n        checked = set()\n        for band in range(8):\n            for prior, name in self.tables[band].get((value >> (8*band)) & 255, []):\n                if (prior, name) in checked:\n                    continue\n                checked.add((prior, name))\n                if (prior ^ value).bit_count() <= max_distance:\n                    return name\n        return None\n\n\ndef extract_home(zip_path, destination, expected_zip, expected_test, expected_policy):\n    if sha256_file(zip_path) != expected_zip:\n        raise RuntimeError("ZIP Home diferente da entrada preservada.")\n    destination = Path(destination)\n    records, hashes, near = [], set(), NearIndex()\n    with zipfile.ZipFile(zip_path) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError("Há membros repetidos no ZIP Home.")\n        manifest = json.loads(z.read("verificacao_preparacao.json"))\n        if (manifest["test"]["sha256"] != expected_test or\n                manifest["test"]["imagens"] != 403 or\n                {int(k): v for k, v in manifest["classes"].items()} != dict(enumerate(NAMES))):\n            raise RuntimeError("Manifesto Home incompatível.")\n        root = manifest["dataset_dir"]\n        items = {r["arquivo"]: r for r in manifest["arquivos"]}\n        if len(items) != len(manifest["arquivos"]):\n            raise RuntimeError("Manifesto Home com membros repetidos.")\n        for name in z.namelist():\n            safe_member(name)\n            if name.startswith("codigo/") and name.endswith(".py"):\n                raw = z.read(name)\n                if name in items and hashlib.sha256(raw).hexdigest() != items[name]["sha256"]:\n                    raise RuntimeError("Código Home diferente do manifesto.")\n                out = destination / name\n                out.parent.mkdir(parents=True, exist_ok=True)\n                out.write_bytes(raw)\n        for name, item in sorted(items.items()):\n            is_image = (name.startswith(f"{root}/images/") and\n                        Path(name).suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"})\n            if is_image:\n                hashes.add(item["sha256"])\n            role = None\n            if name.startswith(f"{root}/images/train/raw_train/"):\n                role = "home_train"\n            elif name.startswith(f"{root}/images/val/raw_train/"):\n                role = "home_val"\n            if not is_image or role is None:\n                continue\n            raw = z.read(name)\n            if hashlib.sha256(raw).hexdigest() != item["sha256"]:\n                raise RuntimeError(f"Imagem Home alterada: {name}")\n            image = destination / name\n            image.parent.mkdir(parents=True, exist_ok=True)\n            image.write_bytes(raw)\n            label_name = name.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"\n            if label_name not in items:\n                raise RuntimeError("Label Home ausente do manifesto.")\n            label_raw = z.read(label_name)\n            if hashlib.sha256(label_raw).hexdigest() != items[label_name]["sha256"]:\n                raise RuntimeError("Label Home alterado.")\n            with Image.open(image) as im:\n                width, height = im.size\n                im.verify()\n            gt = parse_yolo(label_raw, width, height, source=label_name)\n            record = {"id": "home:" + name, "name": name.split("/raw_train/")[-1],\n                      "image": image, "role": role, "coorte": "home_val_historica",\n                      "width": width, "height": height, "known_mask": ALL_KNOWN,\n                      "sha256": item["sha256"], "label_sha256": items[label_name]["sha256"],\n                      "gt": gt}\n            record["phash"] = phash(image)\n            near.add(record["phash"], name)\n            records.append(record)\n    counts = Counter(r["role"] for r in records)\n    if counts != {"home_train": 1824, "home_val": 456}:\n        raise RuntimeError(f"Home inesperado: {dict(counts)}")\n    train_hash = {r["sha256"] for r in records if r["role"] == "home_train"}\n    val_hash = {r["sha256"] for r in records if r["role"] == "home_val"}\n    if train_hash & val_hash:\n        raise RuntimeError("Home TRAIN e VAL atuais compartilham imagens em bytes.")\n    policy_file = destination / "codigo/src/politica_alertas.py"\n    if not policy_file.is_file() or sha256_file(policy_file) != expected_policy:\n        raise RuntimeError("Política V2 diferente da versão preservada.")\n    audit = {"home_train_atual": 1824, "home_val_atual": 456,\n             "test_home_preservado": 403, "test_extraido": False,\n             "intersecao_bytes_home_train_val_atual": 0,\n             "divergencia_documental_1822_1824": "Não reconciliada; o pacote atual tem 1824.",\n             "separacao_atual_nao_comprova_historico_baseline": True}\n    return records, hashes, near, audit\n\n\ndef audit_baseline_metadata(checkpoint, expected_hash, home_records, evidence=None):\n    if sha256_file(checkpoint) != expected_hash:\n        raise RuntimeError("Peso baseline diferente da referência.")\n    # Só o checkpoint autenticado pelo SHA esperado é desserializado.\n    import torch\n    obj = torch.load(checkpoint, map_location="cpu", weights_only=False)\n    args = obj.get("train_args", {})\n    fields = ["data", "epochs", "batch", "imgsz", "seed", "optimizer", "lr0",\n              "project", "name", "fraction"]\n    audit = {"modelo_sha256": expected_hash, "data_checkpoint": str(obj.get("date")),\n             "train_args_preservados": {k: args[k] for k in fields if k in args},\n             "estado_proveniencia_original": "nao_determinado",\n             "home_val_usada_no_treino_original": None,\n             "motivo": "Caminhos e argumentos não contêm a lista de imagens vistas no treino original."}\n    del obj\n    if evidence is not None:\n        required = {"modelo_sha256", "train_image_sha256", "val_image_sha256"}\n        if not required <= evidence.keys() or evidence["modelo_sha256"] != expected_hash:\n            audit["evidencia_original"] = "incompativel_ou_sem_vinculo_ao_peso"\n        else:\n            train = set(evidence["train_image_sha256"])\n            val = set(evidence["val_image_sha256"])\n            valid_sha = lambda x: isinstance(x, str) and len(x) == 64 and all(c in "0123456789abcdef" for c in x)\n            if not train or any(not valid_sha(h) for h in train | val):\n                raise ValueError("Manifesto original precisa de hashes SHA-256 válidos.")\n            current = {r["sha256"] for r in home_records if r["role"] == "home_val"}\n            overlap = train & current\n            complete = evidence.get("manifesto_treino_completo") is True\n            audit.update(estado_proveniencia_original="manifesto_documental_analisado",\n                         evidencia_original="documento_vinculado_ao_sha_do_modelo",\n                         home_val_usada_no_treino_original=True if overlap else (False if complete else None),\n                         manifesto_declarado_completo=complete,\n                         imagens_home_val_no_manifesto_train_original=len(overlap),\n                         imagens_home_val_no_manifesto_val_original=len(current & val),\n                         motivo="Conclusão limitada à completude e correção do manifesto apresentado.")\n    return audit\n\n\ndef stable_order(iid, seed=42):\n    return hashlib.sha256(f"{seed}|{iid}".encode()).hexdigest(), iid\n\n\ndef balanced_ids(records, n, seed=42, per_class=45):\n    """Diversidade por classe e preenchimento determinístico, sem usar previsões."""\n    if len(records) < n:\n        raise ValueError(f"Há somente {len(records)} imagens elegíveis; exigidas {n}.")\n    groups = {c: sorted((iid for iid, r in records.items() if\n                         r["positive_mask"] & (1 << c)), key=lambda x: stable_order(x, seed))\n              for c in range(12)}\n    order = sorted(groups, key=lambda c: (len(groups[c]), c))\n    chosen, counts = set(), Counter()\n    # Diversificar exemplos negativos quando as seis classes têm ausência verificada.\n    negatives = sorted((iid for iid, r in records.items() if\n                        r["known_mask"] & ALERT_MASK == ALERT_MASK and\n                        not r["positive_mask"] & ALERT_MASK), key=lambda x: stable_order(x, seed))\n    for iid in negatives[:min(len(negatives), n//10)]:\n        chosen.add(iid)\n        counts.update(c for c in range(12) if records[iid]["positive_mask"] & (1 << c))\n    for target in range(1, per_class+1):\n        for cid in order:\n            if len(chosen) >= n:\n                break\n            if counts[cid] >= target:\n                continue\n            for iid in groups[cid]:\n                if iid not in chosen:\n                    chosen.add(iid)\n                    counts.update(c for c in range(12) if records[iid]["positive_mask"] & (1 << c))\n                    break\n    for iid in sorted(records, key=lambda x: stable_order(x, seed)):\n        if len(chosen) == n:\n            break\n        chosen.add(iid)\n    return sorted(chosen, key=lambda x: stable_order(x, seed))\n\n\ndef split_pool(records, n_reserve, seed=42):\n    """Estratos pela classe positiva mais rara; a reserva não entra na inferência."""\n    frequencies = Counter(c for r in records.values() for c in range(12)\n                          if r["positive_mask"] & (1 << c))\n    groups = defaultdict(list)\n    for iid, record in records.items():\n        classes = record["classes_positivas_avaliaveis"]\n        primary = min(classes, key=lambda c: (frequencies[c], c)) if classes else -1\n        groups[primary].append(iid)\n    total = len(records)\n    if not 0 < n_reserve < total:\n        raise ValueError("A reserva precisa ser menor que o conjunto novo.")\n    quotas = {c: len(ids)*n_reserve//total for c, ids in groups.items()}\n    order = sorted(groups, key=lambda c: (-(len(groups[c])*n_reserve % total), c))\n    for cid in order[:n_reserve-sum(quotas.values())]:\n        quotas[cid] += 1\n    selected = set()\n    for cid, ids in groups.items():\n        selected.update(sorted(ids, key=lambda x: stable_order(x, seed+1))[:quotas[cid]])\n    if len(selected) != n_reserve:\n        raise RuntimeError("A divisão não produziu a reserva planejada.")\n    return {iid: ("reserva_futura" if iid in selected else "val_nova") for iid in records}\n\n\ndef filter_metadata(records, metadata_path, prior_ids):\n    metadata, exclusions = {}, Counter()\n    for row in csv_rows(metadata_path):\n        iid = row["ImageID"]\n        if iid not in records:\n            continue\n        if row["Subset"] != "validation":\n            raise ValueError("A metadata contém uma origem diferente de validation.")\n        if iid in prior_ids:\n            exclusions["id_presente_nos_100k"] += 1\n            continue\n        try:\n            rotation = float(row["Rotation"])\n        except (TypeError, ValueError):\n            rotation = float("nan")\n        # Evita suposições sobre transformação das caixas e orientação do JPEG.\n        if rotation != 0:\n            exclusions["rotacao_nao_zero_ou_desconhecida"] += 1\n            continue\n        metadata[iid] = {k: row.get(k, "") for k in\n                         ["ImageID", "OriginalLandingURL", "License", "Author", "Rotation"]}\n    result = {iid: {**r, "creditos": metadata[iid]} for iid, r in records.items()\n              if iid in metadata}\n    return result, dict(exclusions)\n\n\ndef download_images(candidates, ids, directory, workers=8):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n\n    def one(iid):\n        record = dict(candidates[iid])\n        path = directory / f"oi_val_{iid}.jpg"\n        try:\n            download(f"https://open-images-dataset.s3.amazonaws.com/validation/{iid}.jpg", path)\n            with Image.open(path) as image:\n                image.verify()\n            with Image.open(path) as image:\n                width, height = image.size\n                orientation = image.getexif().get(274, 1)\n                if width < 32 or height < 32 or orientation != 1:\n                    raise ValueError("JPEG pequeno ou com orientação EXIF não trivial.")\n                image.load()\n            tiny = {g["classe_id"] for g in record["gt_normalizado"] if\n                    (g["bbox_norm"][2]-g["bbox_norm"][0])*width < 1 or\n                    (g["bbox_norm"][3]-g["bbox_norm"][1])*height < 1}\n            # A avaliação da ausência fica desconhecida para a classe afetada.\n            record["restricoes_cobertura"] = dict(record["restricoes_cobertura"])\n            for cid in tiny:\n                for key in ["known_mask", "positive_mask", "negative_mask"]:\n                    record[key] &= ~(1 << cid)\n                record["restricoes_cobertura"][str(cid)] = ["caixa_menor_que_um_pixel"]\n            record["classes_positivas_avaliaveis"] = [c for c in range(12)\n                                                      if record["positive_mask"] & (1 << c)]\n            if not (record["positive_mask"] or record["known_mask"] & ALERT_MASK == ALERT_MASK):\n                raise ValueError("Nenhuma cobertura útil após conferir as dimensões reais.")\n            record.update(image=path, width=width, height=height, sha256=sha256_file(path),\n                          phash=phash(path), name=path.name)\n            return record, None\n        except Exception as exc:\n            return None, {"id": iid, "motivo": type(exc).__name__, "detalhe": str(exc)[:240]}\n\n    valid, failures = {}, []\n    with ThreadPoolExecutor(max_workers=workers) as executor:\n        for done, (record, failure) in enumerate(executor.map(one, ids), 1):\n            if record:\n                valid[record["id"]] = record\n            else:\n                failures.append(failure)\n            if done % 100 == 0 or done == len(ids):\n                print(f"Novas imagens: {done}/{len(ids)}; válidas: {len(valid)}", flush=True)\n    return valid, failures\n\n\ndef deduplicate(records, prior_hashes, home_near, seed=42):\n    accepted, excluded, seen_hash, near = {}, [], set(), NearIndex()\n    for iid in sorted(records, key=lambda x: stable_order(x, seed)):\n        record = records[iid]\n        reason, similar = None, None\n        if record["sha256"] in prior_hashes:\n            reason = "bytes_presentes_nos_dados_anteriores"\n        elif record["sha256"] in seen_hash:\n            reason = "bytes_repetidos_no_conjunto_novo"\n        elif (similar := home_near.find(record["phash"])) is not None:\n            reason = "semelhanca_phash_com_home_train_ou_val"\n        elif (similar := near.find(record["phash"])) is not None:\n            reason = "semelhanca_phash_dentro_do_conjunto_novo"\n        if reason:\n            excluded.append({"id": iid, "motivo": reason, "semelhante_a": similar})\n            continue\n        accepted[iid] = record\n        seen_hash.add(record["sha256"])\n        near.add(record["phash"], iid)\n    return accepted, excluded\n\n\ndef freeze_dataset(records, roles, directory, source_hashes, prior_split_hash, seed=42):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    manifest, runtime = [], []\n    for iid in sorted(roles):\n        r, role = records[iid], roles[iid]\n        boxes = []\n        for g in r["gt_normalizado"]:\n            x1, y1, x2, y2 = g["bbox_norm"]\n            boxes.append({**g, "bbox": [x1*r["width"], y1*r["height"],\n                                       x2*r["width"], y2*r["height"]]})\n        coverage = {k: r[k] for k in ["known_mask", "positive_mask", "negative_mask",\n                                      "restricoes_cobertura"]}\n        label = directory / "labels" / role / f"oi_val_{iid}.txt"\n        coverage_file = directory / "coverage" / role / f"oi_val_{iid}.json"\n        label.parent.mkdir(parents=True, exist_ok=True)\n        lines = []\n        for g in r["gt_normalizado"]:\n            x1, y1, x2, y2 = g["bbox_norm"]\n            lines.append(f"{g[\'classe_id\']} {(x1+x2)/2:.10f} {(y1+y2)/2:.10f} {(x2-x1):.10f} {(y2-y1):.10f}")\n        label.write_text("\\n".join(lines)+("\\n" if lines else ""), encoding="utf-8")\n        save_json(coverage_file, coverage)\n        image_rel = f"images/{role}/oi_val_{iid}.jpg"\n        item = {"id": iid, "role": role, "image": image_rel,\n                "label": label.relative_to(directory).as_posix(),\n                "coverage": coverage_file.relative_to(directory).as_posix(),\n                "sha256": r["sha256"], "label_sha256": sha256_file(label),\n                "coverage_sha256": sha256_file(coverage_file),\n                "width": r["width"], "height": r["height"], "known_mask": r["known_mask"],\n                "positive_mask": r["positive_mask"], "negative_mask": r["negative_mask"],\n                "phash": f"{r[\'phash\']:016x}", "creditos": r["creditos"],\n                "gt_normalizado": r["gt_normalizado"]}\n        manifest.append(item)\n        runtime.append({**r, "role": role, "coorte": "openimages_val_nova",\n                        "gt": boxes, "label": label, "coverage": coverage_file})\n    metadata = {"versao": "11-validacao-independente-v1", "seed": seed, "nomes": NAMES,\n                "fontes_metadata_sha256": source_hashes, "origem_split": "validation",\n                "divisao_100k_anterior_sha256": prior_split_hash,\n                "contagens": dict(Counter(roles.values())),\n                "criterio": "Diversidade de classes, hash determinístico e pHash; sem previsões",\n                "cobertura": "Positivo explícito na raiz ou negativo na raiz/ancestral; classes problemáticas excluídas",\n                "independencia_limitada_ao_historico_conhecido": True,\n                "pretreinamentos_publicos_nao_auditados": True,\n                "reserva_futura_avaliada": False, "arquivos": manifest}\n    dataset_hash = hashlib.sha256(canonical(metadata).encode()).hexdigest()\n    metadata["dataset_sha256"] = dataset_hash\n    save_json(directory / "manifesto_validacao_nova.json", metadata)\n    save_csv(directory / "divisao.csv", [{k: r[k] for k in\n                                        ["id", "role", "sha256", "known_mask", "positive_mask"]}\n                                       for r in manifest])\n    archive = directory.parent / f"validacao_independente_11_{dataset_hash[:12]}.zip"\n    # Os JPEGs permanecem sem recompressão.\n    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_STORED) as z:\n        z.write(directory / "manifesto_validacao_nova.json", "manifesto_validacao_nova.json")\n        z.write(directory / "divisao.csv", "divisao.csv")\n        for r, item in zip(sorted(runtime, key=lambda r: r["id"]), manifest):\n            z.write(r["image"], item["image"])\n            z.write(r["label"], item["label"])\n            z.write(r["coverage"], item["coverage"])\n    return runtime, metadata, archive\n\n\ndef support_rows(records):\n    rows = []\n    for role in sorted({r["role"] for r in records}):\n        group = [r for r in records if r["role"] == role]\n        for cid, name in enumerate(NAMES):\n            gt = [g for r in group for g in r["gt"] if\n                  g["classe_id"] == cid and r["known_mask"] & (1 << cid)]\n            rows.append({"conjunto": role, "classe_id": cid, "classe": name,\n                         "imagens_cobertura_conhecida": sum(bool(r["known_mask"] & (1 << cid)) for r in group),\n                         "imagens_positivas_avaliaveis": sum(bool(r["positive_mask"] & (1 << cid)) for r in group),\n                         "caixas_avaliaveis": len(gt), "suporte_reduzido": len(gt) < 30})\n    return rows\n', 'diag11_reports.py': '"""Comparação pareada, indícios dos erros e galeria de inspeção."""\nfrom __future__ import annotations\n\nimport gzip\nimport html\nimport json\nimport zipfile\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont\n\nfrom diag11_core import (NAMES, LEVEL, alert_on, canonical, ratio, save_csv,\n                         save_json, with_policy)\nfrom diag11_data import stable_order\n\n\ndef paired_bootstrap(old_images, new_images, repetitions=1000, seed=42):\n    old = {r["id"]: r for r in old_images}\n    new = {r["id"]: r for r in new_images}\n    if set(old) != set(new) or len(old) != len(old_images) or len(new) != len(new_images):\n        raise ValueError("Bootstrap exige as mesmas imagens, sem repetições.")\n    ids = sorted(old)\n    fields = ["tp_alerta", "fn_alerta", "fp_alerta", "alta_sem_alerta", "alta_ref"]\n    a = np.array([[old[i][k] for k in fields] for i in ids], dtype=float)\n    b = np.array([[new[i][k] for k in fields] for i in ids], dtype=float)\n\n    def scores(x):\n        tp, fn, fp, missing, high = x\n        return [100*ratio(tp, tp+fp) if tp+fp else np.nan,\n                100*ratio(tp, tp+fn) if tp+fn else np.nan,\n                100*ratio(2*tp, 2*tp+fp+fn) if 2*tp+fp+fn else np.nan,\n                fn, fp, missing, 100*ratio(missing, high) if high else np.nan]\n\n    point = np.array(scores(b.sum(axis=0))) - np.array(scores(a.sum(axis=0)))\n    rng = np.random.default_rng(seed)\n    results = []\n    for _ in range(repetitions):\n        index = rng.integers(0, len(ids), len(ids))\n        results.append(np.array(scores(b[index].sum(axis=0))) -\n                       np.array(scores(a[index].sum(axis=0))))\n    samples = np.asarray(results)\n    labels = ["precision_alertas_pp", "recall_alertas_pp", "f1_alertas_pp",\n              "fn_alertas", "fp_alertas", "alta_sem_alerta", "taxa_alta_sem_alerta_pp"]\n    rows = []\n    for j, name in enumerate(labels):\n        values = samples[np.isfinite(samples[:, j]), j]\n        low, high = np.quantile(values, [.025, .975]) if len(values) else (None, None)\n        rows.append({"metrica": name, "delta_candidato_menos_antigo": float(point[j]) if np.isfinite(point[j]) else None,\n                     "ic95_inferior": low, "ic95_superior": high,\n                     "reamostragens_validas": len(values), "imagens": len(ids),\n                     "metodo": "Bootstrap pareado por imagem; não corrige possível exposição prévia"})\n    return rows\n\n\ndef comparison_rows(evaluations):\n    rows = []\n    for cohort, models in evaluations.items():\n        for name, result in models.items():\n            s, a = result["summary"], result["summary"]["alertas_por_objeto"]\n            rows.append({"coorte": cohort, "modelo": name, "imagens": s["imagens"],\n                         "precision_alertas": a["precision"], "recall_alertas": a["recall"],\n                         "f1_alertas": a["f1"], "tp_alerta": a["tp"], "fn_alerta": a["fn"],\n                         "fp_alerta": a["fp"], "alta_ref": s["alta_ref"],\n                         "alta_sem_alerta": s["alta_sem_alerta"],\n                         "map50_recorte": s["map50_recorte"],\n                         "map50_95_recorte": s["map50_95_recorte"]})\n    return rows\n\n\ndef diagnosis_rows(evaluations):\n    rows = []\n    for cohort, models in evaluations.items():\n        for model, result in models.items():\n            events = result["events"]\n            for cid, name in enumerate(NAMES):\n                group = [e for e in events if e["classe_id"] == cid]\n                for indication in sorted({x for e in group for x in e["indicios"].split("|") if x}):\n                    part = [e for e in group if indication in e["indicios"].split("|")]\n                    rows.append({"coorte": cohort, "modelo": model, "classe": name,\n                                 "indicio": indication, "eventos": len(part),\n                                 "fn_alerta": sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in part),\n                                 "alta_sem_alerta": sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in part),\n                                 "observacao": "Indícios podem se sobrepor; não estabelecem a causa do erro."})\n    return rows\n\n\ndef load_prediction_cache(path):\n    records = {}\n    with gzip.open(path, "rt", encoding="utf-8") as f:\n        for line in f:\n            r = json.loads(line)\n            if r["id"] in records:\n                raise ValueError("Cache tem imagem repetida.")\n            records[r["id"]] = r["predicoes"]\n    return records\n\n\ndef select_gallery_cases(old, new, limit=24, seed=42):\n    a, b = {r["id"]: r for r in old}, {r["id"]: r for r in new}\n    if set(a) != set(b):\n        raise ValueError("Galeria exige os dois modelos nas mesmas imagens.")\n    chosen = {}\n\n    def take(label, ordered, number):\n        used = 0\n        for iid in ordered:\n            if iid not in chosen and len(chosen) < limit:\n                chosen[iid] = label\n                used += 1\n                if used == number:\n                    break\n\n    worse = sorted(a, key=lambda i: (-(b[i]["alta_sem_alerta"]-a[i]["alta_sem_alerta"]),\n                                     -(b[i]["fn_alerta"]-a[i]["fn_alerta"]), stable_order(i, seed)))\n    better = sorted(a, key=lambda i: (b[i]["alta_sem_alerta"]-a[i]["alta_sem_alerta"],\n                                      b[i]["fn_alerta"]-a[i]["fn_alerta"], stable_order(i, seed)))\n    false = sorted(a, key=lambda i: (-(a[i]["fp_alerta"]+b[i]["fp_alerta"]), stable_order(i, seed)))\n    take("Candidato com mais omissões", [i for i in worse if\n         (b[i]["alta_sem_alerta"], b[i]["fn_alerta"]) >\n         (a[i]["alta_sem_alerta"], a[i]["fn_alerta"])], limit//4)\n    take("Candidato com menos omissões", [i for i in better if\n         (b[i]["alta_sem_alerta"], b[i]["fn_alerta"]) <\n         (a[i]["alta_sem_alerta"], a[i]["fn_alerta"])], limit//4)\n    take("Falsos alertas avaliáveis", [i for i in false if a[i]["fp_alerta"]+b[i]["fp_alerta"]], limit//4)\n    take("Amostra por hash, independente do erro", sorted(a, key=lambda i: stable_order(i, seed)), limit)\n    return [{"id": i, "categoria": label} for i, label in chosen.items()]\n\n\ndef font(size):\n    candidates = ["/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",\n                  "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf"]\n    for path in candidates:\n        if Path(path).is_file():\n            return ImageFont.truetype(path, size)\n    return ImageFont.load_default()\n\n\ndef draw_panel(record, objects, title):\n    canvas = Image.new("RGB", (540, 470), "#101a28")\n    draw = ImageDraw.Draw(canvas)\n    draw.text((12, 10), title, fill="#f5f7fa", font=font(18))\n    with Image.open(record["image"]) as image:\n        image = image.convert("RGB")\n        scale = min(520/image.width, 390/image.height)\n        image = image.resize((max(1, round(image.width*scale)),\n                              max(1, round(image.height*scale))), Image.Resampling.LANCZOS)\n    x0, y0 = (540-image.width)//2, 46+(390-image.height)//2\n    canvas.paste(image, (x0, y0))\n    sx, sy = image.width/record["width"], image.height/record["height"]\n    colours = {"alto": "#ff7b8a", "medio": "#ffde70",\n               "baixo": "#6ee7b7", "sem_prioridade": "#adbac7"}\n    for obj in objects:\n        cid = obj["classe_id"]\n        known = bool(record["known_mask"] & (1 << cid))\n        colour = colours[obj["prioridade"]] if known else "#a9afba"\n        x1, y1, x2, y2 = obj["bbox"]\n        box = [x0+x1*sx, y0+y1*sy, x0+x2*sx, y0+y2*sy]\n        draw.rectangle(box, outline=colour, width=2)\n        label = obj["classe"]\n        if "confianca" in obj:\n            label += f" {obj[\'confianca\']:.2f}"\n        label += " / " + obj["prioridade"] if known else " / sem cobertura"\n        # Limita o rótulo à largura visível; não altera a caixa nem as métricas.\n        tx, ty = max(0, min(box[0], 535)), max(44, box[1]-15)\n        text_box = draw.textbbox((tx, ty), label, font=font(11))\n        draw.rectangle(text_box, fill="#101a28")\n        draw.text((tx, ty), label, fill=colour, font=font(11))\n    draw.text((12, 445), "Rosa: alta | amarelo: média | verde: baixa | cinza: sem referência",\n              fill="#c6ced9", font=font(10))\n    return canvas\n\n\ndef render_case(record, old_pred, new_pred, policy, destination):\n    gt = with_policy([g for g in record["gt"] if record["known_mask"] & (1 << g["classe_id"])],\n                     record["width"], record["height"], policy)\n    old = with_policy([p for p in old_pred if p["confianca"] >= .25],\n                      record["width"], record["height"], policy)\n    new = with_policy([p for p in new_pred if p["confianca"] >= .25],\n                      record["width"], record["height"], policy)\n    panels = [draw_panel(record, gt, "Referência V2 avaliável"),\n              draw_panel(record, old, "Modelo antigo / confiança 0,25"),\n              draw_panel(record, new, "Candidato / confiança 0,25")]\n    result = Image.new("RGB", (1620, 470), "#101a28")\n    for i, panel in enumerate(panels):\n        result.paste(panel, (540*i, 0))\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    result.save(destination, quality=88)\n    return destination\n\n\ndef format_percent(value):\n    return "sem suporte" if value is None else f"{100*value:.2f}%"\n\n\ndef build_report(output_dir, evaluations, cohorts, audit, bootstrap, policy, dataset_manifest, seed=42):\n    output_dir = Path(output_dir)\n    comparison = comparison_rows(evaluations)\n    diagnosis = diagnosis_rows(evaluations)\n    save_csv(output_dir / "comparacao_modelos.csv", comparison)\n    save_csv(output_dir / "diagnostico_erros.csv", diagnosis)\n    save_csv(output_dir / "bootstrap_pareado.csv", bootstrap)\n    gallery = []\n    for cohort, models in evaluations.items():\n        records = {r["id"]: r for r in cohorts[cohort]}\n        caches = {name: load_prediction_cache(result["cache_file"]) for name, result in models.items()}\n        cases = select_gallery_cases(models["antigo"]["images"], models["candidato"]["images"], seed=seed)\n        old_images = {r["id"]: r for r in models["antigo"]["images"]}\n        new_images = {r["id"]: r for r in models["candidato"]["images"]}\n        for i, case in enumerate(cases):\n            record = records[case["id"]]\n            relative = f"galeria/{cohort}_{i:03}.jpg"\n            render_case(record, caches["antigo"][record["id"]],\n                        caches["candidato"][record["id"]], policy, output_dir/relative)\n            gallery.append({**case, "coorte": cohort, "imagem": record["name"], "arquivo": relative,\n                            "antigo": {k: old_images[record["id"]][k] for k in\n                                       ["tp_alerta", "fn_alerta", "fp_alerta", "alta_sem_alerta"]},\n                            "candidato": {k: new_images[record["id"]][k] for k in\n                                          ["tp_alerta", "fn_alerta", "fp_alerta", "alta_sem_alerta"]},\n                            "creditos": record.get("creditos", {})})\n    save_json(output_dir / "indice_galeria.json", gallery)\n    new_records = cohorts["openimages_val_nova"]\n    save_csv(output_dir / "revisao_cenarios.csv", [\n        {"id": r["id"], "imagem": r["name"], "ambiente_interno": "nao_revisado",\n         "proximo_do_uso_pretendido": "nao_revisado", "anotacao_parece_correta": "nao_revisado",\n         "observacao": "", "origem": r.get("creditos", {}).get("OriginalLandingURL", "")}\n        for r in new_records])\n    provenance = audit["baseline_original"]["estado_proveniencia_original"]\n    table = "<table><thead><tr><th>Conjunto</th><th>Modelo</th><th>Precisão alertas</th><th>Recall alertas</th><th>F1</th><th>FN</th><th>Alta sem alerta</th></tr></thead><tbody>"\n    for r in comparison:\n        table += "<tr>" + "".join("<td>"+html.escape(str(x))+"</td>" for x in\n                     [r["coorte"], r["modelo"], format_percent(r["precision_alertas"]),\n                      format_percent(r["recall_alertas"]), format_percent(r["f1_alertas"]),\n                      r["fn_alerta"], r["alta_sem_alerta"]]) + "</tr>"\n    table += "</tbody></table>"\n    cards = []\n    for case in gallery:\n        credit = case["creditos"]\n        source_url = str(credit.get("OriginalLandingURL", ""))\n        license_url = str(credit.get("License", ""))\n        attribution = html.escape(str(credit.get("Author", "")))\n        if source_url.startswith(("https://", "http://")):\n            attribution += \' · <a href="\'+html.escape(source_url, quote=True)+\'">Fonte original</a>\'\n        if license_url.startswith(("https://", "http://")):\n            attribution += \' · <a href="\'+html.escape(license_url, quote=True)+\'">Licença</a>\'\n        details = (f"Antigo: FN {case[\'antigo\'][\'fn_alerta\']}, FP {case[\'antigo\'][\'fp_alerta\']}, "\n                   f"alta sem alerta {case[\'antigo\'][\'alta_sem_alerta\']}. "\n                   f"Candidato: FN {case[\'candidato\'][\'fn_alerta\']}, FP {case[\'candidato\'][\'fp_alerta\']}, "\n                   f"alta sem alerta {case[\'candidato\'][\'alta_sem_alerta\']}.")\n        cards.append("<article><h3>"+html.escape(case["coorte"]+" / "+case["categoria"])+\n                     "</h3><p>"+html.escape(case["imagem"]+" — "+details)+"</p>"+\n                     \'<img loading="lazy" src="\'+case["arquivo"]+\'" alt="Caixas de referência e dos dois modelos">\'+\n                     "<p class=small>"+attribution+\n                     " · Sobreposição de caixas para avaliação.</p></article>")\n    html_report = """<!doctype html><html lang="pt-BR"><meta charset="utf-8">\n<meta name="viewport" content="width=device-width,initial-scale=1">\n<title>TCC — Diagnóstico dos detectores</title><style>\nbody{font:16px/1.6 system-ui,sans-serif;background:#f3f5f7;color:#142135;margin:0}\nmain{max-width:1400px;margin:auto;padding:32px}h1{font-size:32px;line-height:1.2}\n.note,article,table{background:white;border:1px solid #dbe1e8;border-radius:12px}\n.note,article{padding:20px;margin:20px 0}table{border-collapse:collapse;width:100%}\ntd,th{padding:10px;text-align:left;border-bottom:1px solid #e6eaf0}th{background:#e8eff8}\nimg{max-width:100%;height:auto}.small{font-size:12px;color:#526174}.scroll{overflow:auto}\n</style><main><h1>Diagnóstico comparativo dos detectores</h1>\n<p>Pesos congelados · 12 classes · política V2 · confiança 0,25 · sem treinamento</p>\n<div class=note><strong>Como interpretar</strong><p>\nA Home VAL é histórica e exploratória. A validação externa é nova em relação aos IDs\ne bytes dos dados conhecidos do projeto; pré-treinamentos públicos não foram auditados.\nO recorte Open Images tem cobertura parcial e não garante cenas internas.\nAs métricas de alertas medem concordância com a V2, não risco físico independente.\nA reserva futura não foi inferida. Nenhum modelo foi promovido automaticamente.</p>\n<p>Proveniência documental do baseline: <strong>""" + html.escape(provenance) + """</strong>.</p></div>\n<h2>Alertas por objeto avaliável</h2><div class=scroll>""" + table + """</div>\n<p>Os CSVs trazem suporte por classe, AP no recorte, contagens e intervalos pareados.\nNão compare APs de protocolos e distribuições diferentes como se fossem o mesmo benchmark.</p>\n<h2>Casos para inspeção</h2><p>Os casos abaixo priorizam diferenças e erros, além de uma amostra por hash.\nEsta galeria não é uma estimativa da frequência dos erros. As contagens acima usam todas as imagens.\nObjetos sem cobertura aparecem em cinza e não são classificados automaticamente como falsos positivos.</p>\n""" + "".join(cards) + """<h2>Próxima decisão</h2><p>\nRevisar os cenários e os indícios dos erros; só depois definir o treino de controle.\nOs pesos antigos, o candidato e os testes anteriores permanecem preservados.</p></main></html>"""\n    (output_dir / "relatorio_diagnostico.html").write_text(html_report, encoding="utf-8")\n    lines = ["# Diagnóstico 11", "", "Comparação sem novo treinamento e sem seleção automática.",\n             "", f"Proveniência original do baseline: {provenance}.",\n             f"Validação externa: {len(new_records)} imagens.",\n             f"Reserva futura não inferida: {dataset_manifest[\'contagens\'].get(\'reserva_futura\', 0)} imagens.",\n             "", "| Conjunto | Modelo | Precisão alertas | Recall alertas | F1 | FN | Alta sem alerta |",\n             "|---|---|---:|---:|---:|---:|---:|"]\n    for r in comparison:\n        lines.append("| "+" | ".join(map(str, [r["coorte"], r["modelo"],\n                     format_percent(r["precision_alertas"]), format_percent(r["recall_alertas"]),\n                     format_percent(r["f1_alertas"]), r["fn_alerta"], r["alta_sem_alerta"]]))+" |")\n    lines += ["", "A nova cobertura é conservadora e a seleção busca diversidade de classes; não representa a frequência natural do uso.",\n              "A Home VAL continua exploratória enquanto a origem do primeiro treinamento não estiver esclarecida.",\n              "Semelhança por pHash é um filtro de suspeitas, não uma prova de identidade de cena.",\n              "Os indícios de erro podem se sobrepor e não identificam causas de forma conclusiva.",\n              "A reserva atual não foi inferida; não há avaliação independente de risco físico.",\n              "", "Veja relatorio_diagnostico.html para a galeria e bootstrap_pareado.csv para a incerteza das diferenças."]\n    (output_dir / "relatorio_diagnostico.md").write_text("\\n".join(lines)+"\\n", encoding="utf-8")\n    return comparison, diagnosis, gallery\n\n\ndef package_reports(output_dir):\n    output_dir = Path(output_dir)\n    archive = output_dir / "pacote_diagnostico_11.zip"\n    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=4) as z:\n        for path in sorted(output_dir.rglob("*")):\n            if path.is_file() and path != archive and not path.name.endswith(".tmp"):\n                z.write(path, path.relative_to(output_dir))\n    return archive\n', 'exp10_partial.py': '"""Experimento 09: caixas humanas parciais, fontes equilibradas e Home VAL."""\nfrom __future__ import annotations\n\nimport hashlib\nimport inspect\nimport io\nimport json\nimport math\nimport shutil\nimport zipfile\nfrom collections import Counter\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport ultralytics\nimport yaml\nfrom PIL import Image\nfrom torch.utils.data import DataLoader, Sampler\nfrom ultralytics.models.yolo.detect import DetectionTrainer\nfrom ultralytics.nn.tasks import DetectionModel\nfrom ultralytics.utils.loss import v8DetectionLoss\nfrom ultralytics.utils.torch_utils import unwrap_model\n\n\nVERSION = "10-openimages-80-20-1ep-3.0"\nULTRALYTICS_VERSION = "8.4.150"\nLOSS_SOURCE_SHA = "357053aa9a24ad2c4d541c3ada3a27232a10d679f89c7b4957b9a84a7da4ec1d"\n\nNOMES = [\n    "bed", "sofa", "chair", "table", "lamp", "tv",\n    "laptop", "wardrobe", "window", "door",\n    "potted plant", "photo frame"\n]\n\nALL_KNOWN = (1 << 12) - 1\n\n\ndef sha256_file(p):\n    h = hashlib.sha256()\n\n    with Path(p).open("rb") as f:\n        for b in iter(lambda: f.read(4 * 1024 * 1024), b""):\n            h.update(b)\n\n    return h.hexdigest()\n\n\ndef canonical_json(x):\n    return json.dumps(\n        x,\n        sort_keys=True,\n        ensure_ascii=False,\n        separators=(",", ":")\n    )\n\n\ndef save_json(p, x):\n    p = Path(p)\n    p.parent.mkdir(parents=True, exist_ok=True)\n\n    tmp = p.with_name(p.name + ".tmp")\n    tmp.write_text(\n        json.dumps(x, ensure_ascii=False, indent=2),\n        encoding="utf-8"\n    )\n    tmp.replace(p)\n\n\ndef check_api():\n    if ultralytics.__version__ != ULTRALYTICS_VERSION:\n        raise RuntimeError(\n            f"Instale ultralytics=={ULTRALYTICS_VERSION} "\n            "e reinicie o ambiente se necessário."\n        )\n\n    if sha256_file(inspect.getfile(v8DetectionLoss)) != LOSS_SOURCE_SHA:\n        raise RuntimeError(\n            "A implementação da perda difere da versão verificada; "\n            "interrompido antes do treino."\n        )\n\n\nclass BCEComCobertura(nn.Module):\n    """Zera perdas negativas de classes desconhecidas; conserva positivos."""\n\n    def __init__(self, original=None):\n        super().__init__()\n        self.original = original or nn.BCEWithLogitsLoss(reduction="none")\n        self.known = None\n\n    def forward(self, logits, target):\n        if self.known is None:\n            raise RuntimeError(\n                "Cobertura não foi transmitida à perda parcial."\n            )\n\n        known = self.known.to(\n            device=logits.device,\n            dtype=torch.bool\n        )\n\n        if known.shape != (logits.shape[0], logits.shape[-1]):\n            raise RuntimeError(\n                "Dimensões de cobertura incompatíveis com batch/classes."\n            )\n\n        permitted = known[:, None, :] | (target > 0)\n\n        return self.original(logits, target) * permitted.to(logits.dtype)\n\n\nclass DetectionLossParcial(v8DetectionLoss):\n    """Altera somente a BCE; caixa, DFL, assignment e ganhos são os padrões."""\n\n    def __init__(self, model):\n        super().__init__(model)\n\n        if (\n            self.nc != 12\n            or not isinstance(self.bce, nn.BCEWithLogitsLoss)\n            or self.bce.reduction != "none"\n        ):\n            raise RuntimeError(\n                "Cabeça ou perda de detecção não é a esperada "\n                "para este experimento."\n            )\n\n        self.bce = BCEComCobertura(self.bce)\n\n    def loss(self, preds, batch):\n        if "class_known" not in batch:\n            raise RuntimeError(\n                "Batch de treinamento sem cobertura; "\n                "não substituir por negativos presumidos."\n            )\n\n        self.bce.known = batch["class_known"]\n\n        try:\n            return super().loss(preds, batch)\n        finally:\n            self.bce.known = None\n\n\nclass SamplerDuasFontes(Sampler):\n    """Cada batch contém metade Home e metade OI; uma época cobre todos os OI."""\n\n    def __init__(self, home_indices, oi_indices, batch_size, seed=42):\n        self.home = list(home_indices)\n        self.oi = list(oi_indices)\n\n        if (\n            not self.home\n            or not self.oi\n            or batch_size < 2\n            or batch_size % 2\n        ):\n            raise ValueError(\n                "São necessárias duas fontes e um batch par >=2."\n            )\n\n        self.batch = int(batch_size)\n        self.half = self.batch // 2\n        self.batches = math.ceil(len(self.oi) / self.half)\n        self.seed = int(seed)\n        self.epoch = 0\n\n    def set_epoch(self, epoch):\n        self.epoch = int(epoch)\n\n    def __len__(self):\n        return self.batches * self.batch\n\n    def __iter__(self):\n        rng = np.random.default_rng(self.seed + self.epoch)\n\n        home, oi = [], []\n        needed = self.batches * self.half\n\n        while len(home) < needed:\n            home.extend(rng.permutation(self.home).tolist())\n\n        while len(oi) < needed:\n            oi.extend(rng.permutation(self.oi).tolist())\n\n        for start in range(0, needed, self.half):\n            batch = (\n                home[start:start + self.half]\n                + oi[start:start + self.half]\n            )\n            rng.shuffle(batch)\n            yield from batch\n\n\nclass LoaderFinito(DataLoader):\n    def reset(self):\n        # Um novo iterador é criado em cada época.\n        pass\n\n\ndef epoch_sampler(trainer):\n    sampler = getattr(trainer.train_loader, "sampler", None)\n\n    if isinstance(sampler, SamplerDuasFontes):\n        sampler.set_epoch(trainer.epoch)\n\n\nclass TrainerOpenImages(DetectionTrainer):\n    # Configurado antes de criar o trainer.\n    # Não é anexado ao DetectionModel salvo.\n    source_records = None\n\n    def __init__(self, *args, **kwargs):\n        check_api()\n        super().__init__(*args, **kwargs)\n        self.add_callback("on_train_epoch_start", epoch_sampler)\n\n    def get_dataloader(\n        self,\n        dataset_path,\n        batch_size=16,\n        rank=0,\n        mode="train"\n    ):\n        if mode != "train":\n            return super().get_dataloader(\n                dataset_path,\n                batch_size,\n                rank,\n                mode\n            )\n\n        if self.source_records is None:\n            raise RuntimeError(\n                "As fontes/coberturas ainda não foram configuradas."\n            )\n\n        if self.world_size > 1 or self.args.compile or self.args.rect:\n            raise RuntimeError(\n                "Este experimento usa uma GPU, "\n                "compile=False e rect=False."\n            )\n\n        for k in ["mosaic", "mixup", "cutmix", "copy_paste"]:\n            if float(getattr(self.args, k, 0)) != 0:\n                raise RuntimeError(\n                    f"{k} deve ficar em zero para preservar "\n                    "a cobertura por imagem."\n                )\n\n        dataset = self.build_dataset(\n            dataset_path,\n            mode,\n            batch_size\n        )\n\n        home, oi = [], []\n\n        for i, arquivo in enumerate(dataset.im_files):\n            key = Path(arquivo).resolve().as_posix()\n\n            if key not in self.source_records:\n                raise RuntimeError(\n                    f"Imagem sem proveniência/cobertura: {arquivo}"\n                )\n\n            source = self.source_records[key]["source"]\n\n            if source == "home":\n                home.append(i)\n            elif source == "openimages":\n                oi.append(i)\n            else:\n                raise RuntimeError(\n                    "Fonte de treinamento não reconhecida."\n                )\n\n        sampler = SamplerDuasFontes(\n            home,\n            oi,\n            batch_size,\n            self.args.seed\n        )\n\n        generator = torch.Generator().manual_seed(self.args.seed)\n\n        return LoaderFinito(\n            dataset,\n            batch_size=batch_size,\n            sampler=sampler,\n            num_workers=self.args.workers,\n            pin_memory=self.device.type == "cuda",\n            collate_fn=dataset.collate_fn,\n            generator=generator,\n            drop_last=False\n        )\n\n    def preprocess_batch(self, batch):\n        batch = super().preprocess_batch(batch)\n\n        masks = []\n\n        for arquivo in batch["im_file"]:\n            key = Path(arquivo).resolve().as_posix()\n\n            if key not in self.source_records:\n                raise RuntimeError(\n                    "Batch contém imagem fora do manifesto de treino."\n                )\n\n            bitmask = self.source_records[key]["known_mask"]\n\n            masks.append([\n                bool(bitmask & (1 << cid))\n                for cid in range(12)\n            ])\n\n        batch["class_known"] = torch.tensor(\n            masks,\n            dtype=torch.bool,\n            device=self.device\n        )\n\n        model = unwrap_model(self.model)\n        criterion = getattr(model, "criterion", None)\n\n        if criterion is None:\n            # Modelo padrão, já colocado no dispositivo.\n            # A EMA foi criada antes desta etapa e valida Home\n            # com a perda padrão de cobertura completa.\n            model.criterion = DetectionLossParcial(model)\n\n        elif not isinstance(criterion, DetectionLossParcial):\n            raise RuntimeError(\n                "O treino iniciou com uma perda diferente "\n                "da perda parcial."\n            )\n\n        return batch\n\n\ndef self_test_bce():\n    logits = torch.zeros(\n        (2, 3, 12),\n        requires_grad=True\n    )\n\n    targets = torch.zeros_like(logits)\n    targets[1, 0, 7] = 1\n\n    known = torch.ones(\n        (2, 12),\n        dtype=torch.bool\n    )\n    known[1, 7] = False\n    known[1, 10] = False\n\n    partial = BCEComCobertura()\n    partial.known = known\n    partial(logits, targets).sum().backward()\n\n    assert logits.grad[1, :, 10].eq(0).all()\n    assert logits.grad[1, 1:, 7].eq(0).all()\n    assert logits.grad[1, 0, 7].item() < 0\n    assert logits.grad[0].ne(0).all()\n\n    full = BCEComCobertura()\n    full.known = torch.ones(\n        (2, 12),\n        dtype=torch.bool\n    )\n\n    torch.testing.assert_close(\n        full(logits.detach(), targets),\n        nn.BCEWithLogitsLoss(reduction="none")(\n            logits.detach(),\n            targets\n        )\n    )\n\n    return {\n        "gradiente_negativo_desconhecido_zero": True,\n        "caixa_positiva_preservada": True,\n        "cobertura_completa_igual_bce_padrao": True\n    }\n\n\ndef read_safe(z, name):\n    path = PurePosixPath(name)\n\n    if path.is_absolute() or ".." in path.parts or "\\\\" in name:\n        raise RuntimeError(\n            f"Caminho inseguro dentro de pacote: {name}"\n        )\n\n    return z.read(name)\n\n\ndef validate_yolo(raw, nclasses=12, source="<anotação>"):\n    """\n    Ignora linhas vazias e valida as linhas preenchidas.\n    Em caso de erro, informa arquivo, linha e conteúdo.\n    """\n    result = []\n\n    try:\n        text = raw.decode("utf-8-sig")\n    except UnicodeDecodeError as exc:\n        raise RuntimeError(\n            f"Anotação YOLO não está em UTF-8: {source}"\n        ) from exc\n\n    for line_number, line in enumerate(text.splitlines(), 1):\n        if not line.strip():\n            continue\n\n        values = line.split()\n        location = f"{source}, linha {line_number}"\n\n        if len(values) != 5:\n            raise RuntimeError(\n                f"Anotação YOLO inválida em {location}: "\n                "esperados 5 campos "\n                "(classe x y largura altura), "\n                f"encontrados {len(values)}. "\n                f"Conteúdo: {line[:160]!r}"\n            )\n\n        try:\n            cid, *bbox = map(float, values)\n        except ValueError as exc:\n            raise RuntimeError(\n                f"Campo não numérico em {location}: "\n                f"{line[:160]!r}"\n            ) from exc\n\n        if (\n            not cid.is_integer()\n            or not 0 <= cid < nclasses\n            or not all(math.isfinite(x) for x in bbox)\n        ):\n            raise RuntimeError(\n                f"Classe ou caixa YOLO inválida em {location}: "\n                f"{line[:160]!r}"\n            )\n\n        x, y, w, h = bbox\n\n        if not (\n            0 <= x <= 1\n            and 0 <= y <= 1\n            and 0 < w <= 1\n            and 0 < h <= 1\n            and x - w / 2 >= -1e-6\n            and x + w / 2 <= 1 + 1e-6\n            and y - h / 2 >= -1e-6\n            and y + h / 2 <= 1 + 1e-6\n        ):\n            raise RuntimeError(\n                f"Caixa YOLO fora dos limites em {location}: "\n                f"{line[:160]!r}"\n            )\n\n        result.append(int(cid))\n\n    if not result:\n        raise RuntimeError(\n            f"Imagem desta preparação sem caixas: {source}"\n        )\n\n    return result\n\n\ndef write_checked(dest, raw, expected_sha):\n    if hashlib.sha256(raw).hexdigest() != expected_sha:\n        raise RuntimeError(\n            "Conteúdo do pacote diferente do SHA-256 no manifesto."\n        )\n\n    dest = Path(dest)\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    dest.write_bytes(raw)\n\n\ndef prepare_training(\n    zip_home,\n    oi_drive,\n    local,\n    results,\n    expected,\n    batches_oi=None,\n    home_train_count=1824,\n    home_val_count=456,\n    oi_count=100000,\n    package_fetcher=None\n):\n    """Snapshot Home TRAIN/VAL e de todos os lotes liberados para 100k."""\n\n    local = Path(local)\n    results = Path(results)\n    oi_drive = Path(oi_drive)\n\n    local.mkdir(parents=True, exist_ok=True)\n    results.mkdir(parents=True, exist_ok=True)\n\n    prep_cfg = json.loads(\n        (oi_drive / "configuracao_preparacao.json").read_text()\n    )\n\n    cfg_sha = hashlib.sha256(\n        canonical_json(prep_cfg).encode()\n    ).hexdigest()\n\n    if (\n        cfg_sha != expected["oi_config_sha256"]\n        or prep_cfg["nomes"] != NOMES\n    ):\n        raise RuntimeError(\n            "Preparação Open Images diferente da auditada no piloto."\n        )\n\n    home_zip_local = local / "home_sun_snapshot.zip"\n    shutil.copyfile(zip_home, home_zip_local)\n\n    if sha256_file(home_zip_local) != expected["home_zip_sha256"]:\n        raise RuntimeError(\n            "ZIP Home+SUN diferente do preservado no 07b/08."\n        )\n\n    records, home_val = {}, []\n    extract_home = local / "base_existente"\n\n    with zipfile.ZipFile(home_zip_local) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError(\n                "ZIP Home+SUN tem nomes repetidos."\n            )\n\n        manifest = json.loads(\n            read_safe(z, "verificacao_preparacao.json")\n        )\n\n        if (\n            manifest["test"]["imagens"] != 403\n            or manifest["test"]["sha256"] != expected["test_sha256"]\n        ):\n            raise RuntimeError(\n                "TEST preservado diferente do esperado."\n            )\n\n        if (\n            {int(k): v for k, v in manifest["classes"].items()}\n            != dict(enumerate(NOMES))\n        ):\n            raise RuntimeError(\n                "Classes/IDs da base Home incompatíveis."\n            )\n\n        root = manifest["dataset_dir"]\n\n        for item in manifest["arquivos"]:\n            name = item["arquivo"]\n            p = PurePosixPath(name)\n\n            in_home_train = (\n                name.startswith(f"{root}/images/train/raw_train/")\n                or name.startswith(f"{root}/labels/train/raw_train/")\n            )\n\n            in_home_val = (\n                name.startswith(f"{root}/images/val/raw_train/")\n                or name.startswith(f"{root}/labels/val/raw_train/")\n            )\n\n            code_or_meta = (\n                name.startswith("codigo/")\n                or (\n                    p.suffix in {\n                        ".json", ".csv", ".yaml", ".yml", ".txt"\n                    }\n                    and "/images/" not in name\n                    and "/labels/" not in name\n                )\n            )\n\n            if not (in_home_train or in_home_val or code_or_meta):\n                continue\n\n            raw = read_safe(z, name)\n            dest = extract_home / name\n            write_checked(dest, raw, item["sha256"])\n\n            if name.startswith(f"{root}/images/train/raw_train/"):\n                records[dest.resolve().as_posix()] = {\n                    "source": "home",\n                    "known_mask": ALL_KNOWN,\n                    "sha256": item["sha256"]\n                }\n\n            elif name.startswith(f"{root}/images/val/raw_train/"):\n                home_val.append(\n                    dest.resolve().as_posix()\n                )\n\n        listed = {\n            item["arquivo"]\n            for item in manifest["arquivos"]\n        }\n\n        # O hash do ZIP também autentica o código auxiliar\n        # que não esteja listado individualmente no manifesto.\n        for name in set(z.namelist()) - listed:\n            if name.startswith("codigo/") and not name.endswith("/"):\n                raw = read_safe(z, name)\n\n                write_checked(\n                    extract_home / name,\n                    raw,\n                    hashlib.sha256(raw).hexdigest()\n                )\n\n    home_dataset = extract_home / root\n    home_images = list(records)\n\n    if (\n        len(home_images) != home_train_count\n        or len(home_val) != home_val_count\n    ):\n        raise RuntimeError(\n            "Home TRAIN/VAL inesperados: "\n            f"{len(home_images)}/{len(home_val)}; "\n            f"esperados {home_train_count}/{home_val_count}."\n        )\n\n    for image in [*home_images, *home_val]:\n        label = Path(\n            image.replace("/images/", "/labels/")\n        ).with_suffix(".txt")\n\n        validate_yolo(\n            label.read_bytes(),\n            source=str(label)\n        )\n\n    packages = json.loads(\n        (oi_drive / "manifesto_pacotes.json").read_text()\n    )\n\n    if packages["config_sha256"] != cfg_sha:\n        raise RuntimeError(\n            "Manifesto dos lotes tem configuração diferente."\n        )\n\n    selected = sorted(\n        packages["lotes"],\n        key=lambda r: r["arquivo"]\n    )\n    if batches_oi is None:\n        batches_oi = len(selected)\n\n    if len(selected) != batches_oi:\n        raise RuntimeError(\n            "Quantidade de lotes difere da liberação para 100k."\n        )\n\n    oi_ids, oi_hashes, package_snapshot = set(), set(), []\n    oi_root = local / "openimages"\n    per_class = Counter()\n\n    for item in selected:\n        name = item["arquivo"]\n\n        if PurePosixPath(name).name != name:\n            raise RuntimeError(\n                "Nome de pacote não é um nome simples."\n            )\n\n        snapshot = local / name\n\n        if package_fetcher is None:\n            shutil.copyfile(oi_drive / "lotes" / name, snapshot)\n        else:\n            print(f"Pacote {len(package_snapshot) + 1}/{len(selected)}: {name}", flush=True)\n            package_fetcher(item, snapshot)\n\n        if (\n            sha256_file(snapshot) != item["zip_sha256"]\n            or snapshot.stat().st_size != item["zip_bytes"]\n        ):\n            raise RuntimeError(\n                f"Pacote alterado ou incompleto: {name}"\n            )\n\n        with zipfile.ZipFile(snapshot) as z:\n            if len(z.namelist()) != len(set(z.namelist())):\n                raise RuntimeError(\n                    "ZIP Open Images tem membros repetidos."\n                )\n\n            m = json.loads(\n                read_safe(z, "manifesto_lote.json")\n            )\n\n            if (\n                m["config_sha256"] != cfg_sha\n                or m["origem_split"] != "train"\n            ):\n                raise RuntimeError(\n                    "Lote não é da seleção do treino "\n                    "Open Images auditado."\n                )\n\n            for r in m["aceitas"]:\n                if r["id"] in oi_ids or r["sha256"] in oi_hashes:\n                    raise RuntimeError(\n                        "ID ou bytes duplicados nos lotes usados no treino."\n                    )\n\n                oi_ids.add(r["id"])\n                oi_hashes.add(r["sha256"])\n\n                im_raw = read_safe(z, r["imagem"])\n                label_raw = read_safe(z, r["label"])\n                cov_raw = read_safe(z, r["cobertura"])\n\n                for key, raw, hkey in [\n                    ("imagem", im_raw, "sha256"),\n                    ("label", label_raw, "label_sha256"),\n                    ("cobertura", cov_raw, "cobertura_sha256")\n                ]:\n                    write_checked(\n                        oi_root / r[key],\n                        raw,\n                        r[hkey]\n                    )\n\n                cls = validate_yolo(\n                    label_raw,\n                    source=f"{name}:{r[\'label\']}"\n                )\n\n                cov = json.loads(cov_raw)\n\n                if (\n                    cov["known_mask"] != r["known_mask"]\n                    or not isinstance(r["known_mask"], int)\n                    or not 0 <= r["known_mask"] <= ALL_KNOWN\n                ):\n                    raise RuntimeError(\n                        "Cobertura do lote inconsistente."\n                    )\n\n                known_list = (\n                    set(cov["classes_positivas_verificadas_exaustivas"])\n                    | set(cov["classes_negativas_verificadas"])\n                )\n\n                if sum(1 << cid for cid in known_list) != r["known_mask"]:\n                    raise RuntimeError(\n                        "Lista de classes conhecidas "\n                        "não corresponde à máscara."\n                    )\n\n                if set(cls) & set(cov["classes_negativas_verificadas"]):\n                    raise RuntimeError(\n                        "Caixa positiva contradiz ausência verificada."\n                    )\n\n                with Image.open(io.BytesIO(im_raw)) as im:\n                    im.load()\n\n                    if im.size != (r["width"], r["height"]):\n                        raise RuntimeError(\n                            "Dimensões da imagem diferem do manifesto."\n                        )\n\n                image = (\n                    oi_root / r["imagem"]\n                ).resolve().as_posix()\n\n                records[image] = {\n                    "source": "openimages", "known_mask": r["known_mask"], "sha256": r["sha256"],\n                    "oi_id": r["id"], "lote": name,\n                    "classe_primaria": int(r.get("classe_primaria_selecao", min(cls))),\n                    "classes_com_caixa": cls, "width": r["width"], "height": r["height"],\n                    "label_sha256": r["label_sha256"], "cobertura_sha256": r["cobertura_sha256"],\n                }\n\n                per_class.update(cls)\n\n        package_snapshot.append(item)\n\n        # Remove apenas a cópia local do ZIP do lote.\n        snapshot.unlink()\n\n    if len(oi_ids) != oi_count:\n        raise RuntimeError(\n            f"Treinamento esperava {oi_count} imagens OI, "\n            f"encontrou {len(oi_ids)}."\n        )\n\n    if set(home_val) & set(records):\n        raise RuntimeError(\n            "Imagem de validação aparece no treino."\n        )\n\n    val_hashes = {\n        sha256_file(x)\n        for x in home_val\n    }\n\n    if val_hashes & {\n        r["sha256"]\n        for r in records.values()\n    }:\n        raise RuntimeError(\n            "Imagem de treino coincide em bytes "\n            "com a validação Home."\n        )\n\n    # O 08 já realizou a triagem de contaminação.\n    # Aqui repetimos a verificação por bytes contra Home VAL.\n    train_list = local / "train.txt"\n    val_list = local / "home_val.txt"\n\n    train_list.write_text(\n        "\\n".join(sorted(records)) + "\\n",\n        encoding="utf-8"\n    )\n\n    val_list.write_text(\n        "\\n".join(sorted(home_val)) + "\\n",\n        encoding="utf-8"\n    )\n\n    data_yaml = local / "data_home_openimages.yaml"\n\n    data_yaml.write_text(\n        yaml.safe_dump(\n            {\n                "path": str(local.resolve()),\n                "train": str(train_list),\n                "val": str(val_list),\n                "names": dict(enumerate(NOMES))\n            },\n            sort_keys=False,\n            allow_unicode=True\n        ),\n        encoding="utf-8"\n    )\n\n    save_json(\n        local / "fontes_cobertura_treino.json",\n        records\n    )\n\n    summary = {\n        "versao": VERSION,\n        "home_train_imagens_unicas": len(home_images),\n        "home_val_imagens": len(home_val),\n        "openimages_imagens_unicas": len(oi_ids),\n        "treino_imagens_unicas": len(records),\n        "caixas_openimages": sum(per_class.values()),\n        "caixas_openimages_por_classe": dict(per_class),\n        "lotes_utilizados": package_snapshot,\n        "home_zip_sha256": expected["home_zip_sha256"],\n        "test_sha256": expected["test_sha256"],\n        "test_imagens": 403,\n        "test_extraido_ou_avaliado": False,\n        "sun_incluida_no_novo_treino": False,\n        "amostragem": (\n            "50% Home / 50% OI em cada batch; "\n            "Home reamostrado por índices, sem duplicar arquivos"\n        ),\n        "selecao_checkpoint": (\n            "mAP50-95 Home VAL; triagem de alertas após treino"\n        ),\n        "tratamento_parcial": (\n            "Zero gradiente BCE para negativos de classes desconhecidas; "\n            "caixas positivas preservadas"\n        ),\n        "config_preparacao_sha256": cfg_sha\n    }\n\n    save_json(\n        results / "resumo_dados_treino.json",\n        summary\n    )\n\n    save_json(\n        results / "lotes_utilizados.json",\n        package_snapshot\n    )\n\n    TrainerOpenImages.source_records = records\n\n    return {\n        "yaml": data_yaml,\n        "home_dataset": home_dataset,\n        "codigo": extract_home / "codigo",\n        "records": records,\n        "home_val": home_val,\n        "resumo": summary\n    }\n\n\ndef export_standard_checkpoint(source, dest):\n    """Confere que o checkpoint não depende das classes do treinador parcial."""\n\n    obj = torch.load(\n        source,\n        map_location="cpu",\n        weights_only=False\n    )\n\n    model = obj.get("ema") or obj.get("model")\n\n    if type(model) is not DetectionModel:\n        raise RuntimeError(\n            "O checkpoint não contém DetectionModel padrão."\n        )\n\n    model.criterion = None\n    model = model.float().eval()\n\n    if not all(\n        torch.isfinite(t).all()\n        for t in model.state_dict().values()\n        if t.is_floating_point()\n    ):\n        raise RuntimeError(\n            "Checkpoint com pesos não finitos."\n        )\n\n    obj["model"] = model\n    obj["ema"] = None\n    obj["optimizer"] = None\n\n    torch.save(obj, dest)\n\n    return {\n        "arquivo": str(dest),\n        "sha256": sha256_file(dest),\n        "modelo_padrao": True,\n        "criterion_serializada": False\n    }', 'exp12_data.py': '"""Entradas congeladas e replay somente do TRAIN histórico."""\nfrom __future__ import annotations\n\nfrom collections import Counter\nimport csv\nimport gzip\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\nimport yaml\n\nimport diag11_core as core\nimport diag11_data as d11\nfrom exp12_protocol import diverse_subset\n\n\ndef check_digest(raw, expected, label):\n    if hashlib.sha256(raw).hexdigest() != expected:\n        raise RuntimeError("Conteúdo alterado: " + label)\n\n\ndef materialise_home_labels(zip_path, home):\n    with zipfile.ZipFile(zip_path) as z:\n        for record in home:\n            member = record["id"].removeprefix("home:")\n            label_name = member.replace("/images/", "/labels/").rsplit(".", 1)[0]+".txt"\n            core.safe_member(label_name)\n            raw = z.read(label_name)\n            check_digest(raw, record["label_sha256"], label_name)\n            label = Path(str(record["image"]).replace("/images/", "/labels/")).with_suffix(".txt")\n            label.parent.mkdir(parents=True, exist_ok=True)\n            label.write_bytes(raw)\n            record["label"] = label\n\n\ndef read_package11(path, expected_sha, expected_models, expected_dataset):\n    if core.sha256_file(path) != expected_sha:\n        raise RuntimeError("O pacote do diagnóstico 11 mudou.")\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError("Pacote 11 com membros repetidos.")\n        for name in z.namelist():\n            core.safe_member(name)\n        summary = json.loads(z.read("resumo_diagnostico_11.json"))\n        config = json.loads(z.read("configuracao_diagnostico.json"))\n        dataset_location = json.loads(z.read("localizacao_dataset.json"))\n        manifest = json.loads(z.read("manifesto_validacao_nova.json"))\n        if (config["modelos_sha256"] != expected_models or\n                config["dataset_sha256"] != expected_dataset or\n                dataset_location["dataset_sha256"] != expected_dataset or\n                manifest["dataset_sha256"] != expected_dataset or\n                summary["reserva_futura_inferida"] is not False):\n            raise RuntimeError("Identidades ou reserva do diagnóstico 11 incompatíveis.")\n    return summary, config, dataset_location, manifest\n\n\ndef load_frozen_validation(zip_path, destination, expected_archive_sha, expected_dataset_sha,\n                           expected_manifest):\n    if core.sha256_file(zip_path) != expected_archive_sha:\n        raise RuntimeError("ZIP da validação independente diferente do congelado no 11.")\n    destination = Path(destination)\n    val, reserved_ids, frozen_files = [], set(), []\n    with zipfile.ZipFile(zip_path) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError("Dataset novo com membros repetidos.")\n        for member in z.namelist():\n            core.safe_member(member)\n        manifest = json.loads(z.read("manifesto_validacao_nova.json"))\n        if manifest != expected_manifest:\n            raise RuntimeError("O manifesto do pacote difere do diagnóstico 11.")\n        raw_manifest = dict(manifest)\n        claimed = raw_manifest.pop("dataset_sha256")\n        if hashlib.sha256(core.canonical(raw_manifest).encode()).hexdigest() != claimed or claimed != expected_dataset_sha:\n            raise RuntimeError("Hash canônico da validação nova não confere.")\n        if manifest["nomes"] != core.NAMES or manifest["origem_split"] != "validation":\n            raise RuntimeError("Classes ou origem da validação alteradas.")\n        items = manifest["arquivos"]\n        if (Counter(r["role"] for r in items) != {"val_nova": 1000, "reserva_futura": 300}\n                or len({r["id"] for r in items}) != 1300\n                or len({r["sha256"] for r in items}) != 1300):\n            raise RuntimeError("Identidades ou quantidades do conjunto independente incompatíveis.")\n        for item in items:\n            role = item["role"]\n            for field, prefix in [("image", "images"), ("label", "labels"), ("coverage", "coverage")]:\n                if not item[field].startswith(prefix+"/"+role+"/"):\n                    raise RuntimeError("Caminho do dataset não corresponde ao papel declarado.")\n                if item[field] not in z.namelist():\n                    raise RuntimeError("Arquivo declarado ausente no ZIP.")\n            if role == "reserva_futura":\n                reserved_ids.add(item["id"])\n                continue  # Nenhum JPEG, label ou cobertura da reserva é extraído.\n            paths = {}\n            for field, hash_field in [("image", "sha256"), ("label", "label_sha256"),\n                                      ("coverage", "coverage_sha256")]:\n                raw = z.read(item[field])\n                check_digest(raw, item[hash_field], item[field])\n                output = destination/item[field]\n                output.parent.mkdir(parents=True, exist_ok=True)\n                output.write_bytes(raw)\n                paths[field] = output\n                frozen_files.append((output, item[hash_field]))\n            coverage = json.loads(paths["coverage"].read_text())\n            if any(coverage[k] != item[k] for k in ["known_mask", "positive_mask", "negative_mask"]):\n                raise RuntimeError("Cobertura da validação foi alterada.")\n            with Image.open(paths["image"]) as image:\n                if image.size != (item["width"], item["height"]):\n                    raise RuntimeError("Dimensões diferentes do manifesto.")\n                image.verify()\n            parsed = core.parse_yolo(paths["label"].read_bytes(), item["width"], item["height"])\n            if len(parsed) != len(item["gt_normalizado"]):\n                raise RuntimeError("Quantidade de caixas difere do manifesto.")\n            gt = []\n            for original, label_obj in zip(item["gt_normalizado"], parsed):\n                a, b, c, d = original["bbox_norm"]\n                box = [a*item["width"], b*item["height"], c*item["width"], d*item["height"]]\n                if (original["classe_id"] != label_obj["classe_id"] or\n                        not np.allclose(box, label_obj["bbox"], rtol=0, atol=.001)):\n                    raise RuntimeError("Caixa do manifesto diverge do label autenticado.")\n                gt.append({**original, "bbox": box})\n            val.append({**item, **paths, "gt": gt, "role": "val_nova",\n                        "coorte": "openimages_val_nova", "name": paths["image"].name})\n    return val, reserved_ids, frozen_files\n\n\ndef load_replay(package_items, selected_history, local_dir, fetch_package, expected_cfg,\n                forbidden_ids, forbidden_hashes, protected_near, validator, final_size=2000, seed=42):\n    """Lê só os membros escolhidos dos cinco lotes existentes, com SHA integral."""\n    local_dir = Path(local_dir)\n    local_dir.mkdir(parents=True, exist_ok=True)\n    wanted = {r["id"]: r for r in selected_history}\n    protected_ids, protected_hashes = set(forbidden_ids), set(forbidden_hashes)\n    records, exclusions = {}, []\n    frozen, seen, package_receipts = [], set(), []\n    for item in package_items:\n        pkg = local_dir/item["arquivo"]\n        fetch_package(item, pkg)\n        if pkg.stat().st_size != item["zip_bytes"] or core.sha256_file(pkg) != item["zip_sha256"]:\n            raise RuntimeError("Pacote de replay incompleto ou alterado.")\n        with zipfile.ZipFile(pkg) as z:\n            if len(z.namelist()) != len(set(z.namelist())):\n                raise RuntimeError("Lote de replay com membros repetidos.")\n            for name in z.namelist():\n                core.safe_member(name)\n            manifest = json.loads(z.read("manifesto_lote.json"))\n            if manifest["config_sha256"] != expected_cfg or manifest["origem_split"] != "train":\n                raise RuntimeError("Lote de replay não pertence ao TRAIN autenticado.")\n            for r in manifest["aceitas"]:\n                if r["id"] not in wanted:\n                    continue\n                if r["id"] in seen:\n                    raise RuntimeError("ID de replay repetido entre lotes.")\n                seen.add(r["id"])\n                prior = wanted[r["id"]]\n                if (prior["split"] != "train" or prior["lote"] != item["arquivo"] or\n                        prior["imagem_sha256"] != r["sha256"] or\n                        int(prior["known_mask"]) != r["known_mask"]):\n                    raise RuntimeError("Replay diverge da divisão 80/20 histórica.")\n                if r["id"] in protected_ids or r["sha256"] in protected_hashes:\n                    raise RuntimeError("Replay coincide com dado protegido.")\n                raw_image, raw_label, raw_cov = (z.read(r[k]) for k in ["imagem", "label", "cobertura"])\n                for raw, key in [(raw_image, "sha256"), (raw_label, "label_sha256"), (raw_cov, "cobertura_sha256")]:\n                    check_digest(raw, r[key], r["id"]+"/"+key)\n                classes = validator(raw_label, source=item["arquivo"]+":"+r["label"])\n                cov = json.loads(raw_cov)\n                known_classes = (set(cov["classes_positivas_verificadas_exaustivas"]) |\n                                 set(cov["classes_negativas_verificadas"]))\n                if (not all(isinstance(cid, int) and 0 <= cid < 12 for cid in known_classes)\n                        or sum(1 << cid for cid in known_classes) != r["known_mask"]\n                        or cov["known_mask"] != r["known_mask"]\n                        or set(classes) & set(cov["classes_negativas_verificadas"])):\n                    raise RuntimeError("Máscara ou anotação parcial de replay incompatível.")\n                image = local_dir/"images/train"/f"oi_{r[\'id\']}.jpg"\n                label = local_dir/"labels/train"/f"oi_{r[\'id\']}.txt"\n                coverage_file = local_dir/"coverage/train"/f"oi_{r[\'id\']}.json"\n                for path, raw in [(image, raw_image), (label, raw_label), (coverage_file, raw_cov)]:\n                    path.parent.mkdir(parents=True, exist_ok=True)\n                    path.write_bytes(raw)\n                with Image.open(image) as im:\n                    width, height = im.size\n                    im.verify()\n                if (width, height) != (r["width"], r["height"]):\n                    raise RuntimeError("Dimensões de replay incompatíveis.")\n                near = protected_near.find(d11.phash(image))\n                if near:\n                    exclusions.append({"id": r["id"], "motivo": "semelhanca_phash_com_validacao_ou_reserva", "semelhante_a": near})\n                    continue\n                gt = core.parse_yolo(raw_label, width, height)\n                records[r["id"]] = {\n                    **prior, "oi_id": r["id"], "id": r["id"], "source": "openimages",\n                    "image": image, "label": label, "coverage": coverage_file,\n                    "known_mask": r["known_mask"], "sha256": r["sha256"],\n                    "label_sha256": r["label_sha256"], "coverage_sha256": r["cobertura_sha256"],\n                    "width": width, "height": height, "classes_com_caixa": classes,\n                    "gt": gt, "name": image.name, "role": "train", "coorte": "replay_train",\n                }\n        package_receipts.append(item)\n        pkg.unlink()\n    if seen != set(wanted):\n        raise RuntimeError("Alguns IDs planejados não foram encontrados nos cinco lotes.")\n    if len(records) < final_size:\n        raise RuntimeError(f"Replay com somente {len(records)} elegíveis; exigidos {final_size}.")\n    chosen = diverse_subset(list(records.values()), final_size, seed)\n    output = {Path(r["image"]).resolve().as_posix(): r for r in chosen}\n    for record in output.values():\n        frozen += [(record["image"], record["sha256"]), (record["label"], record["label_sha256"]),\n                   (record["coverage"], record["coverage_sha256"])]\n    return output, exclusions, frozen, package_receipts\n\n\ndef make_training_inputs(home, replay, destination):\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    home_train = {Path(r["image"]).resolve().as_posix():\n                  {**r, "source": "home", "known_mask": core.ALL_KNOWN}\n                  for r in home if r["role"] == "home_train"}\n    val = [Path(r["image"]).resolve().as_posix() for r in home if r["role"] == "home_val"]\n    if len(home_train) != 1824 or len(val) != 456:\n        raise RuntimeError("Home diferente da configuração congelada.")\n    val_file = destination/"home_val.txt"\n    val_file.write_text("\\n".join(sorted(val))+"\\n", encoding="utf-8")\n    arms = {}\n    for name, records in [("A_home", home_train), ("B_home_replay", {**home_train, **replay})]:\n        path = destination/(name+".txt")\n        path.write_text("\\n".join(sorted(records))+"\\n", encoding="utf-8")\n        data_file = destination/(name+".yaml")\n        data_file.write_text(yaml.safe_dump({"path": str(destination.resolve()), "train": str(path.resolve()),\n                                           "val": str(val_file.resolve()), "names": dict(enumerate(core.NAMES))},\n                                          sort_keys=False, allow_unicode=True), encoding="utf-8")\n        arms[name] = {"records": records, "yaml": data_file}\n    return arms\n\n\ndef restore_result(package_path, cohort, model, cache_dir):\n    cache_dir = Path(cache_dir)/cohort/model\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(package_path) as z:\n        prefix = cohort+"/"+model+"/"\n        summary = json.loads(z.read(prefix+"metricas.json"))\n        def rows(name):\n            return list(csv.DictReader(io.StringIO(z.read(prefix+name).decode("utf-8-sig"))))\n        images, events, classes = rows("alertas_por_imagem.csv"), rows("avaliacao_objetos.csv"), rows("metricas_por_classe.csv")\n        for row in images:\n            for k in ["tp_alerta", "fn_alerta", "fp_alerta", "alta_ref", "alta_sem_alerta", "tp_detector",\n                      "fn_detector", "fp_detector", "gt_avaliaveis", "gt_ignorados", "predicoes_ignoradas_conf_025"]:\n                row[k] = int(row[k])\n            for k in ["cobertura_completa_alertas", "ha_alerta_ref", "ha_alerta_pred"]:\n                row[k] = row[k] == "True"\n        for row in events:\n            row["classe_id"] = int(row["classe_id"])\n            for k in ["ha_alerta_ref", "ha_alerta_pred"]:\n                row[k] = row[k] == "True"\n        cache = cache_dir/"predicoes_conf_0001.jsonl.gz"\n        cache.write_bytes(z.read(prefix+"predicoes_conf_0001.jsonl.gz"))\n    return {"summary": summary, "images": images, "events": events, "classes": classes, "cache_file": cache}\n\n\ndef verify_frozen(files):\n    for path, sha in files:\n        if core.sha256_file(path) != sha:\n            raise RuntimeError("Arquivo protegido alterado: "+str(path))\n', 'exp12_protocol.py': '"""Orçamento e seleção do experimento 12, sem dependência de Torch."""\nfrom __future__ import annotations\n\nfrom collections import Counter\nimport hashlib\nimport math\nimport numpy as np\n\nARMS = {\n    "A_home": {"home_per_batch": 16, "oi_per_batch": 0},\n    "B_home_replay": {"home_per_batch": 12, "oi_per_batch": 4},\n}\n\n\nclass BudgetSampler:\n    """Fluxos por fonte independentes; quantidade fixa de batches por época."""\n    def __init__(self, home, oi, batch_size, batches, oi_per_batch, seed=42):\n        self.home, self.oi = list(home), list(oi)\n        self.batch_size, self.batches = int(batch_size), int(batches)\n        self.oi_per_batch = int(oi_per_batch)\n        self.home_per_batch = self.batch_size - self.oi_per_batch\n        self.seed, self.epoch = int(seed), 0\n        if (not self.home or self.batches <= 0 or self.batch_size <= 0 or\n                not 0 <= self.oi_per_batch < self.batch_size or\n                (self.oi_per_batch and not self.oi) or set(self.home) & set(self.oi)):\n            raise ValueError("Fontes ou orçamento inválidos.")\n\n    def __len__(self):\n        return self.batches*self.batch_size\n\n    def set_epoch(self, epoch):\n        self.epoch = int(epoch)\n\n    def _stream(self, values, needed, source):\n        if not needed:\n            return []\n        rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.epoch, source]))\n        result = []\n        while len(result) < needed:\n            result.extend(rng.permutation(values).tolist())\n        return result[:needed]\n\n    def __iter__(self):\n        home = self._stream(self.home, self.batches*self.home_per_batch, 0)\n        oi = self._stream(self.oi, self.batches*self.oi_per_batch, 1)\n        rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.epoch, 2]))\n        for b in range(self.batches):\n            batch = home[b*self.home_per_batch:(b+1)*self.home_per_batch]\n            batch += oi[b*self.oi_per_batch:(b+1)*self.oi_per_batch]\n            rng.shuffle(batch)\n            yield from batch\n\n\ndef order_id(iid, seed=42):\n    return hashlib.sha256(f"{seed}|{iid}".encode()).hexdigest(), iid\n\n\ndef diverse_subset(rows, n=2000, seed=42, per_class=40):\n    if n <= 0 or n > len(rows) or len({r["id"] for r in rows}) != len(rows):\n        raise ValueError("Tamanho do replay ou identidade dos IDs inválidos.")\n    mapping = {r["id"]: r for r in rows}\n    groups = {c: sorted([r["id"] for r in rows if int(r["classe_primaria"]) == c],\n                        key=lambda iid: order_id(iid, seed)) for c in range(12)}\n    chosen = set()\n    for k in range(per_class):\n        for c in sorted(groups, key=lambda cid: (len(groups[cid]), cid)):\n            if k < len(groups[c]) and len(chosen) < n:\n                chosen.add(groups[c][k])\n    for iid in sorted(mapping, key=lambda x: order_id(x, seed)):\n        if len(chosen) == n:\n            break\n        chosen.add(iid)\n    return [mapping[iid] for iid in sorted(chosen)]\n\n\ndef choose_replay(history_rows, n=2200, seed=42, per_class=40, forbidden_hashes=()):\n    batches = {f"lote_{i:06}.zip" for i in range(1, 6)}\n    rows = [r for r in history_rows if r["lote"] in batches]\n    if len(rows) != 5000 or any(r["split"] != "train" for r in rows):\n        raise ValueError("Os cinco lotes do piloto precisam conter 5.000 imagens só de treino.")\n    forbidden = set(forbidden_hashes)\n    return diverse_subset([r for r in rows if r["imagem_sha256"] not in forbidden],\n                          n, seed, per_class)\n\n\ndef validate_metric_block(block, label):\n    for k in ["precision", "recall", "f1"]:\n        x = block["alertas_por_objeto"][k]\n        if x is None or not math.isfinite(x):\n            raise ValueError(f"Métrica sem suporte ou não finita: {label}/{k}")\n    for k in ["alta_ref", "alta_sem_alerta"]:\n        if int(block[k]) < 0:\n            raise ValueError("Contagem inválida.")\n\n\ndef make_rule(baseline_home, candidate_external, dataset_sha, policy_sha):\n    validate_metric_block(baseline_home, "baseline_home")\n    validate_metric_block(candidate_external, "candidate_external")\n    return {\n        "versao": "12-regra-duas-validacoes-v1",\n        "home": {\n            "precision_maior_que": baseline_home["alertas_por_objeto"]["precision"],\n            "recall_minimo": baseline_home["alertas_por_objeto"]["recall"],\n            "alta_sem_alerta_maximo": baseline_home["alta_sem_alerta"],\n            "objetos_alerta_ref": baseline_home["alertas_por_objeto"]["tp"]+baseline_home["alertas_por_objeto"]["fn"],\n            "alta_ref": baseline_home["alta_ref"],\n        },\n        "externa": {\n            "precision_minima": candidate_external["alertas_por_objeto"]["precision"],\n            "recall_minimo": candidate_external["alertas_por_objeto"]["recall"],\n            "alta_sem_alerta_maximo": candidate_external["alta_sem_alerta"],\n            "objetos_alerta_ref": candidate_external["alertas_por_objeto"]["tp"]+candidate_external["alertas_por_objeto"]["fn"],\n            "alta_ref": candidate_external["alta_ref"],\n        },\n        "dataset_sha256": dataset_sha, "politica_sha256": policy_sha,\n        "conf_alertas": .25, "testes_consultados": False,\n        "tolerancia_numerica": 1e-12,\n        "criterio": "Home supera precisão do baseline, preserva recall e omissões altas; externa preserva precisão, recall e omissões altas do candidato 10.",\n        "desempate": "Precisão Home, recall Home, omissões altas Home, F1 externa, menos atualizações, identificador.",\n        "sem_elegivel": "Preservar baseline em uso; candidato 10 e todos os checkpoints continuam disponíveis.",\n    }\n\n\ndef judge(home, external, rule):\n    validate_metric_block(home, "home")\n    validate_metric_block(external, "external")\n    h, e = home["alertas_por_objeto"], external["alertas_por_objeto"]\n    rh, re = rule["home"], rule["externa"]\n    for observed, expected in [(h["tp"]+h["fn"], rh["objetos_alerta_ref"]),\n                               (e["tp"]+e["fn"], re["objetos_alerta_ref"]),\n                               (home["alta_ref"], rh["alta_ref"]),\n                               (external["alta_ref"], re["alta_ref"])]:\n        if observed != expected:\n            raise ValueError("Os denominadores de referência mudaram entre checkpoints.")\n    eps = rule["tolerancia_numerica"]\n    checks = {\n        "home_precision_superior": h["precision"] > rh["precision_maior_que"]+eps,\n        "home_recall_preservado": h["recall"] >= rh["recall_minimo"]-eps,\n        "home_altas_preservadas": home["alta_sem_alerta"] <= rh["alta_sem_alerta_maximo"],\n        "externa_precision_preservada": e["precision"] >= re["precision_minima"]-eps,\n        "externa_recall_preservado": e["recall"] >= re["recall_minimo"]-eps,\n        "externa_altas_preservadas": external["alta_sem_alerta"] <= re["alta_sem_alerta_maximo"],\n    }\n    return {**checks, "elegivel": all(checks.values()),\n            "retencao_externa": all(v for k, v in checks.items() if k.startswith("externa_"))}\n\n\ndef select_checkpoints(entries, rule):\n    if not entries:\n        raise ValueError("Nenhum checkpoint avaliado.")\n    assessed = []\n    for entry in entries:\n        checks = judge(entry["home"], entry["external"], rule)\n        assessed.append({**entry, "checks": checks})\n    eligible = [r for r in assessed if r["checks"]["elegivel"]]\n\n    def ranking(r):\n        h = r["home"]["alertas_por_objeto"]\n        return (-h["precision"], -h["recall"], r["home"]["alta_sem_alerta"],\n                -r["external"]["alertas_por_objeto"]["f1"], r["updates"], r["id"])\n\n    selected = min(eligible, key=ranking) if eligible else None\n    # Apenas para inspeção. A classificação exploratória não muda os requisitos.\n    retained = [r for r in assessed if r["checks"]["retencao_externa"]]\n    exploratory = min(retained, key=lambda r: (\n        -r["home"]["alertas_por_objeto"]["recall"],\n        r["home"]["alta_sem_alerta"],\n        -r["home"]["alertas_por_objeto"]["precision"],\n        -r["external"]["alertas_por_objeto"]["f1"],\n        r["updates"], r["id"])) if retained else None\n    return {"eligible": selected, "exploratory": exploratory, "assessed": assessed}\n', 'exp12_training.py': '"""Treino finito e avaliação isolada dos checkpoints."""\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom contextlib import contextmanager\nfrom pathlib import Path\nimport math\nimport random\nimport hashlib\n\nimport numpy as np\nimport torch\nfrom ultralytics import YOLO\nfrom ultralytics.utils.metrics import ap_per_class\nfrom ultralytics.utils.torch_utils import unwrap_model\n\nimport diag11_core as core\nfrom exp10_partial import TrainerOpenImages, LoaderFinito, export_standard_checkpoint\nfrom exp12_protocol import BudgetSampler\n\n\ndef state_fingerprint(model):\n    digest = hashlib.sha256()\n    for name, value in sorted(unwrap_model(model).state_dict().items()):\n        digest.update(name.encode())\n        digest.update(str(tuple(value.shape)).encode())\n        digest.update(value.detach().cpu().float().contiguous().numpy().tobytes())\n    return digest.hexdigest()\n\n\ndef start_epoch(trainer):\n    trainer.train_loader.sampler.set_epoch(trainer.epoch)\n    trainer.start_epoch_updates = trainer.actual_updates\n\n\ndef check_epoch(trainer):\n    expected = (trainer.epoch+1)*trainer.control_spec["batches"]\n    if trainer.actual_updates != expected or trainer.batches_seen != expected:\n        raise RuntimeError("Orçamento de atualizações ou batches diferente do planejado.")\n    spec = trainer.control_spec\n    expected_sources = {"home": expected*(spec["batch"]-spec["oi_per_batch"])}\n    if spec["oi_per_batch"]:\n        expected_sources["openimages"] = expected*spec["oi_per_batch"]\n    if dict(trainer.source_views) != expected_sources:\n        raise RuntimeError("Exposição efetiva por fonte diferente da proporção planejada.")\n\n\nclass ControlledTrainer(TrainerOpenImages):\n    control_spec = None\n\n    def __init__(self, *args, **kwargs):\n        self.actual_updates, self.batches_seen = 0, 0\n        self.source_views = Counter()\n        super().__init__(*args, **kwargs)\n        if self.control_spec is None:\n            raise ValueError("Especificação do braço não configurada.")\n        self.add_callback("on_train_epoch_start", start_epoch)\n        self.add_callback("on_train_epoch_end", check_epoch)\n\n    def get_dataloader(self, dataset_path, batch_size=16, rank=0, mode="train"):\n        if mode != "train":\n            return super().get_dataloader(dataset_path, batch_size, rank, mode)\n        spec = self.control_spec\n        if (batch_size != spec["batch"] or self.args.nbs != spec["batch"]\n                or self.args.amp or self.world_size > 1 or self.args.compile\n                or self.args.rect or self.args.warmup_epochs != 0):\n            raise RuntimeError("Batch, acumulação, precisão ou dispositivo mudou o orçamento fixado.")\n        if any(float(getattr(self.args, k, 0)) != 0 for k in\n               ["mosaic", "mixup", "cutmix", "copy_paste"]):\n            raise RuntimeError("Augmentação composta não é compatível com a máscara por imagem.")\n        dataset = self.build_dataset(dataset_path, mode, batch_size)\n        home, oi = [], []\n        for index, path in enumerate(dataset.im_files):\n            key = Path(path).resolve().as_posix()\n            if key not in self.source_records:\n                raise RuntimeError("Imagem sem proveniência no loader controlado.")\n            source = self.source_records[key]["source"]\n            if source == "home":\n                home.append(index)\n            elif source == "openimages":\n                oi.append(index)\n            else:\n                raise RuntimeError("Fonte de treino não reconhecida.")\n        if len(home) != spec["home_unique"] or len(oi) != spec["oi_unique"]:\n            raise RuntimeError("Contagem de imagens únicas do braço foi alterada.")\n        sampler = BudgetSampler(home, oi, batch_size, spec["batches"],\n                                spec["oi_per_batch"], self.args.seed)\n        return LoaderFinito(dataset, batch_size=batch_size, sampler=sampler,\n                            num_workers=self.args.workers,\n                            pin_memory=self.device.type == "cuda",\n                            collate_fn=dataset.collate_fn,\n                            generator=torch.Generator().manual_seed(self.args.seed),\n                            drop_last=False)\n\n    def preprocess_batch(self, batch):\n        if len(batch["im_file"]) != self.control_spec["batch"]:\n            raise RuntimeError("Batch incompleto ou redução automática de batch detectada.")\n        if self.accumulate != 1:\n            raise RuntimeError("A acumulação deixou de ser uma atualização por batch.")\n        result = super().preprocess_batch(batch)\n        self.batches_seen += 1\n        for path in batch["im_file"]:\n            self.source_views[self.source_records[Path(path).resolve().as_posix()]["source"]] += 1\n        return result\n\n    def optimizer_step(self):\n        if not torch.isfinite(self.loss.detach()).all():\n            raise FloatingPointError("Loss não finita; o braço foi interrompido.")\n        grads = [torch.isfinite(p.grad).all() for p in self.model.parameters() if p.grad is not None]\n        if grads and not torch.stack(grads).all():\n            raise FloatingPointError("Gradiente não finito; o braço foi interrompido.")\n        if any(not math.isclose(group["lr"], self.control_spec["lr0"], rel_tol=1e-8, abs_tol=1e-14)\n               for group in self.optimizer.param_groups):\n            raise RuntimeError("Taxa de aprendizado diferente da taxa constante predefinida.")\n        super().optimizer_step()\n        self.actual_updates += 1\n\n    def _handle_nan_recovery(self, epoch):\n        # A recuperação silenciosa repetiria épocas e invalidaria o orçamento.\n        if self.loss is not None and not torch.isfinite(self.loss.detach()).all():\n            raise FloatingPointError("Treino não finito; recuperação automática desativada neste controle.")\n        if self.fitness is not None and not math.isfinite(float(self.fitness)):\n            raise FloatingPointError("Fitness não finito.")\n        return False\n\n    def save_model(self):\n        for model in [unwrap_model(self.model), self.ema.ema]:\n            if not all(torch.isfinite(v).all() for v in model.state_dict().values()\n                       if v.is_floating_point()):\n                raise FloatingPointError("Peso não finito antes de salvar o checkpoint.")\n        return super().save_model()\n\n\n@contextmanager\ndef isolated_evaluation():\n    """Validação não altera os fluxos de aleatoriedade do treino."""\n    py_state, np_state = random.getstate(), np.random.get_state()\n    cpu_state = torch.get_rng_state()\n    cuda_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None\n    threads = torch.get_num_threads()\n    cudnn = (torch.backends.cudnn.benchmark, torch.backends.cudnn.deterministic)\n    deterministic = (torch.are_deterministic_algorithms_enabled(),\n                     torch.is_deterministic_algorithms_warn_only_enabled())\n    try:\n        yield\n    finally:\n        random.setstate(py_state)\n        np.random.set_state(np_state)\n        torch.set_rng_state(cpu_state)\n        if cuda_state is not None:\n            torch.cuda.set_rng_state_all(cuda_state)\n        torch.set_num_threads(threads)\n        torch.backends.cudnn.benchmark, torch.backends.cudnn.deterministic = cudnn\n        torch.use_deterministic_algorithms(deterministic[0], warn_only=deterministic[1])\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n\n\ndef evaluate_checkpoint(weight, checkpoint_id, cohorts, output_dir, policy, matcher, device):\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    with isolated_evaluation():\n        model = YOLO(str(weight))\n        if {int(k): str(v) for k, v in model.names.items()} != dict(enumerate(core.NAMES)):\n            raise RuntimeError("Classes do checkpoint incompatíveis com as validações.")\n        result = {}\n        for cohort, records in cohorts.items():\n            if any(r["role"] not in {"home_val", "val_nova"} for r in records):\n                raise RuntimeError("Reserva ou treino entrou na avaliação de checkpoint.")\n            result[cohort] = core.evaluate_model(\n                model, records, "candidato", output_dir/cohort, policy, matcher, device, ap_per_class)\n        del model\n    return result\n\n\ndef metric_entry(checkpoint_id, arm, epoch, updates, weight, evaluated):\n    return {"id": checkpoint_id, "arm": arm, "epoch": epoch, "updates": updates,\n            "weight": str(weight), "weight_sha256": core.sha256_file(weight),\n            "home": evaluated["home_val_historica"]["summary"],\n            "external": evaluated["openimages_val_nova"]["summary"]}\n\n\ndef make_checkpoint_callback(arm, cohorts, output_dir, policy, matcher, device,\n                             entries, results, backup):\n    output_dir = Path(output_dir)\n    done = set()\n\n    def after_save(trainer):\n        epoch = trainer.epoch+1\n        if epoch in done:\n            return\n        check_epoch(trainer)\n        checkpoint_id = f"{arm}_e{epoch:02}"\n        weight = output_dir/"pesos_checkpoints"/(checkpoint_id+".pt")\n        weight.parent.mkdir(parents=True, exist_ok=True)\n        export = export_standard_checkpoint(trainer.last, weight)\n        core.save_json(output_dir/"exports"/(checkpoint_id+".json"), export)\n        # Preserva o peso antes da avaliação mais demorada.\n        backup("checkpoint salvo: "+checkpoint_id)\n        evaluated = evaluate_checkpoint(weight, checkpoint_id, cohorts,\n                                        output_dir/"avaliacoes"/checkpoint_id,\n                                        policy, matcher, device)\n        entry = metric_entry(checkpoint_id, arm, epoch, trainer.actual_updates, weight, evaluated)\n        entries.append(entry)\n        results[checkpoint_id] = evaluated\n        core.save_json(output_dir/"checkpoints_avaliados.json", entries)\n        core.save_json(output_dir/"orcamento"/(checkpoint_id+".json"),\n                       {"arm": arm, "epoch": epoch, "updates": trainer.actual_updates,\n                        "batches_seen": trainer.batches_seen, "source_views": dict(trainer.source_views),\n                        "batch": trainer.control_spec["batch"], "amp": False, "accumulate": trainer.accumulate})\n        done.add(epoch)\n        backup("checkpoint avaliado: "+checkpoint_id)\n    return after_save\n', 'mix18_data.py': '"""Dataset comum: cópias com bytes e cobertura originais para os dois braços."""\nfrom __future__ import annotations\nfrom pathlib import Path\nfrom collections import Counter\nimport shutil\nimport yaml\nimport diag11_core as core\nimport mix18_protocol as protocol\n\ndef make_common(records,home_val,destination,output):\n    destination=Path(destination);output=Path(output);destination.mkdir(parents=True,exist_ok=True)\n    if any(r[\'role\']!=\'train\' or r[\'source\'] not in {\'home\',\'openimages\'} for r in records):raise ValueError(\'Somente TRAIN entra no dataset comum.\')\n    if len({r[\'id\'] for r in records})!=len(records):raise ValueError(\'Treino com IDs repetidos.\')\n    required=sum(Path(r[\'image\']).stat().st_size for r in records)+256*1024**2\n    if shutil.disk_usage(destination).free<required:raise RuntimeError(\'Espaço insuficiente para o treino comum.\')\n    mapping={};manifest=[];frozen=[]\n    for r in protocol.logical_order(records):\n        image=destination/\'images/train\'/protocol.filename(r)\n        label=destination/\'labels/train\'/Path(protocol.filename(r)).with_suffix(\'.txt\')\n        image.parent.mkdir(parents=True,exist_ok=True);label.parent.mkdir(parents=True,exist_ok=True)\n        if core.sha256_file(r[\'image\'])!=r[\'sha256\'] or core.sha256_file(r[\'label\'])!=r[\'label_sha256\']:raise ValueError(\'Bytes de treino alterados antes da cópia.\')\n        if r[\'source\']==\'home\' and r[\'known_mask\']!=4095:raise ValueError(\'Home não pode receber máscara experimental.\')\n        shutil.copy2(r[\'image\'],image);shutil.copy2(r[\'label\'],label)\n        if core.sha256_file(image)!=r[\'sha256\'] or core.sha256_file(label)!=r[\'label_sha256\']:raise ValueError(\'Cópia do treino divergente.\')\n        obj={**r,\'image\':image,\'label\':label}\n        mapping[image.resolve().as_posix()]=obj\n        manifest.append({k:r[k] for k in [\'id\',\'source\',\'role\',\'sha256\',\'label_sha256\',\'known_mask\',\'width\',\'height\']})\n        frozen.extend([(image,r[\'sha256\']),(label,r[\'label_sha256\'])])\n    train=destination/\'train.txt\';train.write_text(\'\\n\'.join(sorted(mapping))+\'\\n\')\n    val=destination/\'home_val.txt\';val.write_text(\'\\n\'.join(str(Path(r[\'image\']).resolve()) for r in home_val)+\'\\n\')\n    data=destination/\'data.yaml\';data.write_text(yaml.safe_dump({\'path\':str(destination.resolve()),\'train\':str(train.resolve()),\'val\':str(val.resolve()),\'names\':dict(enumerate(core.NAMES))},sort_keys=False,allow_unicode=True))\n    frozen.extend([(f,core.sha256_file(f)) for f in [train,val,data]])\n    core.save_json(output/\'manifesto_treino_comum_18.json\',manifest)\n    return {a:{\'records\':mapping,\'yaml\':data} for a in protocol.ARMS},frozen\n', 'mix18_eval.py': '"""Avaliação fixa em 0,25 com avaliador autenticado e isolamento do RNG."""\nfrom __future__ import annotations\nimport gzip,json\nfrom pathlib import Path\nfrom ultralytics import YOLO\nimport diag11_core as core\nimport conf14_evaluation as e14\nfrom exp12_training import isolated_evaluation\nimport mix18_protocol as p16\n\ndef evaluate(weight,checkpoint_id,cohorts,output,policy,matcher,environment,device):\n    output=Path(output);sha=core.sha256_file(weight);results={}\n    if set(cohorts)!={\'home_val_historica\',\'openimages_val_nova\'}:raise ValueError(\'Coortes inesperadas.\')\n    with isolated_evaluation():\n        model=YOLO(str(weight))\n        names=model.names if isinstance(model.names,dict) else dict(enumerate(model.names))\n        if names!=dict(enumerate(core.NAMES)):raise ValueError(\'Classes do modelo incompatíveis.\')\n        for cohort,records in cohorts.items():\n            target=output/cohort\n            cache,meta=e14.obtain_cache(model,records,target/\'cache\',sha,environment,device)\n            preds=e14.read_cache(cache,records)\n            result=e14.evaluate_cache(records,preds,.25,policy,matcher,target)\n            result[\'summary\'][\'modelo\']=checkpoint_id\n            for key in [\'images\',\'events\']:\n                for row in result[key]:row[\'modelo\']=checkpoint_id\n            core.save_json(target/\'metricas.json\',result[\'summary\'])\n            core.save_csv(target/\'alertas_por_imagem.csv\',result[\'images\']);core.save_csv(target/\'avaliacao_objetos.csv\',result[\'events\'])\n            # Cache compacto preserva todas as previsões que participaram da associação em 0,25.\n            compact=target/\'predicoes_conf_025.jsonl.gz\'\n            with gzip.open(compact,\'wt\',encoding=\'utf-8\',compresslevel=4) as f:\n                for r in records:f.write(json.dumps({\'id\':r[\'id\'],\'image_sha256\':r[\'sha256\'],\'label_sha256\':r[\'label_sha256\'],\n                    \'predicoes\':[p for p in preds[r[\'id\']] if p[\'confianca\']>=.25]},ensure_ascii=False)+\'\\n\')\n            result[\'cache_manifest\']=meta;result[\'compact_cache_sha256\']=core.sha256_file(compact)\n            results[cohort]=result\n        del model\n    p16.save_gz(output/\'avaliacao_snapshot.json.gz\',core.json_clean(results))\n    return results\n\ndef entry(checkpoint_id,arm,epoch,updates,relative_weight,weight_sha,evaluated):\n    return {\'id\':checkpoint_id,\'arm\':arm,\'epoch\':epoch,\'updates\':updates,\n            \'weight_relative\':relative_weight,\'weight_sha256\':weight_sha,\n            \'home\':evaluated[\'home_val_historica\'][\'summary\'],\n            \'external\':evaluated[\'openimages_val_nova\'][\'summary\']}\n', 'mix18_history.py': '"""Proveniência dos dois resultados anteriores, sem reutilizar seus pesos ajustados."""\nfrom __future__ import annotations\nimport json,zipfile\nfrom datetime import datetime,timezone\nfrom pathlib import Path\nimport diag11_core as core\n\ndef read_result(path,expected_sha,member):\n    if core.sha256_file(path)!=expected_sha:raise ValueError(\'Resultado anterior diferente do analisado.\')\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist())!=len(set(z.namelist())):raise ValueError(\'Pacote com membros repetidos.\')\n        for name in z.namelist():core.safe_member(name)\n        if z.testzip():raise ValueError(\'Pacote anterior com erro CRC.\')\n        return json.loads(z.read(member))\n\ndef load(package16,package17,expected,output):\n    summary16=read_result(package16,expected[\'pacote_16\'],\'resumo_experimento_16.json\')\n    summary17=read_result(package17,expected[\'pacote_17\'],\'resumo_calibracao_17.json\')\n    if not summary16[\'concluido\'] or summary16[\'checkpoint_final_elegivel\'] is not None:\n        raise ValueError(\'Resultado 16 incompatível com o planejamento.\')\n    if (not summary17[\'concluido\'] or summary17[\'eligible_calibration\']!=0\n        or summary17[\'distinct_configurations_evaluated_verification\']!=0):\n        raise ValueError(\'Resultado 17 incompatível com o planejamento.\')\n    durations=[]\n    with zipfile.ZipFile(package16) as z:\n        context=json.loads(z.read(\'contexto_execucao.json\'))\n        if context[\'context_key\']!=expected[\'contexto_16\']:raise ValueError(\'Contexto 16 diferente.\')\n        entries=json.loads(z.read(\'checkpoints_avaliados.json\'))\n        oldcontrol=next(e for e in entries if e[\'arm\']==\'C_original\' and e[\'epoch\']==3)\n        for arm in [\'C_original\',\'M_home_unknown7\']:\n            marker=json.loads(z.read(\'bracos/\'+arm+\'/concluido.json\'))\n            started=json.loads(z.read(marker[\'attempt_relative\']+\'/tentativa_iniciada.json\'))\n            start=datetime.strptime(started[\'started_at\'],\'%Y%m%d_%H%M%S_%f\').replace(tzinfo=timezone.utc)\n            end=datetime.fromisoformat(marker[\'completed_at\'])\n            durations.append({\'arm_16\':arm,\'seconds\':(end-start).total_seconds(),\n                              \'includes\':\'treino, avaliações por época e backups; não mede só GPU\'})\n    result={\'pacote16_sha256\':expected[\'pacote_16\'],\'pacote17_sha256\':expected[\'pacote_17\'],\n            \'resultado16\':summary16[\'status\'],\'resultado17\':summary17[\'status\'],\n            \'controle16_e03\':oldcontrol,\'tempos_16\':durations,\n            \'decisao\':\'refazer 12/4 e comparar com 8/8 na mesma execução, ambos a partir de C10\',\n            \'estimativa_T4_minutos\':[20,40],\'tempo_pode_exceder\':\'downloads e salvamentos lentos, instalação ou nova tentativa\'}\n    core.save_json(Path(output)/\'historico_e_estimativa.json\',result)\n    return result\n', 'mix18_io.py': '"""Drive, recibos atômicos e pacote leve; credenciais ficam fora dos resultados."""\nfrom __future__ import annotations\nimport base64\nimport configparser\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport zipfile\n\nimport diag11_core as core\n\nclass DigestMismatch(RuntimeError):\n    pass\n\nclass Drive:\n    def __init__(self, binary, config, root=\'gdrive_tcc:TCC_Visao_Computacional\'):\n        self.binary, self.config, self.root = str(binary), str(config), root\n\n    @classmethod\n    def from_kaggle(cls):\n        from kaggle_secrets import UserSecretsClient\n        root = Path(\'/tmp/tcc_rclone\')\n        root.mkdir(parents=True, exist_ok=True, mode=0o700)\n        root.chmod(0o700)\n        try:\n            secret = UserSecretsClient().get_secret(\'GDRIVE_RCLONE_CONFIG_B64\')\n        except Exception:\n            raise RuntimeError(\'Habilite o Secret existente GDRIVE_RCLONE_CONFIG_B64 em Add-ons → Secrets.\') from None\n        try:\n            raw = base64.b64decode(secret.strip(), validate=True)\n            cfg = configparser.ConfigParser(interpolation=None)\n            cfg.read_string(raw.decode(\'utf-8-sig\'))\n        except Exception:\n            raise RuntimeError(\'A configuração do Secret rclone não é válida.\') from None\n        if (not cfg.has_section(\'gdrive_tcc\') or cfg.get(\'gdrive_tcc\', \'type\', fallback=\'\') != \'drive\'\n                or not cfg.get(\'gdrive_tcc\', \'client_id\', fallback=\'\')\n                or not cfg.get(\'gdrive_tcc\', \'token\', fallback=\'\')\n                or cfg.get(\'gdrive_tcc\', \'scope\', fallback=\'\') != \'drive\'):\n            raise RuntimeError(\'O Secret precisa da conexão gdrive_tcc autorizada para leitura e gravação em Meu Drive.\')\n        cfg.remove_option(\'gdrive_tcc\', \'team_drive\')\n        config = root / \'gdrive_tcc.conf\'\n        with config.open(\'w\', encoding=\'utf-8\') as f:\n            cfg.write(f)\n        config.chmod(0o600)\n        del secret, raw, cfg\n        binary = root / \'rclone\'\n        if not binary.is_file():\n            executable = shutil.which(\'rclone\')\n            if not executable:\n                import os\n                prefix = [] if os.geteuid() == 0 else [\'sudo\']\n                subprocess.run(prefix + [\'apt-get\', \'update\', \'-qq\'], check=True)\n                subprocess.run(prefix + [\'apt-get\', \'install\', \'-y\', \'--no-install-recommends\', \'rclone\'], check=True)\n                executable = shutil.which(\'rclone\')\n            if not executable:\n                raise RuntimeError(\'Executável rclone indisponível após a instalação.\')\n            shutil.copy2(executable, binary)\n            binary.chmod(0o700)\n        return cls(binary, config)\n\n    def _call(self, args):\n        return subprocess.run([self.binary, \'--config\', self.config] + list(args)\n                              + [\'--retries\', \'2\', \'--low-level-retries\', \'3\',\n                                 \'--contimeout\', \'15s\', \'--timeout\', \'60s\'],\n                              capture_output=True, text=True)\n\n    def exists(self, relative):\n        result = self._call([\'lsjson\', self.root + \'/\' + relative, \'--stat\'])\n        if result.returncode == 0:\n            return True\n        if result.returncode in {3, 4}:\n            return False\n        raise RuntimeError(\'Falha ao consultar Drive (código \' + str(result.returncode) + \'): \' + relative)\n\n    def fetch(self, relative, target, expected_sha=None, optional=False):\n        target = Path(target)\n        target.parent.mkdir(parents=True, exist_ok=True)\n        receipt = target.with_name(target.name + \'.download.json\')\n        if target.is_file() and target.stat().st_size:\n            local_sha = core.sha256_file(target)\n            prior = {}\n            if receipt.is_file():\n                try:\n                    prior = json.loads(receipt.read_text())\n                except (ValueError, OSError):\n                    pass\n            if ((expected_sha and local_sha == expected_sha)\n                    or (not expected_sha and prior.get(\'origem\') == relative and prior.get(\'sha256\') == local_sha)):\n                print(\'Entrada local conferida:\', relative, flush=True)\n                return target\n        if optional and not self.exists(relative):\n            return None\n        print(\'Baixando:\', relative, flush=True)\n        tmp = target.with_name(target.name + \'.part\')\n        result = self._call([\'copyto\', self.root + \'/\' + relative, str(tmp)])\n        if result.returncode:\n            raise RuntimeError(\'Download interrompido (código \' + str(result.returncode) + \'): \' + relative)\n        if not tmp.is_file() or not tmp.stat().st_size:\n            raise RuntimeError(\'Download vazio: \' + relative)\n        actual = core.sha256_file(tmp)\n        if expected_sha and actual != expected_sha:\n            tmp.replace(target.with_name(target.name + \'.rejeitado_\' + actual[:12]))\n            raise DigestMismatch(\'SHA-256 diferente da fonte autenticada: \' + relative)\n        tmp.replace(target)\n        core.save_json(receipt, {\'origem\': relative, \'sha256\': actual, \'bytes\': target.stat().st_size})\n        return target\n\n    def backup(self, local, relative, required=True):\n        print(\'Salvando resultados no Drive...\', flush=True)\n        result = self._call([\'copy\', str(local), self.root + \'/\' + relative,\n                             \'--exclude\', \'**/*.tmp\', \'--exclude\', \'**/*.part\', \'--exclude\', \'**/__pycache__/**\'])\n        if result.returncode and required:\n            raise RuntimeError(\'Falha no salvamento no Drive; resultados locais preservados. Código \' + str(result.returncode))\n        return result.returncode == 0\n\n    def restore(self, relative, local):\n        if not self.exists(relative):\n            return False\n        result = self._call([\'copy\', self.root + \'/\' + relative, str(local),\n                             \'--exclude\', \'*.zip\', \'--exclude\', \'**/*.tmp\', \'--exclude\', \'**/*.part\'])\n        if result.returncode:\n            raise RuntimeError(\'Não foi possível recuperar o resultado anterior; código \' + str(result.returncode))\n        return True\n', 'mix18_protocol.py': '"""Experimento 18: duas proporções, mesmo ponto inicial e orçamento."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport math\nfrom pathlib import Path\nimport numpy as np\nimport control16_protocol as historical\nfrom exp12_protocol import BudgetSampler\n\nVERSION=\'18-proporcao-fontes-1.0.0\'\nARMS=(\'C12_4\',\'E8_8\')\nSPEC={k:v for k,v in historical.SPEC.items() if k!=\'oi_per_batch\'}\nSPECS={a:{**SPEC,\'oi_per_batch\':oi} for a,oi in zip(ARMS,(4,8))}\nAUGMENTATION_ZERO=historical.AUGMENTATION_ZERO\ncanonical=historical.canonical\ndigest=historical.digest\nsave_gz=historical.save_gz\nload_gz=historical.load_gz\nfilename=historical.filename\nlogical_order=historical.logical_order\nschedule=historical.schedule\nsafe_judge=historical.safe_judge\n\ndef validate_specs(specs=SPECS):\n    if tuple(specs)!=ARMS:raise ValueError(\'Braços diferentes do protocolo.\')\n    if any({k:v for k,v in specs[a].items() if k!=\'oi_per_batch\'}!=SPEC for a in ARMS):\n        raise ValueError(\'Somente a proporção por fonte pode variar.\')\n    if [specs[a][\'oi_per_batch\'] for a in ARMS]!=[4,8]:raise ValueError(\'Proporção não autorizada.\')\n    return True\n\ndef audit_source_streams(records,specs=SPECS):\n    """Compara os fluxos antes da permutação das posições dentro do batch."""\n    validate_specs(specs)\n    ordered=logical_order(records)\n    indices={s:[i for i,r in enumerate(ordered) if r[\'source\']==s] for s in (\'home\',\'openimages\')}\n    checks=[]\n    for epoch in range(SPEC[\'epochs\']):\n        streams={}\n        for arm in ARMS:\n            spec=specs[arm]\n            sampler=BudgetSampler(indices[\'home\'],indices[\'openimages\'],spec[\'batch\'],spec[\'batches\'],spec[\'oi_per_batch\'],spec[\'seed\'])\n            sampler.set_epoch(epoch)\n            streams[arm]={s:sampler._stream(indices[s],spec[\'batches\']*(spec[\'batch\']-spec[\'oi_per_batch\'] if s==\'home\' else spec[\'oi_per_batch\']),j)\n                          for j,s in enumerate((\'home\',\'openimages\'))}\n        for source in indices:\n            a,b=(streams[arm][source] for arm in ARMS);n=min(len(a),len(b))\n            if a[:n]!=b[:n]:raise ValueError(\'Os fluxos aleatórios por fonte deixaram de ser comuns.\')\n            checks.append({\'epoch\':epoch+1,\'source\':source,\'common_prefix\':n,\n                           \'common_prefix_sha256\':digest([ordered[i][\'id\'] for i in a[:n]])})\n    return {\'fluxos_por_fonte_com_prefixo_comum\':True,\'antes_permutacao_intrabatch\':True,\'checks\':checks}\n\ndef target_counts(record):\n    rows=np.array([list(map(float,l.split())) for l in Path(record[\'label\']).read_text(encoding=\'utf-8-sig\').splitlines() if l.strip()],dtype=np.float32).reshape(-1,5)\n    # O loader autenticado remove linhas integralmente repetidas; nenhuma classe é removida.\n    return dict(Counter(str(int(row[0])) for row in np.unique(rows,axis=0)))\n\ndef audit_runs(results,plans,records,specs=SPECS):\n    validate_specs(specs)\n    reference={r[\'id\']:r for r in records}\n    if len(reference)!=len(records):raise ValueError(\'IDs repetidos no treino.\')\n    targets={iid:target_counts(r) for iid,r in reference.items()}\n    initial=[];summaries=[]\n    for arm in ARMS:\n        result=results[arm];spec=specs[arm];logs=result[\'logs\'];plan=plans[arm]\n        if len(logs)!=spec[\'epochs\']*spec[\'batches\'] or len(logs)!=len(plan):raise ValueError(\'Orçamento observado incompleto.\')\n        initial.append(result[\'marker\'][\'initial_state_sha256\'])\n        views=Counter();ids=set()\n        for row,p in zip(logs,plan):\n            if any(row[k]!=p[k] for k in [\'step\',\'epoch\',\'ids\',\'input_signature\']):raise ValueError(\'Batch diverge do plano por fonte.\')\n            counts=Counter(reference[i][\'source\'] for i in row[\'ids\'])\n            expected={\'home\':spec[\'batch\']-spec[\'oi_per_batch\'],\'openimages\':spec[\'oi_per_batch\']}\n            if counts!=expected or row[\'source_counts\']!=expected:raise ValueError(\'Proporção efetiva diferente do braço.\')\n            if row[\'gradient_audit\'][\'unknown_negative_max\']!=0:raise ValueError(\'Gradiente negativo vazou em classe desconhecida.\')\n            if len(row[\'known_masks\'])!=spec[\'batch\'] or len(row[\'gt_per_image_by_class\'])!=spec[\'batch\']:\n                raise ValueError(\'Máscaras/alvos do batch incompletos.\')\n            for j,iid in enumerate(row[\'ids\']):\n                r=reference[iid]\n                actual={str(k):v for k,v in row[\'gt_per_image_by_class\'][j].items()}\n                if row[\'known_masks\'][j]!=r[\'known_mask\'] or actual!=targets[iid]:raise ValueError(\'Alvos ou cobertura alterados entre fontes.\')\n                if r[\'source\']==\'home\' and r[\'known_mask\']!=4095:raise ValueError(\'Home precisa manter cobertura completa.\')\n            views.update(counts);ids.update(row[\'ids\'])\n        if dict(views)!=result[\'budget\'][\'source_views\']:raise ValueError(\'Exposições divergentes do orçamento salvo.\')\n        summaries.append({\'arm\':arm,\'batches\':len(logs),\'source_views\':dict(views),\n                          \'unique_exposed\':dict(Counter(reference[i][\'source\'] for i in ids))})\n    if len(set(initial))!=1:raise ValueError(\'Pesos iniciais diferentes entre os braços.\')\n    return {\'auditoria_conferida\':True,\'mesmo_estado_inicial\':True,\'mesmos_dados_disponiveis\':True,\n            \'rotulos_e_cobertura_originais_preservados\':True,\'gradiente_negativo_desconhecido_zero\':True,\n            \'mesmo_total_atualizacoes\':True,\'diferenca_planejada\':\'quantidade de imagens por fonte em cada batch\',\n            \'mesmos_ids_por_batch\':False,\'bracos\':summaries,\n            \'fluxos\':audit_source_streams(records,specs)}\n\ndef select_final(entries,rule):\n    assessed=[{**e,\'checks\':safe_judge(e[\'home\'],e[\'external\'],rule)} for e in entries]\n    primary=[e for e in assessed if e[\'arm\'] in ARMS and e[\'epoch\']==SPEC[\'epochs\']]\n    if len(primary)!=2 or {e[\'arm\'] for e in primary}!=set(ARMS):raise ValueError(\'Exigidos os dois finais e03.\')\n    candidate=next(e for e in primary if e[\'arm\']==ARMS[1])\n    eligible=candidate if candidate[\'checks\'][\'elegivel\'] else None\n    return {\'assessed\':assessed,\'primary_ids\':[e[\'id\'] for e in primary],\n            \'candidate_id\':candidate[\'id\'],\'eligible\':eligible,\n            \'status\':\'E8_8_e03_elegivel_em_validacao\' if eligible else \'E8_8_e03_nao_cumpre_seis_criterios\',\n            \'checkpoint_predefinido\':\'E8_8_e03\',\'controle_predefinido\':\'C12_4_e03\',\n            \'intermediarios\':\'diagnostico; sem seleção por melhor época\',\n            \'modelo_substituido\':False,\'modelo_em_uso_continua_baseline\':True}\n', 'mix18_reports.py': '"""Desfecho pareado e03, comparações por classe e pacote compacto."""\nfrom __future__ import annotations\nimport html,json\nfrom pathlib import Path\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport diag11_core as core\nimport mix18_protocol as p16\n\ndef table(rows):\n    if not rows:return \'<p>Sem registros.</p>\'\n    keys=list(rows[0]);out=\'<div class="scroll"><table><tr>\'+\'\'.join(\'<th>\'+html.escape(k)+\'</th>\' for k in keys)+\'</tr>\'\n    for r in rows:out+=\'<tr>\'+\'\'.join(\'<td>\'+html.escape(str(r.get(k,\'\')))+\'</td>\' for k in keys)+\'</tr>\'\n    return out+\'</table></div>\'\n\ndef report(output,entries,evaluations,references,rule,pairing,config,budgets):\n    output=Path(output);selection=p16.select_final(entries,rule);rows=[]\n    for e in selection[\'assessed\']:\n        row={k:e[k] for k in [\'id\',\'arm\',\'epoch\',\'updates\',\'weight_sha256\']}\n        for field,prefix in [(\'home\',\'home\'),(\'external\',\'externa\')]:\n            s=e[field];row.update({prefix+\'_\'+k:v for k,v in s[\'alertas_por_objeto\'].items()})\n            row[prefix+\'_alta_sem_alerta\']=s[\'alta_sem_alerta\']\n        row.update(e[\'checks\']);row[\'desfecho_primario\']=e[\'id\'] in selection[\'primary_ids\'];rows.append(row)\n    core.save_csv(output/\'comparacao_checkpoints.csv\',rows);core.save_json(output/\'selecao_checkpoints.json\',selection)\n    finals={e[\'arm\']:e for e in selection[\'assessed\'] if e[\'id\'] in selection[\'primary_ids\']}\n    c,m=[finals[a] for a in p16.ARMS];delta=[];class_rows=[];byimage=[]\n    for cohort,label in [(\'home_val_historica\',\'home\'),(\'openimages_val_nova\',\'externa\')]:\n        ca=evaluations[c[\'id\']][cohort];ma=evaluations[m[\'id\']][cohort]\n        for metric in [\'tp\',\'fn\',\'fp\',\'precision\',\'recall\',\'f1\']:\n            x=ca[\'summary\'][\'alertas_por_objeto\'][metric];y=ma[\'summary\'][\'alertas_por_objeto\'][metric]\n            delta.append({\'coorte\':cohort,\'metrica\':metric,\'controle_e03\':x,\'equilibrado_e03\':y,\n                          \'delta_equilibrado_menos_controle\':y-x if x is not None and y is not None else None,\n                          \'unidade\':\'proporcao; multiplicar delta por 100 para pontos percentuais\' if metric in {\'precision\',\'recall\',\'f1\'} else \'contagem\'})\n        for metric in [\'alta_ref\',\'alta_sem_alerta\']:\n            x=ca[\'summary\'][metric];y=ma[\'summary\'][metric]\n            delta.append({\'coorte\':cohort,\'metrica\':metric,\'controle_e03\':x,\'equilibrado_e03\':y,\n                          \'delta_equilibrado_menos_controle\':y-x,\'unidade\':\'contagem\'})\n        cc={r[\'classe_id\']:r for r in ca[\'classes\']};mc={r[\'classe_id\']:r for r in ma[\'classes\']}\n        for cid,name in enumerate(core.NAMES):\n            row={\'coorte\':cohort,\'classe_id\':cid,\'classe\':name}\n            for key in [\'gt_avaliaveis\',\'tp\',\'fn\',\'fp\',\'precision\',\'recall\',\'alta_sem_alerta\']:\n                x,y=cc[cid][key],mc[cid][key]\n                row.update({\'controle_\'+key:x,\'equilibrado_\'+key:y,\'delta_\'+key:y-x if x is not None and y is not None else None})\n            class_rows.append(row)\n        ci={r[\'id\']:r for r in ca[\'images\']};mi={r[\'id\']:r for r in ma[\'images\']}\n        if ci.keys()!=mi.keys():raise RuntimeError(\'Comparação final exige os mesmos IDs.\')\n        for iid in ci:\n            r={\'coorte\':cohort,\'id\':iid}\n            for key in [\'tp_alerta\',\'fn_alerta\',\'fp_alerta\',\'alta_sem_alerta\']:\n                r.update({\'controle_\'+key:ci[iid][key],\'equilibrado_\'+key:mi[iid][key],\'delta_\'+key:mi[iid][key]-ci[iid][key]})\n            byimage.append(r)\n    core.save_csv(output/\'efeito_pareado_e03.csv\',delta)\n    core.save_csv(output/\'efeito_pareado_e03_por_classe.csv\',class_rows)\n    core.save_csv(output/\'efeito_pareado_e03_por_imagem.csv\',byimage)\n    fig,axes=plt.subplots(3,2,figsize=(12,9),layout=\'constrained\')\n    initial=next(e for e in entries if e[\'arm\']==\'inicial\')\n    for col,(field,title) in enumerate([(\'home\',\'Home VAL histórica\'),(\'external\',\'Validação externa de desenvolvimento\')]):\n        for arm,color in zip(p16.ARMS,[\'#2176b5\',\'#d65a2f\']):\n            seq=[initial]+sorted([e for e in entries if e[\'arm\']==arm],key=lambda e:e[\'epoch\'])\n            x=[e[\'updates\'] for e in seq]\n            for row,metric in enumerate([\'precision\',\'recall\',\'alta_sem_alerta\']):\n                ys=[e[field][metric] if metric==\'alta_sem_alerta\' else e[field][\'alertas_por_objeto\'][metric] for e in seq]\n                ys=[None if y is None else y if metric==\'alta_sem_alerta\' else 100*y for y in ys]\n                axes[row,col].plot(x,ys,\'o-\',label=arm,color=color)\n                axes[row,col].grid(alpha=.25);axes[row,col].set_ylabel(metric+(\' (%)\' if metric!=\'alta_sem_alerta\' else \'\'))\n        axes[0,col].set_title(title);axes[0,col].legend(fontsize=8);axes[2,col].set_xlabel(\'Atualizações por braço\')\n    fig.savefig(output/\'curvas_validacao.png\',dpi=150);plt.close(fig)\n    chosen=selection[\'eligible\']\n    status=(\'O candidato equilibrado e03 cumpriu os seis requisitos nas validações: \'+chosen[\'id\']+\'.\' if chosen else\n            \'O candidato equilibrado e03 não cumpriu os seis requisitos. O baseline continua como referência.\')\n    summary={\'versao\':p16.VERSION,\'concluido\':True,\'status\':selection[\'status\'],\'mensagem\':status,\n        \'checkpoint_final_elegivel\':chosen[\'id\'] if chosen else None,\'desfecho_primario\':\'comparacao_e03_vs_e03\',\n        \'atualizacoes_finais_por_braco\':budgets[0][\'updates\'],\n        \'orcamentos\':budgets,\'pareamento\':pairing,\'novos_bracos_concluidos\':2,\n        \'modelo_substituido\':False,\'modelo_em_uso_continua_baseline\':True,\n        \'rotulos_originais_preservados\':True,\'projecao_treino_home7_documentada\':False,\n        \'validacao_rotulos_alterados\':False,\'reserva_futura_extraida\':False,\'reserva_futura_inferida\':False,\n        \'test_home_inferido\':False,\'test_oi20k_inferido\':False,\'preenchimento_manual_obrigatorio\':False,\n        \'limites\':[\'Uma semente e orçamento curto; não prova superioridade geral ou causalidade fora deste controle.\',\n                  \'As duas validações já participaram do desenvolvimento; não são teste final independente.\',\n                  \'Home pode ter diferenças semânticas também na avaliação, preservada igualmente para ambos.\',\n                  \'O candidato final e03 foi pré-fixado; cumprir limites não prova superioridade estatística sobre 12/4.\',\n                  \'Alertas medem concordância com a política V2, sem validação independente de risco físico.\',\n                  \'A origem do treino original do baseline permanece não determinada.\'],\n        \'configuracao\':config,\'candidato_predefinido\':\'E8_8_e03\',\'controle_predefinido\':\'C12_4_e03\'}\n    core.save_json(output/\'resumo_experimento_18.json\',summary)\n    display_rows=[]\n    for row in rows:\n        display_rows.append({\'Checkpoint\':row[\'id\'],\'Primário e03\':row[\'desfecho_primario\'],\n            \'P Home\':None if row[\'home_precision\'] is None else f"{100*row[\'home_precision\']:.2f}%",\n            \'R Home\':None if row[\'home_recall\'] is None else f"{100*row[\'home_recall\']:.2f}%",\n            \'Alta omitida Home\':row[\'home_alta_sem_alerta\'],\n            \'P externa\':None if row[\'externa_precision\'] is None else f"{100*row[\'externa_precision\']:.2f}%",\n            \'R externa\':None if row[\'externa_recall\'] is None else f"{100*row[\'externa_recall\']:.2f}%",\n            \'Alta omitida externa\':row[\'externa_alta_sem_alerta\'],\'Seis critérios\':row[\'elegivel\']})\n    page=\'\'\'<!doctype html><html lang="pt-BR"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Experimento 18 — Proporção das fontes</title><style>body{font:16px/1.6 system-ui;background:#eef3f8;color:#152538;margin:0}main{max-width:1450px;margin:auto;padding:30px}.note{padding:20px;background:white;border:1px solid #dce5ee;border-radius:10px;margin:20px 0}.scroll{overflow:auto}table{border-collapse:collapse;background:white;width:100%}td,th{border:1px solid #dce5ee;padding:9px;text-align:left}img{max-width:100%;height:auto}h1{line-height:1.15}</style><main><p>TCC · Visão computacional</p><h1>Experimento 18 — Mistura 12/4 versus 8/8</h1>\'\'\'\n    page+=\'<div class="note"><strong>\'+html.escape(status)+\'</strong><p>Controle C12_4: 12 Home + 4 Open Images por batch. Candidato E8_8: 8 + 8. Ambos preservam os rótulos e a cobertura originais e partem do candidato 10. A comparação principal usa e03 contra e03; o candidato foi fixado antes do treino.</p></div>\'\n    page+=\'<h2>Checkpoints nas duas validações</h2>\'+table(display_rows)+\'<img src="curvas_validacao.png" alt="Precisão, recall e omissões altas nas duas validações">\'\n    page+=\'<h2>Efeito pareado no desfecho fixo e03</h2>\'+table(delta)\n    page+=\'<h2>Pareamento e orçamento</h2><p>Mesmo conjunto disponível, estado inicial, 342 atualizações e parâmetros. A diferença planejada é a exposição por fonte: 4.104 Home / 1.368 OI no controle, 2.736 / 2.736 no candidato. Os IDs de cada batch diferem conforme a composição. Os fluxos por fonte compartilham prefixos antes da permutação interna. Ver auditoria_fontes.json e os diários de batches.</p>\'\n    page+=\'<p>Ver efeito_pareado_e03_por_classe.csv para conferir se os ganhos e perdas se concentram em alguma classe. O atendimento aos limites históricos e o efeito da proporção sobre o controle são resultados distintos; uma semente não demonstra superioridade geral.</p>\'\n    page+=\'<h2>Limites</h2><ul>\'+\'\'.join(\'<li>\'+html.escape(t)+\'</li>\' for t in summary[\'limites\'])+\'</ul></main></html>\'\n    (output/\'relatorio_experimento_18.html\').write_text(page,encoding=\'utf-8\')\n    (output/\'LEIA_RESULTADO.md\').write_text(\'# Experimento 18\\n\\n\'+status+\'\\n\\nResultados em pacote_experimento_18.zip. O desfecho principal é e03 contra e03; e01/e02 são diagnósticos. Nenhum modelo em uso foi substituído. Pesos padrão e caches completos ficam separados no Drive. A confiança permanece em 0,25; não é uma nova busca de limiares. Não há formulário para preencher.\\n\',encoding=\'utf-8\')\n    return summary,selection\n', 'mix18_run.py': '"""Execução com backups por época e reaproveitamento apenas de braços completos."""\nfrom __future__ import annotations\nimport json,time,gc\nfrom pathlib import Path\nimport torch\nfrom ultralytics import YOLO\nimport diag11_core as core\nfrom exp10_partial import export_standard_checkpoint\nfrom exp12_training import check_epoch\nimport mix18_protocol as p16\nimport mix18_training as tr16\nimport mix18_eval as ev16\nimport mix18_state as state\n\ndef run_arm(arm,data,initial,initial_state,spec,planned,context_key,cohorts,policy,matcher,environment,\n            output,local,device,backup,arguments):\n    output=Path(output)\n    reused=state.load_completed(output,arm,context_key,initial_state,spec,planned)\n    if reused:\n        print(\'Braço completo conferido e reutilizado:\',arm,flush=True)\n        return reused\n    attempt=state.new_attempt(output,arm);entries=[];evaluations={};done=set();start=time.monotonic()\n    spec={**spec,\'seed\':arguments[\'seed\']}\n    Trainer=type(\'Trainer_\'+arm,(tr16.SourceMixTrainer,),{\n        \'source_records\':data[\'records\'],\'control_spec\':spec,\'arm\':arm,\n        \'expected_schedule\':planned,\'attempt_dir\':attempt,\'expected_initial_state\':initial_state})\n    model=YOLO(str(initial))\n    def after_save(trainer):\n        epoch=trainer.epoch+1\n        if epoch in done:return\n        check_epoch(trainer)\n        if len(trainer.batch_log)!=trainer.actual_updates:raise RuntimeError(\'Diário de batches incompleto.\')\n        cid=f\'{arm}_e{epoch:02d}\';weight=attempt/\'pesos\'/(cid+\'.pt\');weight.parent.mkdir(exist_ok=True)\n        export=export_standard_checkpoint(trainer.last,weight)\n        core.save_json(attempt/\'exports\'/(cid+\'.json\'),export)\n        budget={\'arm\':arm,\'epoch\':epoch,\'updates\':trainer.actual_updates,\'batches_seen\':trainer.batches_seen,\n                \'source_views\':dict(trainer.source_views),\'amp\':False,\'accumulate\':trainer.accumulate}\n        core.save_json(attempt/\'orcamento\'/(cid+\'.json\'),budget)\n        backup(\'checkpoint salvo: \'+cid)\n        evaluation_dir=attempt/\'avaliacoes\'/cid\n        evaluated=ev16.evaluate(weight,cid,cohorts,evaluation_dir,policy,matcher,environment,device)\n        entry=ev16.entry(cid,arm,epoch,trainer.actual_updates,str(weight.relative_to(output)),core.sha256_file(weight),evaluated)\n        entry[\'snapshot_relative\']=str((evaluation_dir/\'avaliacao_snapshot.json.gz\').relative_to(output))\n        entries.append(entry);evaluations[cid]=evaluated;done.add(epoch)\n        core.save_json(attempt/\'checkpoints.json\',entries)\n        elapsed=time.monotonic()-start\n        print(f\'{arm}: época curta {epoch}/{spec["epochs"]}; {trainer.actual_updates}/{spec["epochs"]*spec["batches"]} atualizações; tempo {elapsed/60:.1f} min\',flush=True)\n        backup(\'checkpoint avaliado: \'+cid)\n    model.add_callback(\'on_model_save\',after_save)\n    try:\n        print(\'Iniciando\',arm,\'| atualizações planejadas:\',spec[\'epochs\']*spec[\'batches\'],flush=True)\n        model.train(trainer=Trainer,data=str(data[\'yaml\']),project=str(Path(local)/\'treinos\'),\n                    name=arm+\'_\'+attempt.name,**arguments)\n        trainer=model.trainer;check_epoch(trainer)\n        budget={\'arm\':arm,\'epoch\':spec[\'epochs\'],\'updates\':trainer.actual_updates,\'batches_seen\':trainer.batches_seen,\n                \'source_views\':dict(trainer.source_views),\'amp\':False,\'accumulate\':trainer.accumulate}\n        marker=state.seal(output,arm,attempt,context_key,entries,budget,initial_state,spec)\n        backup(\'braço concluído: \'+arm)\n        result={\'marker\':marker,\'attempt\':attempt,\'entries\':entries,\'evaluations\':evaluations,\n                \'budget\':budget,\'logs\':list(trainer.batch_log),\'elapsed_seconds\':time.monotonic()-start}\n        del trainer,model;gc.collect()\n        if torch.cuda.is_available():torch.cuda.empty_cache()\n        return result\n    except Exception as exc:\n        core.save_json(attempt/\'tentativa_interrompida.json\',{\'arm\':arm,\'concluido\':False,\'error_type\':type(exc).__name__,\n            \'message\':str(exc)[:1200],\'last_evaluated_epochs\':sorted(done),\n            \'restart_policy\':\'reiniciar este braço do candidato 10; nunca carregar o otimizador interrompido\'})\n        try:backup(\'tentativa interrompida: \'+arm,obrigatorio=False)\n        except Exception:pass\n        raise\n', 'mix18_state.py': '"""Recuperação por braço completo; tentativa incompleta reinicia do ponto comum."""\nfrom __future__ import annotations\nimport json,zipfile\nfrom pathlib import Path,PurePosixPath\nfrom datetime import datetime,timezone\nimport diag11_core as core\nimport mix18_protocol as p16\n\ndef safe_child(root,relative):\n    q=PurePosixPath(relative)\n    if q.is_absolute() or \'..\' in q.parts or \'\\\\\' in relative:raise ValueError(\'Caminho de recuperação inválido.\')\n    root=Path(root).resolve();path=root/relative\n    if not path.resolve().is_relative_to(root):raise ValueError(\'Caminho fora da execução.\')\n    return path\n\ndef new_attempt(output,arm):\n    if arm not in p16.ARMS:raise ValueError(\'Braço desconhecido.\')\n    stamp=datetime.now(timezone.utc).strftime(\'%Y%m%d_%H%M%S_%f\')\n    path=Path(output)/\'bracos\'/arm/\'tentativas\'/stamp;path.mkdir(parents=True)\n    core.save_json(path/\'tentativa_iniciada.json\',{\'arm\':arm,\'started_at\':stamp,\n                    \'resume_mode\':\'reinicio_do_checkpoint_inicial; sem carregar otimizador ou EMA de tentativa interrompida\'})\n    return path\n\ndef validate_budget(budget,spec):\n    count=spec[\'epochs\']*spec[\'batches\']\n    if budget[\'updates\']!=count or budget[\'batches_seen\']!=count:raise ValueError(\'Orçamento final divergente.\')\n    expected={\'home\':count*(spec[\'batch\']-spec[\'oi_per_batch\']),\'openimages\':count*spec[\'oi_per_batch\']}\n    if budget[\'source_views\']!=expected or budget[\'amp\'] or budget[\'accumulate\']!=1:raise ValueError(\'Exposição/precisão/acumulação divergente.\')\n\ndef seal(output,arm,attempt,context_key,entries,budget,initial_state,spec):\n    output=Path(output);attempt=Path(attempt)\n    validate_budget(budget,spec)\n    if len(entries)!=spec[\'epochs\'] or {e[\'epoch\'] for e in entries}!=set(range(1,spec[\'epochs\']+1)):\n        raise ValueError(\'Checkpoints do braço incompletos.\')\n    core.save_json(attempt/\'checkpoints.json\',entries);core.save_json(attempt/\'orcamento_final.json\',budget)\n    files={str(f.relative_to(attempt)):core.sha256_file(f) for f in sorted(attempt.rglob(\'*\')) if f.is_file() and not f.name.endswith(\'.tmp\')}\n    if not files:raise ValueError(\'Tentativa vazia.\')\n    obj={\'concluido\':True,\'arm\':arm,\'context_key\':context_key,\'attempt_relative\':str(attempt.relative_to(output)),\n         \'files\':files,\'entries_sha256\':p16.digest(entries),\'initial_state_sha256\':initial_state,\n         \'budget\':budget,\'completed_at\':datetime.now(timezone.utc).isoformat()}\n    core.save_json(output/\'bracos\'/arm/\'concluido.json\',obj)\n    return obj\n\ndef load_completed(output,arm,context_key,initial_state,spec,planned):\n    output=Path(output);marker=output/\'bracos\'/arm/\'concluido.json\'\n    if not marker.is_file():return None\n    try:\n        obj=json.loads(marker.read_text())\n        if not obj[\'concluido\'] or obj[\'arm\']!=arm or obj[\'context_key\']!=context_key or obj[\'initial_state_sha256\']!=initial_state:\n            raise ValueError(\'Contexto/inicialização do braço incompatível.\')\n        attempt=safe_child(output,obj[\'attempt_relative\'])\n        if attempt.parent.parent!=(output/\'bracos\'/arm).resolve():raise ValueError(\'Tentativa fora do braço esperado.\')\n        for rel,sha in obj[\'files\'].items():\n            f=safe_child(attempt,rel)\n            if not f.is_file() or core.sha256_file(f)!=sha:raise ValueError(\'Arquivo concluído ausente ou alterado: \'+rel)\n        entries=json.loads((attempt/\'checkpoints.json\').read_text())\n        if p16.digest(entries)!=obj[\'entries_sha256\']:raise ValueError(\'Entradas de checkpoints alteradas.\')\n        if len(entries)!=spec[\'epochs\'] or {e[\'epoch\'] for e in entries}!=set(range(1,spec[\'epochs\']+1)):\n            raise ValueError(\'Lista de checkpoints concluídos incompleta.\')\n        budget=json.loads((attempt/\'orcamento_final.json\').read_text());validate_budget(budget,spec)\n        if budget!=obj[\'budget\']:raise ValueError(\'Orçamento diverge do marcador de conclusão.\')\n        logs=[json.loads(line) for line in (attempt/\'auditoria_batches.jsonl\').read_text().splitlines() if line.strip()]\n        if len(logs)!=len(planned):raise ValueError(\'Diário de batches incompleto.\')\n        for row,p in zip(logs,planned):\n            if any(row[k]!=p[k] for k in [\'step\',\'epoch\',\'ids\',\'input_signature\']):raise ValueError(\'Cronograma concluído incompatível.\')\n            if row[\'gradient_audit\'][\'unknown_negative_max\']!=0:raise ValueError(\'Auditoria de gradiente inconsistente.\')\n        evaluations={}\n        for e in entries:\n            weight=safe_child(output,e[\'weight_relative\'])\n            if core.sha256_file(weight)!=e[\'weight_sha256\']:raise ValueError(\'Peso de checkpoint alterado.\')\n            snapshot=safe_child(output,e[\'snapshot_relative\'])\n            evaluations[e[\'id\']]=p16.load_gz(snapshot)\n        return {\'marker\':obj,\'attempt\':attempt,\'entries\':entries,\'budget\':budget,\'logs\':logs,\'evaluations\':evaluations}\n    except (KeyError,TypeError,ValueError,OSError,EOFError) as exc:\n        core.save_json(output/\'recuperacao\'/(arm+\'_rejeitado.json\'),{\'arm\':arm,\'motivo\':str(exc)[:500],\n                       \'acao\':\'reiniciar braço do checkpoint inicial; resultado incompleto não entra na seleção\'})\n        print(\'Resultado incompleto/incompatível: o braço será reiniciado:\',arm,flush=True)\n        return None\n\ndef package(output):\n    output=Path(output);archive=output/\'pacote_experimento_18.zip\';tmp=archive.with_name(archive.name+\'.tmp\')\n    selected=[]\n    for arm in p16.ARMS:\n        marker=output/\'bracos\'/arm/\'concluido.json\'\n        if marker.is_file():\n            try:selected.append(json.loads(marker.read_text())[\'attempt_relative\'])\n            except (ValueError,KeyError):pass\n    with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_DEFLATED,compresslevel=4) as z:\n        for f in sorted(output.rglob(\'*\')):\n            if not f.is_file() or f in {archive,tmp} or f.name.endswith(\'.tmp\') or f.suffix in {\'.pt\',\'.pkl\'}:continue\n            rel=f.relative_to(output)\n            if f.name==\'predicoes_conf_0p001.jsonl.gz\':continue\n            if rel.parts[0]==\'bracos\' and \'tentativas\' in rel.parts:\n                attempt=str(Path(*rel.parts[:4]))\n                if attempt not in selected:\n                    if f.name in {\'tentativa_iniciada.json\',\'tentativa_interrompida.json\'}:z.write(f,rel)\n                    continue\n            z.write(f,rel)\n    tmp.replace(archive)\n    return archive\n', 'mix18_training.py': '"""Treino por fonte com alvos originais e auditoria da BCE parcial."""\nfrom __future__ import annotations\nimport hashlib,json,random,math\nfrom pathlib import Path\nfrom collections import Counter\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom ultralytics.nn.tasks import DetectionModel\nfrom ultralytics.utils.torch_utils import unwrap_model\nimport diag11_core as core\nfrom exp10_partial import BCEComCobertura,DetectionLossParcial\nfrom exp12_training import ControlledTrainer,state_fingerprint\nimport mix18_protocol as p16\n\n\nclass AuditedBCE(BCEComCobertura):\n    def __init__(self,original=None):\n        super().__init__(original);self.home_mask=None;self.audit_step=0;self.last_audit=None\n\n    def forward(self,logits,target):\n        result=super().forward(logits,target)\n        if logits.requires_grad:\n            known=self.known.detach().clone();home=self.home_mask.detach().clone();step=self.audit_step\n            unknown_negative=(~known[:,None,:]) & (target.detach()==0)\n            positives_unknown=(~known[:,None,:]) & (target.detach()>0)\n            def check_gradient(grad):\n                max_unknown=float(grad[unknown_negative].abs().max().item()) if unknown_negative.any() else 0.\n                home7=float(grad[home,:,7].abs().max().item()) if home.any() else 0.\n                if max_unknown!=0:raise RuntimeError(\'Gradiente negativo não zero em classe desconhecida.\')\n                self.last_audit={\'batch_step\':step,\'unknown_negative_max\':max_unknown,\'home7_max\':home7,\n                                 \'unknown_negative_elements\':int(unknown_negative.sum().item()),\n                                 \'unknown_positive_elements\':int(positives_unknown.sum().item())}\n                return grad\n            logits.register_hook(check_gradient)\n        return result\n\n\nclass SourceMixTrainer(ControlledTrainer):\n    arm=None;expected_schedule=None;attempt_dir=None;expected_initial_state=None\n\n    def __init__(self,*args,**kwargs):\n        self.batch_log=[];self.pending_batch=None\n        super().__init__(*args,**kwargs)\n        if self.arm not in p16.ARMS or self.expected_schedule is None:raise RuntimeError(\'Braço sem protocolo pareado.\')\n        if len(self.expected_schedule)!=self.control_spec[\'epochs\']*self.control_spec[\'batches\']:\n            raise RuntimeError(\'Cronograma incompleto.\')\n        self.add_callback(\'on_train_start\',verify_start)\n\n    def get_dataloader(self,dataset_path,batch_size=16,rank=0,mode=\'train\'):\n        if mode==\'train\':\n            if any(float(getattr(self.args,k,0))!=0 for k in p16.AUGMENTATION_ZERO):\n                raise RuntimeError(\'Todas as augmentações devem ser zero neste pareamento.\')\n            if self.args.workers!=0 or self.args.multi_scale or self.args.rect:\n                raise RuntimeError(\'Use workers=0, sem multi_scale e sem rect no treino pareado.\')\n        loader=super().get_dataloader(dataset_path,batch_size,rank,mode)\n        if mode==\'train\':\n            actual=[self.source_records[Path(f).resolve().as_posix()][\'id\'] for f in loader.dataset.im_files]\n            expected=[r[\'id\'] for r in p16.logical_order(list(self.source_records.values()))]\n            if actual!=expected:raise RuntimeError(\'Ordem de imagens do loader diverge do plano congelado.\')\n            # A biblioteca elimina linhas exatamente repetidas em seu cache; confira o conjunto de alvos.\n            for label in loader.dataset.labels:\n                record=self.source_records[Path(label[\'im_file\']).resolve().as_posix()]\n                raw=Path(record[\'label\']).read_text(encoding=\'utf-8-sig\')\n                rows=np.array([list(map(float,l.split())) for l in raw.splitlines() if l.strip()],dtype=np.float32).reshape(-1,5)\n                rows=np.unique(rows,axis=0)\n                actual_rows=np.concatenate([label[\'cls\'],label[\'bboxes\']],axis=1).astype(np.float32)\n                def canon(x):return sorted(tuple(float(v) for v in row) for row in x)\n                if canon(rows)!=canon(actual_rows):raise RuntimeError(\'O loader alterou os alvos projetados.\')\n        return loader\n\n    def preprocess_batch(self,batch):\n        result=super().preprocess_batch(batch)\n        step=self.batches_seen;planned=self.expected_schedule[step-1]\n        rs=[self.source_records[Path(f).resolve().as_posix()] for f in result[\'im_file\']]\n        ids=[r[\'id\'] for r in rs];signature=p16.digest([(r[\'id\'],r[\'sha256\']) for r in rs])\n        if ids!=planned[\'ids\'] or signature!=planned[\'input_signature\'] or self.epoch+1!=planned[\'epoch\']:\n            raise RuntimeError(\'Batch observado difere do cronograma pareado.\')\n        home=torch.tensor([r[\'source\']==\'home\' for r in rs],device=self.device,dtype=torch.bool)\n        masks=[r[\'known_mask\'] for r in rs]\n        cls=result[\'cls\'].flatten().long();bi=result[\'batch_idx\'].long()\n        if home.any() and not result[\'class_known\'][home].all():raise RuntimeError(\'Home perdeu cobertura original.\')\n        criterion=unwrap_model(self.model).criterion\n        if not isinstance(criterion,DetectionLossParcial):raise RuntimeError(\'Perda parcial não instalada.\')\n        if not isinstance(criterion.bce,AuditedBCE):criterion.bce=AuditedBCE(criterion.bce.original)\n        criterion.bce.home_mask=home;criterion.bce.audit_step=step;criterion.bce.last_audit=None\n        sample=result[\'img\'][:,:,::32,::32].detach().cpu().float().contiguous().numpy().tobytes()\n        gt_counts=[]\n        for i in range(len(rs)):\n            gt_counts.append(dict(Counter(int(v) for v in cls[bi==i].tolist())))\n        self.pending_batch={**planned,\'source_counts\':dict(Counter(r[\'source\'] for r in rs)),\n            \'known_masks\':masks,\'gt_per_image_by_class\':gt_counts,\n            \'tensor_amostra_sha256\':hashlib.sha256(sample).hexdigest()}\n        return result\n\n    def optimizer_step(self):\n        audit=unwrap_model(self.model).criterion.bce.last_audit\n        if audit is None or audit[\'batch_step\']!=self.batches_seen:raise RuntimeError(\'Backward não foi auditado para o batch atual.\')\n        super().optimizer_step()\n        row={**self.pending_batch,\'gradient_audit\':audit}\n        if row[\'step\']!=self.actual_updates:raise RuntimeError(\'Mais de uma atualização ou batch saltado.\')\n        self.batch_log.append(row)\n        path=Path(self.attempt_dir)/\'auditoria_batches.jsonl\';path.parent.mkdir(parents=True,exist_ok=True)\n        with path.open(\'a\',encoding=\'utf-8\') as f:f.write(p16.canonical(row)+\'\\n\')\n\n    def validate(self):\n        # As duas coortes são avaliadas pelo callback externo fixo, no checkpoint EMA exportado.\n        return self.metrics,0.\n\n    def final_eval(self):\n        return None\n\n\ndef verify_start(trainer):\n    actual=state_fingerprint(trainer.model)\n    if actual!=trainer.expected_initial_state:raise RuntimeError(\'Estado inicial diferente do candidato 10.\')\n    core.save_json(Path(trainer.attempt_dir)/\'inicializacao.json\',{\n        \'arm\':trainer.arm,\'state_sha256\':actual,\'same_initial_weights\':True})\n\n\ndef self_test(device=\'cpu\'):\n    """Backward real da BCE e da perda YOLO, usando somente tensores sintéticos."""\n    py=random.getstate();ns=np.random.get_state();ts=torch.get_rng_state()\n    cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None\n    try:\n        logits=torch.zeros((2,4,12),device=device,requires_grad=True);targets=torch.zeros_like(logits)\n        targets[1,0,7]=1;known=torch.ones((2,12),dtype=torch.bool,device=device);known[1,10]=False\n        bce=AuditedBCE();bce.known=known;bce.home_mask=torch.tensor([True,False],device=device);bce.audit_step=1\n        bce(logits,targets).sum().backward()\n        assert logits.grad[0,:,7].gt(0).all() and logits.grad[1,0,7]<0 and logits.grad[1,:,10].eq(0).all()\n        ordinary=nn.BCEWithLogitsLoss(reduction=\'none\')\n        full=BCEComCobertura();full.known=torch.ones_like(known)\n        torch.testing.assert_close(full(logits.detach(),targets),ordinary(logits.detach(),targets))\n        # Integração com assignment e termos de caixa/DFL da versão exata do YOLO.\n        model=DetectionModel(\'yolov8n.yaml\',nc=12,verbose=False).to(device).train()\n        from ultralytics.cfg import get_cfg\n        model.args=get_cfg();criterion=DetectionLossParcial(model)\n        criterion.bce=AuditedBCE();criterion.bce.home_mask=torch.tensor([True,False],device=device);criterion.bce.audit_step=2\n        batch={\'img\':torch.rand(2,3,64,64,device=device),\n               \'batch_idx\':torch.tensor([0,1],device=device),\n               \'cls\':torch.tensor([[2.],[7.]],device=device),\n               \'bboxes\':torch.tensor([[.5,.5,.5,.5],[.5,.5,.4,.4]],device=device),\n               \'class_known\':known}\n        loss,items=criterion(model(batch[\'img\']),batch);loss.sum().backward()\n        assert torch.isfinite(loss).all()\n        assert criterion.bce.last_audit[\'unknown_negative_max\']==0\n        result={\'bce_full_matches_standard\':True,\'home_full_coverage_preserved\':True,\'openimages7_positive_preserved\':True,\n                \'unknown_negative_gradient_zero\':True,\'real_yolo_loss_backward_finite\':True,\n                \'device\':str(device),\'synthetic_only\':True}\n        del model,criterion,batch,loss\n        return result\n    finally:\n        random.setstate(py);np.random.set_state(ns);torch.set_rng_state(ts)\n        if cuda is not None:torch.cuda.set_rng_state_all(cuda)\n', 'src/__init__.py': '', 'src/politica_alertas.py': '"""Parâmetros e funções da V2 original, sem reestimar com o TEST."""\nimport math\n\nNOMES_PT = {"bed": "cama", "sofa": "sofá", "chair": "cadeira", "table": "mesa",\n            "wardrobe": "guarda-roupa", "potted plant": "planta em vaso"}\nVERBOS_PT = {"bed": "detectada", "sofa": "detectado", "chair": "detectada", "table": "detectada",\n             "wardrobe": "detectado", "potted plant": "detectada"}\nORDEM_PRIORIDADE = {"sem_prioridade": 0, "baixo": 1, "medio": 2, "alto": 3}\n\n\ndef avaliar_bbox(classe, bbox, largura, altura, versao="v2"):\n    if versao not in {"v1", "v2"}:\n        raise ValueError("Versão deve ser v1 ou v2.")\n    x1, y1, x2, y2 = map(float, bbox)\n    if largura <= 0 or altura <= 0 or not all(math.isfinite(v) for v in [x1, y1, x2, y2]):\n        raise ValueError("Dimensões ou coordenadas inválidas.")\n    if x2 <= x1 or y2 <= y1:\n        raise ValueError("Caixa sem área positiva.")\n    posicao = classificar_posicao((x1 + x2) / 2, largura)\n    tamanho, area = classificar_tamanho((x2 - x1) * (y2 - y1), largura * altura)\n    vertical, base = classificar_posicao_vertical(y2, altura)\n    resultado = (classificar_risco_v2(classe, posicao, tamanho, vertical) if versao == "v2"\n                 else classificar_risco_v1(classe, posicao, tamanho))\n    nivel = resultado["nivel_risco"]\n    return {"prioridade": nivel if nivel in {"baixo", "medio", "alto"} else "sem_prioridade",\n            "status_original": nivel, "pontuacao": resultado["pontuacao"],\n            "posicao": posicao, "tamanho": tamanho, "regiao_vertical": vertical,\n            "area_relativa": area, "base_vertical_relativa": base,\n            "ajuste_vertical": resultado.get("ajuste_vertical", 0)}\n\n\ndef gerar_mensagem(objeto, nivel_minimo="medio"):\n    if nivel_minimo not in {"medio", "alto"}:\n        raise ValueError("Nível mínimo deve ser medio ou alto.")\n    prioridade = objeto["prioridade"]\n    if ORDEM_PRIORIDADE.get(prioridade, 0) < ORDEM_PRIORIDADE[nivel_minimo]:\n        return ""\n    posicao = {"centro": "central", "esquerda": "esquerda", "direita": "direita"}[objeto["posicao"]]\n    classe = objeto["classe"]\n    return (f"Atenção: {NOMES_PT[classe]} {VERBOS_PT[classe]} na região {posicao} da imagem. "\n            f"Prioridade {\'alta\' if prioridade == \'alto\' else \'média\'}.")\n\n\ndef gerar_mensagens_tabela(df, coluna="nivel_risco_v2"):\n    if df.empty:\n        return []\n    candidatos = df.loc[df[coluna].isin(["medio", "alto"])].copy()\n    candidatos["_ordem"] = candidatos[coluna].map(ORDEM_PRIORIDADE)\n    candidatos = candidatos.sort_values(["_ordem", "ocupacao_imagem"], ascending=[False, False])\n    return [gerar_mensagem({"classe": r["classe"], "posicao": r["posicao"], "prioridade": r[coluna]})\n            for _, r in candidatos.iterrows()]\n\nPESO_OBJETO = {\n    "bed": 3,\n    "sofa": 3,\n    "chair": 3,\n    "table": 3,\n    "wardrobe": 3,\n    "potted plant": 2,\n}\n\nCLASSES_CONTEXTUAIS = {\n    "lamp",\n    "door",\n}\n\nCLASSES_IGNORADAS = {\n    "tv",\n    "laptop",\n    "window",\n    "photo frame",\n}\n\n\nLIMITE_ESQUERDA = 1 / 3\nLIMITE_DIREITA = 2 / 3\n\nLIMITE_AREA_MEDIA = 0.05\nLIMITE_AREA_GRANDE = 0.20\n\nLIMITE_VERTICAL_SUPERIOR = 0.60\nLIMITE_VERTICAL_INFERIOR = 0.80\n\nPESO_POSICAO = {\n    "esquerda": 1,\n    "centro": 3,\n    "direita": 1,\n}\n\nPESO_TAMANHO = {\n    "pequeno": 1,\n    "medio": 2,\n    "grande": 3,\n}\n\nAJUSTE_VERTICAL = {\n    "superior": -2,\n    "intermediaria": -1,\n    "inferior": 0,\n}\n\ndef classificar_posicao(\n    centro_x: float,\n    largura_imagem: int,\n) -> str:\n    centro_normalizado = centro_x / largura_imagem\n\n    if centro_normalizado < LIMITE_ESQUERDA:\n        return "esquerda"\n\n    if centro_normalizado < LIMITE_DIREITA:\n        return "centro"\n\n    return "direita"\n\n\ndef classificar_tamanho(\n    area_box: float,\n    area_imagem: float,\n):\n    proporcao = area_box / area_imagem\n\n    if proporcao < LIMITE_AREA_MEDIA:\n        return "pequeno", proporcao\n\n    if proporcao < LIMITE_AREA_GRANDE:\n        return "medio", proporcao\n\n    return "grande", proporcao\n\n\ndef classificar_posicao_vertical(\n    y2: float,\n    altura_imagem: int,\n):\n    proporcao_vertical = y2 / altura_imagem\n\n    if proporcao_vertical < LIMITE_VERTICAL_SUPERIOR:\n        return "superior", proporcao_vertical\n\n    if proporcao_vertical < LIMITE_VERTICAL_INFERIOR:\n        return "intermediaria", proporcao_vertical\n\n    return "inferior", proporcao_vertical\n\ndef classificar_risco_v1(\n    classe: str,\n    posicao: str,\n    tamanho: str,\n):\n    if classe in CLASSES_IGNORADAS:\n        return {\n            "pontuacao": 0,\n            "nivel_risco": "ignorado",\n            "motivo": "classe sem relevância direta para locomoção",\n        }\n\n    if classe in CLASSES_CONTEXTUAIS:\n        return {\n            "pontuacao": None,\n            "nivel_risco": "contextual",\n            "motivo": "classe exige informação adicional de contexto",\n        }\n\n    if classe not in PESO_OBJETO:\n        return {\n            "pontuacao": None,\n            "nivel_risco": "nao_mapeado",\n            "motivo": "classe não configurada",\n        }\n\n    pontuacao = (\n        PESO_OBJETO[classe]\n        + PESO_POSICAO[posicao]\n        + PESO_TAMANHO[tamanho]\n    )\n\n    if pontuacao <= 4:\n        nivel = "baixo"\n    elif pontuacao <= 6:\n        nivel = "medio"\n    else:\n        nivel = "alto"\n\n    return {\n        "pontuacao": pontuacao,\n        "nivel_risco": nivel,\n        "motivo": (\n            f"{classe} + posição {posicao} + tamanho {tamanho}"\n        ),\n    }\n\ndef classificar_risco_v2(\n    classe: str,\n    posicao: str,\n    tamanho: str,\n    posicao_vertical: str,\n):\n    avaliacao_v1 = classificar_risco_v1(\n        classe=classe,\n        posicao=posicao,\n        tamanho=tamanho,\n    )\n\n    if avaliacao_v1["nivel_risco"] in {\n        "ignorado",\n        "contextual",\n        "nao_mapeado",\n    }:\n        return {\n            "pontuacao": avaliacao_v1["pontuacao"],\n            "nivel_risco": avaliacao_v1["nivel_risco"],\n            "ajuste_vertical": 0,\n            "motivo": avaliacao_v1["motivo"],\n        }\n\n    ajuste_vertical = AJUSTE_VERTICAL[posicao_vertical]\n\n    pontuacao = max(\n        1,\n        avaliacao_v1["pontuacao"] + ajuste_vertical,\n    )\n\n    if pontuacao <= 4:\n        nivel = "baixo"\n    elif pontuacao <= 6:\n        nivel = "medio"\n    else:\n        nivel = "alto"\n\n    return {\n        "pontuacao": pontuacao,\n        "nivel_risco": nivel,\n        "ajuste_vertical": ajuste_vertical,\n        "motivo": (\n            f"{classe} + posição {posicao} "\n            f"+ tamanho {tamanho} "\n            f"+ região vertical {posicao_vertical}"\n        ),\n    }\n', 'src/validacao_alertas.py': '"""Compara detectores na validação Home, com a V2 e os parâmetros do 05d."""\nfrom __future__ import annotations\n\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nfrom .politica_alertas import avaliar_bbox\n\nIMGSZ = 640\nCONF_ALERTA = 0.25\nIOU_NMS = 0.70\nIOU_CORRESPONDENCIA = 0.50\nMAX_DETECCOES = 300\n\n\ndef ler_objetos_yolo(caminho, largura, altura, nomes):\n    objetos = []\n    for n, linha in enumerate(Path(caminho).read_text(encoding="utf-8-sig").splitlines(), 1):\n        if not linha.strip():\n            continue\n        partes = linha.split()\n        if len(partes) != 5:\n            raise ValueError(f"Label inválido: {caminho}, linha {n}")\n        valores = [float(p) for p in partes]\n        if not all(math.isfinite(v) for v in valores):\n            raise ValueError(f"Label não finito: {caminho}, linha {n}")\n        id_float, xc, yc, w, h = valores\n        c = int(id_float)\n        bordas = [xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2]\n        if id_float != c or c not in nomes or w <= 0 or h <= 0 or min(bordas) < -0.0001 or max(bordas) > 1.0001:\n            raise ValueError(f"Classe ou caixa inválida: {caminho}, linha {n}")\n        x1, y1, x2, y2 = np.clip(bordas, 0, 1)\n        objetos.append({"classe_id": c, "classe": nomes[c],\n                        "bbox": [float(x1 * largura), float(y1 * altura),\n                                 float(x2 * largura), float(y2 * altura)]})\n    return objetos\n\n\ndef carregar_home_val(dataset, nomes):\n    dataset = Path(dataset)\n    raiz_imagens = dataset / "images/val"\n    registros = []\n    for imagem in sorted(raiz_imagens.rglob("*")):\n        if not imagem.is_file() or imagem.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:\n            continue\n        relativo = imagem.relative_to(raiz_imagens)\n        if relativo.parts[0] == "sunrgbd":\n            continue\n        label = dataset / "labels/val" / relativo.with_suffix(".txt")\n        with Image.open(imagem) as im:\n            largura, altura = im.size\n            im.verify()\n        registros.append({"imagem": imagem, "nome": relativo.as_posix(),\n                          "largura": largura, "altura": altura,\n                          "gt": ler_objetos_yolo(label, largura, altura, nomes)})\n    if not registros:\n        raise ValueError("Validação Home vazia.")\n    return registros\n\n\ndef iou_matriz(gt, pred):\n    if not gt or not pred:\n        return np.zeros((len(gt), len(pred)), dtype=float)\n    g = np.asarray([o["bbox"] for o in gt], dtype=float)\n    p = np.asarray([o["bbox"] for o in pred], dtype=float)\n    inicio = np.maximum(g[:, None, :2], p[None, :, :2])\n    fim = np.minimum(g[:, None, 2:], p[None, :, 2:])\n    inter = np.prod(np.maximum(0.0, fim - inicio), axis=2)\n    ag = np.prod(np.maximum(0.0, g[:, 2:] - g[:, :2]), axis=1)\n    ap = np.prod(np.maximum(0.0, p[:, 2:] - p[:, :2]), axis=1)\n    uniao = ag[:, None] + ap[None, :] - inter\n    return np.divide(inter, uniao, out=np.zeros_like(inter), where=uniao > 0)\n\n\ndef corresponder_objetos(gt, pred):\n    matriz = iou_matriz(gt, pred)\n    candidatos = []\n    for gi, g in enumerate(gt):\n        for pi, p in enumerate(pred):\n            if g["classe_id"] == p["classe_id"] and matriz[gi, pi] >= IOU_CORRESPONDENCIA:\n                candidatos.append((float(matriz[gi, pi]), gi, pi))\n    candidatos.sort(key=lambda item: (-item[0], item[1], item[2]))\n    usados_g, usados_p, pares = set(), set(), []\n    for iou, gi, pi in candidatos:\n        if gi not in usados_g and pi not in usados_p:\n            pares.append((gi, pi, iou))\n            usados_g.add(gi)\n            usados_p.add(pi)\n    return pares, [i for i in range(len(gt)) if i not in usados_g], [i for i in range(len(pred)) if i not in usados_p]\n\n\ndef contagem_eventos(gt, pred, largura, altura):\n    """Um FN/FP pode decorrer de detecção ausente/extra ou mudança de prioridade."""\n    g = [{**o, **avaliar_bbox(o["classe"], o["bbox"], largura, altura)} for o in gt]\n    p = [{**o, **avaliar_bbox(o["classe"], o["bbox"], largura, altura)} for o in pred]\n    pares, faltantes, extras = corresponder_objetos(g, p)\n    eventos = [(g[gi], p[pi]) for gi, pi, _ in pares]\n    eventos += [(g[gi], None) for gi in faltantes]\n    eventos += [(None, p[pi]) for pi in extras]\n    contagem = {"tp": 0, "fn": 0, "fp": 0, "alta_sem_alerta": 0}\n    for ref, prev in eventos:\n        a = bool(ref and ref["prioridade"] in {"medio", "alto"})\n        b = bool(prev and prev["prioridade"] in {"medio", "alto"})\n        contagem["tp"] += int(a and b)\n        contagem["fn"] += int(a and not b)\n        contagem["fp"] += int(not a and b)\n        contagem["alta_sem_alerta"] += int(bool(ref and ref["prioridade"] == "alto" and not b))\n    return contagem\n\n\ndef avaliar_alertas(modelo, registros, nomes, device=0):\n    modelo_nomes = {int(k): str(v) for k, v in modelo.names.items()}\n    if modelo_nomes != nomes:\n        raise ValueError("Mapa de classes do modelo difere do projeto.")\n    contagem = {"tp": 0, "fn": 0, "fp": 0, "alta_sem_alerta": 0}\n    for numero, r in enumerate(registros, 1):\n        resultado = modelo.predict(str(r["imagem"]), conf=CONF_ALERTA, imgsz=IMGSZ,\n                                   iou=IOU_NMS, max_det=MAX_DETECCOES,\n                                   device=device, verbose=False, save=False)[0]\n        caixas = resultado.boxes\n        pred = [] if caixas is None else [\n            {"classe_id": int(c), "classe": nomes[int(c)], "bbox": [float(v) for v in b]}\n            for c, b in zip(caixas.cls.cpu().numpy(), caixas.xyxy.cpu().numpy())]\n        eventos = contagem_eventos(r["gt"], pred, r["largura"], r["altura"])\n        for k, v in eventos.items():\n            contagem[k] += v\n        if numero % 100 == 0:\n            print(f"Validação dos alertas: {numero}/{len(registros)}")\n    tp, fn, fp = contagem["tp"], contagem["fn"], contagem["fp"]\n    dividir = lambda n, d: n / d if d else None\n    return {"imagens": len(registros), **contagem,\n            "precision_alertas": dividir(tp, tp + fp),\n            "recall_alertas": dividir(tp, tp + fn),\n            "f1_alertas": dividir(2 * tp, 2 * tp + fp + fn),\n            "referencia": "V2 fixa nas anotações; não valida risco físico independentemente",\n            "conf": CONF_ALERTA, "iou_nms": IOU_NMS,\n            "iou_correspondencia": IOU_CORRESPONDENCIA}\n'}
CODIGO_SHA256={'conf14_data.py': 'f73232836278c08edf0f955ae8fe50a33cb1d4eabe2b2cc35a5f410f095bae66', 'conf14_evaluation.py': '3e64c612ffcd7cb75fdf05707affc45cafa91acd1882ab311425c2752ed01656', 'control16_data.py': 'b82dd42c119da0d86c26ff621e8383836285a699d4ad19dd20082614c72c60b4', 'control16_protocol.py': '43665e10eca68b52149fb32bcdcf3fdd8ebc9e755dd59edc097cea6c9f313c22', 'diag11_core.py': '29e7c52fced44ab4898edfbc46477176587f290d2fbdacad4de5f595d4187670', 'diag11_data.py': 'c9f137d4dcc8c8f5fb156aaf25cddf6707d24e74567c05237332c2144e6e2f98', 'diag11_reports.py': '0068af8fd70b616cb642e9fb8a75d503cdc003f18fe79d64fd487bd7822d85e2', 'exp10_partial.py': '86a36f93aa643853f1268dcc133bed7e7344dea269b9d1d58cf54095538fffae', 'exp12_data.py': '0a9de7e99d302a2690f18f51924f4dc80b32861c9d2149ccfe0572ec75499f30', 'exp12_protocol.py': '257d09b0b2aafe94da28bdcb59859778ffb31e25601db75d1058092b943b5759', 'exp12_training.py': '6c51c9965336b2f00bd9b282e2159c0b63f07c5c16859da78da2818bd4202fdc', 'mix18_data.py': '8d07c2134ac391ea545eea0a8796d2b055d478f6355da38411990a8f548b805c', 'mix18_eval.py': 'e6edbeb4a38c10f28d6fdd44eff2534e62bae127e94068e8ccfb7236a67a6356', 'mix18_history.py': 'e2358a8537c0a659d93aff2c117ab95b1be567d6e9650923d4b7d396f93fbafe', 'mix18_io.py': '25300e50e08260a7a0c0601fb65941605d042c97ec23cbb46811e4e0900f14eb', 'mix18_protocol.py': 'f4dfcc83547b439d71d145aff76d658d8e651c91788dcda4a6399c2799694e2c', 'mix18_reports.py': '4eae12dc0e19e97cc6da694f1a96d2ff3b322a11cee303336f2fb801f35e746e', 'mix18_run.py': 'dfbee60f13987f9c4e34be18134306c067b6e9b73c4aa9c3777170913047152b', 'mix18_state.py': '7488edc9fe709f0910c6ae53587f3ab8a414c9c5e592503e612c1cf92c90d4e1', 'mix18_training.py': 'e18be4ed9e39071ede96b7435fc9efcc84e0ed8fb9fb05d2a7b79290334adf2c', 'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/politica_alertas.py': '1aa6b13cb68d8f30a2f388c3577a61a12dcac4c604581877091d3fbdc6042845', 'src/validacao_alertas.py': '1da5dff21a6c6f6ec53d041247ce18d012008c4b26023b52c3fd2745f4c4105d'}
PASTA_CODIGO=PASTA_LOCAL/'codigo';PASTA_CODIGO.mkdir(exist_ok=True)
for nome,fonte in MODULOS_EMBUTIDOS.items():
    if hashlib.sha256(fonte.encode()).hexdigest()!=CODIGO_SHA256[nome]:raise RuntimeError('Módulo embutido alterado: '+nome)
    if nome in EXPECTED['codigo_12'] and CODIGO_SHA256[nome]!=EXPECTED['codigo_12'][nome]:raise RuntimeError('Módulo histórico divergente: '+nome)
    compile(fonte,nome,'exec')
    for raiz in [PASTA_CODIGO,PASTA_RESULTADOS/'codigo']:
        destino=raiz/nome;destino.parent.mkdir(parents=True,exist_ok=True);destino.write_text(fonte,encoding='utf-8')
sys.path.insert(0,str(PASTA_CODIGO))
import diag11_core as core
import diag11_data as dados11
import exp12_data as dados12
import exp10_partial as parcial
import conf14_data as dados14
import conf14_evaluation as avaliador14
import control16_data as dados16
import mix18_protocol as protocolo18
import mix18_data as dados18
import mix18_training as treino18
import mix18_eval as avaliacao18
import mix18_state as estado18
import mix18_run as execucao18
import mix18_reports as relatorios18
import mix18_history as historico18
from mix18_io import Drive
from exp12_training import state_fingerprint
from src import politica_alertas as politica
from src import validacao_alertas as avaliacao_base
if core.sha256_file(politica.__file__)!=EXPECTED['politica'] or core.sha256_file(avaliacao_base.__file__)!=EXPECTED['associador']:
    raise RuntimeError('Política ou associação divergente da versão congelada.')
def enviar_resultados(etapa,obrigatorio=True):
    print('Backup:',etapa,flush=True)
    return DRIVE.backup(PASTA_RESULTADOS,'resultados/'+NOME_EXPERIMENTO,required=obrigatorio)
def progresso(etapa):
    global ULTIMA_ETAPA
    agora=time.monotonic()
    TEMPOS.append({'etapa':etapa,'segundos':agora-ULTIMA_ETAPA,'acumulado_segundos':agora-INICIO_EXECUCAO})
    ULTIMA_ETAPA=agora
    core.save_json(PASTA_RESULTADOS/'tempos_execucao.json',{'exclui_instalacao_pip':True,'etapas':TEMPOS})
    print(f'{etapa} | tempo acumulado: {(agora-INICIO_EXECUCAO)/60:.1f} min',flush=True)
def registrar_falha(exc,etapa):
    core.save_json(PASTA_RESULTADOS/'execucao_interrompida.json',{'concluido':False,'etapa':etapa,
        'tipo':type(exc).__name__,'mensagem':str(exc)[:1400],'modelo_substituido':False,
        'rotulos_originais_preservados':True,'preenchimento_manual_obrigatorio':False})
    pacote=estado18.package(PASTA_RESULTADOS)
    if DRIVE is not None:
        try:enviar_resultados('pacote parcial',obrigatorio=False)
        except Exception:pass
    print('Pacote parcial com o motivo da interrupção:',pacote,flush=True)
    display(FileLink(str(pacote)))
try:
    protocolo18.validate_specs()
    parcial.check_api()
    AUTOTESTE=treino18.self_test('cuda:0' if DEVICE==0 else 'cpu')
    core.save_json(PASTA_RESULTADOS/'autoteste_perda.json',AUTOTESTE)
    print('Autoteste da perda:',AUTOTESTE)
    progresso('API e perda conferidas')
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
except Exception as exc:
    registrar_falha(exc,'API e perda');raise


## 3. Autenticar os resultados anteriores, o replay e as entradas

Confere os pacotes 11, 12, 15, 16 e 17, o candidato 10, o baseline, as divisões
e os manifestos. O replay é exatamente o mesmo do 12/16. O tempo anterior é
lido dos registros reais; pesos ajustados do 16 não serão o ponto de partida.


In [ ]:
try:
    DRIVE=Drive.from_kaggle()
    if shutil.disk_usage(PASTA_LOCAL).free<10*1024**3:raise RuntimeError('Reserve pelo menos 10 GiB livres para as entradas.')
except Exception as exc:
    registrar_falha(exc,'conexão e espaço');raise
def baixar_arquivo_drive(relativo,destino):return DRIVE.fetch(relativo,destino)

ENTRADAS=PASTA_LOCAL/'entradas'
OI_RELATIVO='datasets/openimages_v7_tcc_100k_v1'
P10='resultados/yolov8n_home_openimages_80_20_1ep_kaggle_20261015_030228_068010'
P11='resultados/diagnostico11_20261016_024603_683411'
P12='resultados/ajuste12_20261016_033819_596490'
P15='resultados/auditoria15_20261016_150850_292698'
LOCALIZACAO_11={'pasta_drive':'datasets/validacao_independente_11_v1/'+EXPECTED['dataset_novo'],
               'arquivo':'validacao_independente_11_695b445e15c9.zip'}
try:
    PACOTE_12=baixar_arquivo_drive(P12+'/pacote_experimento_12.zip',ENTRADAS/'pacote_12.zip')
    BUNDLE=dados14.load_source12(PACOTE_12,EXPECTED,PASTA_RESULTADOS)
    PACOTE_15=baixar_arquivo_drive(P15+'/pacote_auditoria_15.zip',ENTRADAS/'pacote_15.zip')
    AUDIT15=dados16.read_audit15(PACOTE_15,EXPECTED['pacote_15'])
    REPLAY12=dados16.read_replay12(PACOTE_12,EXPECTED['pacote_12'],AUDIT15)
    PACOTE_11=baixar_arquivo_drive(P11+'/pacote_diagnostico_11.zip',ENTRADAS/'pacote_11.zip')
    RESUMO11,CONFIG11,LOCAL11,MANIFESTO_NOVO=dados12.read_package11(PACOTE_11,EXPECTED['pacote_11'],
        {'antigo':EXPECTED['baseline'],'candidato':EXPECTED['candidato']},EXPECTED['dataset_novo'])
    if any(LOCAL11[k]!=v for k,v in LOCALIZACAO_11.items()) or CONFIG11['politica_sha256']!=EXPECTED['politica']:
        raise RuntimeError('Origem da validação/política incompatível.')
    PREP=baixar_arquivo_drive(OI_RELATIVO+'/configuracao_preparacao.json',ENTRADAS/'configuracao_preparacao.json')
    PACOTES=baixar_arquivo_drive(OI_RELATIVO+'/manifesto_pacotes.json',ENTRADAS/'manifesto_pacotes.json')
    DIVISAO_CSV=baixar_arquivo_drive(P10+'/divisao_openimages.csv',ENTRADAS/'divisao_10.csv')
    DIVISAO_JSON=baixar_arquivo_drive(P10+'/divisao_openimages.json',ENTRADAS/'divisao_10.json')
    cfg=json.loads(PREP.read_text())
    if hashlib.sha256(core.canonical(cfg).encode()).hexdigest()!=EXPECTED['preparacao'] or cfg['nomes']!=core.NAMES:
        raise RuntimeError('Mapeamento original alterado.')
    if core.sha256_file(PACOTES)!=EXPECTED['manifesto_pacotes']:raise RuntimeError('Manifesto de lotes alterado.')
    HISTORICO=dados11.load_history(DIVISAO_CSV,DIVISAO_JSON,EXPECTED['divisao_10'])
    ZIP_HOME=baixar_arquivo_drive('datasets/homeobjects_sunrgbd_validado.zip',ENTRADAS/'home.zip')
    ZIP_NOVO=baixar_arquivo_drive(LOCAL11['pasta_drive']+'/'+LOCAL11['arquivo'],ENTRADAS/'validacao_11.zip')
    PESOS={'baseline':baixar_arquivo_drive('modelos/yolov8n_baseline/best.pt',ENTRADAS/'baseline.pt'),
           'candidato_10':baixar_arquivo_drive(P10+'/best.pt',ENTRADAS/'candidato_10.pt')}
    for name,key in [('baseline','baseline'),('candidato_10','candidato')]:
        if core.sha256_file(PESOS[name])!=EXPECTED[key]:raise RuntimeError('Peso não corresponde à referência: '+name)

    PACOTE_16=DRIVE.fetch(EXPECTED['pasta_16']+'/pacote_experimento_16.zip',ENTRADAS/'pacote_16.zip',EXPECTED['pacote_16'])
    PACOTE_17=DRIVE.fetch(EXPECTED['pasta_17']+'/pacote_calibracao_17.zip',ENTRADAS/'pacote_17.zip',EXPECTED['pacote_17'])
    HISTORICO_RESULTADOS=historico18.load(PACOTE_16,PACOTE_17,EXPECTED,PASTA_RESULTADOS)
    CONGELADOS=[(p,core.sha256_file(p)) for p in [PACOTE_11,PACOTE_12,PACOTE_15,PACOTE_16,PACOTE_17,PREP,PACOTES,
                    DIVISAO_CSV,DIVISAO_JSON,ZIP_HOME,ZIP_NOVO,*PESOS.values()]]
    progresso('Entradas autenticadas')
    print('Entradas autenticadas. Replay histórico:',len(REPLAY12),'imagens.')
except Exception as exc:
    registrar_falha(exc,'entradas e pesos');raise


## 4. Preparar os mesmos dados para os dois braços

Usa 1.820 Home TRAIN após a quarentena dos mesmos quatro arquivos e 2.000 OI
históricos. Home mantém todos os rótulos, incluindo armários; a cobertura parcial
OI é preservada. Não há relabeling nem preenchimento manual.

Os cinco lotes OI são conferidos e removidos após a extração seletiva do replay.
As validações têm 456 Home + 1.000 externas. Não extrai TEST nem a reserva de 300.


In [ ]:
try:
    HOME,HASHES_HOME,NEAR_HOME,AUDITORIA_HOME=dados11.extract_home(ZIP_HOME,PASTA_LOCAL/'home',
        EXPECTED['home_zip'],EXPECTED['test_home'],EXPECTED['politica'])
    dados12.materialise_home_labels(ZIP_HOME,HOME)
    sys.path.insert(0,str(PASTA_LOCAL/'home/codigo'))
    import src.politica_alertas as politica
    import src.validacao_alertas as avaliacao_base
    if (core.sha256_file(politica.__file__)!=EXPECTED['politica'] or float(avaliacao_base.IOU_CORRESPONDENCIA)!=.5
        or float(avaliacao_base.IOU_NMS)!=.7 or int(avaliacao_base.IMGSZ)!=640):
        raise RuntimeError('Política/associador diferentes do protocolo.')
    VAL_NOVA,IDS_RESERVA,congelados=dados12.load_frozen_validation(ZIP_NOVO,PASTA_LOCAL/'validacao',
        EXPECTED['zip_novo'],EXPECTED['dataset_novo'],MANIFESTO_NOVO)
    CONGELADOS+=congelados
    if len(IDS_RESERVA)!=300 or any('reserva_futura' in p.parts for p in (PASTA_LOCAL/'validacao').rglob('*')):
        raise RuntimeError('Reserva extraída ou identidade incompatível.')
    HOME_VAL=[r for r in HOME if r['role']=='home_val']
    HOME_TREINO=dados16.training_home(HOME,AUDIT15)
    COORTES={'home_val_historica':HOME_VAL,'openimages_val_nova':VAL_NOVA}
    dados14.validate_cohorts(COORTES,BUNDLE,IDS_RESERVA)
    for r in HOME:CONGELADOS.extend([(r['image'],r['sha256']),(r['label'],r['label_sha256'])])
    if {r['sha256'] for r in MANIFESTO_NOVO['arquivos']}&HASHES_HOME:raise RuntimeError('Coincidência de bytes entre Home e a base externa.')
    indice_protegido=dados11.NearIndex()
    for r in HOME_VAL:indice_protegido.add(r['phash'],r['id'])
    for r in MANIFESTO_NOVO['arquivos']:indice_protegido.add(int(r['phash'],16),r['id'])
    proibidos_hash=HASHES_HOME|{r['sha256'] for r in MANIFESTO_NOVO['arquivos']}
    proibidos_ids={r['id'] for r in HISTORICO['rows'] if r['split']=='test'}|{r['id'] for r in MANIFESTO_NOVO['arquivos']}
    pacotes=json.loads(PACOTES.read_text())
    if pacotes['config_sha256']!=EXPECTED['preparacao']:raise RuntimeError('Configuração dos lotes mudou.')
    nomes_lotes={r['lote'] for r in REPLAY12}
    LOTES=sorted([r for r in pacotes['lotes'] if r['arquivo'] in nomes_lotes],key=lambda r:r['arquivo'])
    if nomes_lotes!={f'lote_{i:06}.zip' for i in range(1,6)} or len(LOTES)!=5:raise RuntimeError('Esperados os cinco lotes do replay histórico.')
    def buscar_lote(item,destino):
        return baixar_arquivo_drive(OI_RELATIVO+'/lotes/'+item['arquivo'],destino)
    REPLAY,RECIBOS=dados16.extract_replay_exact(LOTES,REPLAY12,HISTORICO['rows'],PASTA_LOCAL/'replay',
        buscar_lote,EXPECTED['preparacao'],proibidos_ids,proibidos_hash,indice_protegido,AUDIT15)
    for r in REPLAY:CONGELADOS.extend([(r['image'],r['sha256']),(r['label'],r['label_sha256'])])
    REGISTROS_TREINO=HOME_TREINO+REPLAY
    if Counter(r['source'] for r in REGISTROS_TREINO)!={'home':1820,'openimages':2000}:raise RuntimeError('Quantidade de treino divergente.')

    BRACOS,congelados=dados18.make_common(REGISTROS_TREINO,HOME_VAL,PASTA_LOCAL/'treino_comum',PASTA_RESULTADOS)
    CONGELADOS+=congelados
    SPECS={a:dict(protocolo18.SPECS[a]) for a in protocolo18.ARMS}
    CRONOGRAMAS={a:protocolo18.schedule(REGISTROS_TREINO,SPECS[a]) for a in protocolo18.ARMS}
    for a in protocolo18.ARMS:protocolo18.save_gz(PASTA_RESULTADOS/('cronograma_'+a+'.json.gz'),CRONOGRAMAS[a])
    FLUXOS=protocolo18.audit_source_streams(REGISTROS_TREINO,SPECS)
    core.save_json(PASTA_RESULTADOS/'fluxos_por_fonte_planejados.json',FLUXOS)
    core.save_json(PASTA_RESULTADOS/'replay_historico_reutilizado.json',{'ids':REPLAY12,'lotes':RECIBOS})
    core.save_json(PASTA_RESULTADOS/'quarentena_detalhes.json',AUDIT15['quarantine'])
    for p in (PASTA_LOCAL/'home/codigo').rglob('*.py'):
        dest=PASTA_RESULTADOS/'codigo_home'/p.relative_to(PASTA_LOCAL/'home/codigo');dest.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dest)
    progresso('Treino e validações preparados')
    print('Treino comum: 1.820 Home + 2.000 replay; nenhum alvo ou classe removido.')
    print('Validações preservadas: 456 + 1.000. Reserva fechada: 300.')
except Exception as exc:
    registrar_falha(exc,'dados e composição por fonte');raise


## 5. Congelar o protocolo e recuperar apenas braços completos do 18

AdamW, LR constante 3e-6, FP32, batch 16, acumulação 1, semente 42, sem warmup
ou augmentações, 342 atualizações por braço. Todo o restante permanece comum.

Uma execução interrompida pode ser refeita com **Run All**, sem editar células.
O contexto inclui ambiente, dados, cronogramas e código. Só reutiliza um braço
18 completo com todos os hashes conferidos; braço parcial reinicia do candidato
10. Um ambiente diferente gera outro experimento e refaz ambos os braços.


In [ ]:
try:
    protocolo18.validate_specs(SPECS)
    spec=protocolo18.SPEC
    ARGUMENTOS_TREINO=dict(epochs=spec['epochs'],imgsz=640,batch=spec['batch'],nbs=spec['batch'],
        workers=0,device=DEVICE,amp=False,optimizer='AdamW',lr0=spec['lr0'],lrf=1.,cos_lr=False,
        momentum=.937,weight_decay=.0005,warmup_epochs=0.,close_mosaic=0,
        pretrained=True,rect=False,compile=False,seed=SEED,deterministic=True,patience=0,
        save=True,save_period=1,plots=False,cache=False,val=False,conf=.001,iou=.7,max_det=300,
        resume=False,exist_ok=False,verbose=False,**{k:0. for k in protocolo18.AUGMENTATION_ZERO})
    with zipfile.ZipFile(PACOTE_12) as z:REGRA_RAW=z.read('regra_selecao_congelada.json')
    if hashlib.sha256(REGRA_RAW).hexdigest()!=EXPECTED['regra_12_sha256']:raise RuntimeError('Regra 12 alterada.')
    REGRA=json.loads(REGRA_RAW)
    MANIFESTO_TREINO=json.loads((PASTA_RESULTADOS/'manifesto_treino_comum_18.json').read_text())
    PLANO={'versao':protocolo18.VERSION,'specs':SPECS,'seed':SEED,'arms':list(protocolo18.ARMS),
        'initial_checkpoint_sha256':EXPECTED['candidato'],'dados_sha256':protocolo18.digest(MANIFESTO_TREINO),
        'replay12_sha256':protocolo18.digest(REPLAY12),
        'cronogramas_sha256':{a:protocolo18.digest(CRONOGRAMAS[a]) for a in protocolo18.ARMS},
        'regra_12_sha256':EXPECTED['regra_12_sha256'],'primary_endpoint':'E8_8_e03_vs_C12_4_e03',
        'intermediarios':'somente diagnóstico','argumentos_treino':ARGUMENTOS_TREINO,
        'conf_alertas':.25,'augmentation':'zero em ambos','workers':0,
        'variavel_experimental':'oi_per_batch:4->8; home_per_batch:12->8',
        'supervisao':'Home original completa; Open Images parcial original; nenhuma máscara experimental',
        'quarentena_comum':sorted(dados16.QUARANTINE_IDS),'validacao_pre_exposta':True,
        'validacao_rotulos_alterados':False,'reserva_futura_inferida':False,
        'retomada':'somente braço 18 completo verificado; parcial reinicia de C10','codigo_sha256':CODIGO_SHA256}
    CONTEXTO={'plano':PLANO,'ambiente':AMBIENTE,'auditoria15_sha256':EXPECTED['pacote_15'],
        'resultado16_sha256':EXPECTED['pacote_16'],'resultado17_sha256':EXPECTED['pacote_17'],
        'validacao_sha256':EXPECTED['dataset_novo'],'home_sha256':EXPECTED['home_zip']}
    CONTEXT_KEY=protocolo18.digest(CONTEXTO)
    NOME_EXPERIMENTO='fontes18_'+CONTEXT_KEY[:24]
    PASTA_RESULTADOS=Path('/kaggle/working/resultados_tcc')/NOME_EXPERIMENTO
    PASTA_RESULTADOS.mkdir(parents=True,exist_ok=True)
    shutil.copytree(PASTA_BOOT,PASTA_RESULTADOS,dirs_exist_ok=True)
    core.save_json(PASTA_RESULTADOS/'protocolo_congelado_18.json',PLANO)
    core.save_json(PASTA_RESULTADOS/'contexto_execucao.json',{'context_key':CONTEXT_KEY,**CONTEXTO})
    REGRA_PATH=PASTA_RESULTADOS/'regra_selecao_congelada_12.json';REGRA_PATH.write_bytes(REGRA_RAW)
    CONGELADOS.append((REGRA_PATH,EXPECTED['regra_12_sha256']))
    CONGELADOS.extend([(PASTA_CODIGO/n,CODIGO_SHA256[n]) for n in MODULOS_EMBUTIDOS])
    CONGELADOS.extend([(Path(politica.__file__),EXPECTED['politica']),(Path(avaliacao_base.__file__),EXPECTED['associador'])])
    DRIVE.restore('resultados/'+NOME_EXPERIMENTO+'/bracos',PASTA_RESULTADOS/'bracos')
    DRIVE.restore('resultados/'+NOME_EXPERIMENTO+'/referencias',PASTA_RESULTADOS/'referencias')
    CONFIG={'versao':protocolo18.VERSION,'experimento':NOME_EXPERIMENTO,'context_key':CONTEXT_KEY,
        'ambiente':AMBIENTE,'plano':PLANO,'specs':SPECS,'inferencia_reserva':False,
        'origem_baseline':'nao_determinado','projecao_treino_home7':False,'rotulos_originais_preservados':True,
        'historico':HISTORICO_RESULTADOS,
        'entradas_conferidas':[{'arquivo':Path(p).name,'sha256':sha} for p,sha in CONGELADOS[:13]]}
    core.save_json(PASTA_RESULTADOS/'configuracao_experimento_18.json',CONFIG)
    progresso('Protocolo congelado e recuperação conferida')
    enviar_resultados('protocolo antes do treino')
    print('Experimento:',NOME_EXPERIMENTO)
except Exception as exc:
    registrar_falha(exc,'protocolo e recuperação');raise


## 6. Reproduzir as referências antes de treinar

Reavalia baseline e candidato 10 em 0,25. Contadores por imagem e estados GT
devem reproduzir o pacote 12, como no controle 16; divergência bloqueia o treino.
Confere também o estado inicial do modelo. A política V2 e a associação IoU 0,5
são as mesmas, com inferência 640 / confiança 0,001 / NMS 0,7 / max_det 300.


In [ ]:
REFERENCIAS={};REPRODUCAO=[];REPRO_IMAGENS=[];REPRO_OBJETOS=[]
try:
    for name in ['baseline','candidato_10']:
        result=avaliacao18.evaluate(PESOS[name],name,COORTES,PASTA_RESULTADOS/'referencias'/name,
            politica,avaliacao_base.corresponder_objetos,AMBIENTE,DEVICE)
        REFERENCIAS[name]=result
        for coorte,res in result.items():
            resumo,imagens,objetos=avaliador14.check_reproduction(res,BUNDLE['evaluations'][name][coorte],coorte)
            REPRODUCAO.append({'modelo':name,**resumo})
            for row in imagens:row['modelo']=name;row['notebook18']=row.pop('notebook14')
            for row in objetos:row['modelo']=name
            REPRO_IMAGENS+=imagens;REPRO_OBJETOS+=objetos
        core.save_json(PASTA_RESULTADOS/'reproducao_referencias_resumo.json',REPRODUCAO)
        enviar_resultados('referência conferida: '+name)
    core.save_csv(PASTA_RESULTADOS/'reproducao_referencias_por_imagem.csv',REPRO_IMAGENS)
    core.save_csv(PASTA_RESULTADOS/'reproducao_referencias_por_objeto.csv',REPRO_OBJETOS)
    if not all(r['reproduziu'] for r in REPRODUCAO):raise RuntimeError('As referências não reproduziram; treino bloqueado para manter a comparabilidade.')
    aux=YOLO(str(PESOS['candidato_10']));INITIAL_STATE=state_fingerprint(aux.model);del aux
    INICIAL=avaliacao18.entry('candidato_10_inicial','inicial',0,0,'',EXPECTED['candidato'],REFERENCIAS['candidato_10'])
    progresso('Referências reproduzidas')
    print('Referências reproduzidas. Os seis critérios originais permanecem fixos.')
except Exception as exc:
    registrar_falha(exc,'reprodução das referências');raise


## 7. Treinar 12/4 e 8/8 com o mesmo orçamento

Cada época curta exporta o checkpoint EMA em formato padrão, salva no Drive e
avalia as duas validações. A avaliação preserva o estado aleatório do treino.
Os logs conferem os IDs, os hashes, as máscaras e o gradiente negativo das classes
desconhecidas. O pacote inclui as exposições realmente vistas, por fonte e ID.

Correção da auditoria: os alvos são conferidos após o mesmo pipeline do loader, incluindo seus filtros padrão de caixas. Os rótulos, máscaras e critérios permanecem os mesmos. Braços completos do mesmo contexto podem ser reaproveitados.


In [ ]:
import hashlib,types,sys
from pathlib import Path
_FONTE_CORRECAO_18='"""Corrige a referência de contagem após o loader; preserva os treinos do 18."""\nfrom __future__ import annotations\nfrom collections import Counter\nfrom copy import deepcopy\nimport hashlib,inspect,json\nfrom pathlib import Path\nimport numpy as np\nimport yaml\nimport ultralytics\nfrom ultralytics.cfg import get_cfg\nfrom ultralytics.data.build import build_yolo_dataset\nimport ultralytics.data.augment as augment_source\nimport diag11_core as core\nimport exp12_data as frozen_data\nfrom exp12_training import isolated_evaluation\nimport mix18_protocol as protocol\n\nVERSION=\'18-auditoria-pos-loader-1.0.1\'\n\ndef counts(values):\n    return dict(Counter(str(int(v)) for v in values))\n\ndef raw_rows(record):\n    if core.sha256_file(record[\'label\'])!=record[\'label_sha256\']:\n        raise ValueError(\'Rótulo original alterado: \'+record[\'id\'])\n    rows=np.array([list(map(float,l.split())) for l in Path(record[\'label\']).read_text(encoding=\'utf-8-sig\').splitlines() if l.strip()],dtype=np.float32).reshape(-1,5)\n    return np.unique(rows,axis=0)\n\ndef assert_pipeline(dataset):\n    # Albumentations fotométricas não afetam a contagem; uma transformação espacial\n    # externa impediria usar uma referência determinística por imagem.\n    seen=set()\n    def visit(transform):\n        if id(transform) in seen:return\n        seen.add(id(transform))\n        if type(transform).__name__==\'Albumentations\' and getattr(transform,\'contains_spatial\',False):\n            raise ValueError(\'Albumentations espacial inesperada na referência de alvos.\')\n        for child in getattr(transform,\'transforms\',[]) or []:visit(child)\n        pre=getattr(transform,\'pre_transform\',None)\n        if pre is not None:visit(pre)\n    visit(dataset.transforms)\n\ndef loader_reference(records,data,arguments,output):\n    """Repassa as imagens pelo mesmo dataset de TRAIN, sem modelo ou otimizador."""\n    output=Path(output)\n    if ultralytics.__version__!=\'8.4.150\':raise RuntimeError(\'A correção requer ultralytics==8.4.150, como o treino.\')\n    if (arguments.get(\'rect\') or arguments.get(\'workers\')!=0 or arguments.get(\'cache\')\n        or any(float(arguments.get(k,0))!=0 for k in protocol.AUGMENTATION_ZERO)):\n        raise ValueError(\'Parâmetros incompatíveis com a referência determinística do 18.\')\n    wanted={Path(r[\'image\']).resolve().as_posix():r for r in records}\n    if len(wanted)!=len(records) or len({r[\'id\'] for r in records})!=len(records):raise ValueError(\'Imagens/IDs repetidos na referência.\')\n    if set(wanted)!=set(data[\'records\']):raise ValueError(\'Dataset comum diferente dos registros autenticados.\')\n    cfg=get_cfg(overrides=deepcopy(arguments))\n    with Path(data[\'yaml\']).open() as f:dataset_data=yaml.safe_load(f)\n    dataset_data[\'nc\']=len(core.NAMES)\n    names=dataset_data[\'names\']\n    names=dict(enumerate(names)) if isinstance(names,list) else {int(k):v for k,v in names.items()}\n    if names!=dict(enumerate(core.NAMES)):raise ValueError(\'Mapeamento de classes mudou no dataset comum.\')\n    rows=[];differences=[]\n    with isolated_evaluation():\n        dataset=build_yolo_dataset(cfg,dataset_data[\'train\'],cfg.batch,dataset_data,mode=\'train\',rect=False,stride=32)\n        assert_pipeline(dataset)\n        paths=[Path(f).resolve().as_posix() for f in dataset.im_files]\n        if len(paths)!=len(wanted) or set(paths)!=set(wanted):raise ValueError(\'O loader ignorou/adicionou uma imagem.\')\n        for index,key in enumerate(paths):\n            record=wanted[key];original=raw_rows(record);label=dataset.labels[index]\n            stored=np.concatenate([label[\'cls\'],label[\'bboxes\']],axis=1).astype(np.float32)\n            canon=lambda a:sorted(tuple(float(x) for x in r) for r in a)\n            if canon(original)!=canon(stored):raise ValueError(\'O cache bruto do loader diverge do rótulo autenticado: \'+record[\'id\'])\n            sample=dataset[index]\n            if Path(sample[\'im_file\']).resolve().as_posix()!=key:raise ValueError(\'O loader trocou a identidade da imagem.\')\n            raw=counts(original[:,0]);processed=counts(sample[\'cls\'].flatten().tolist())\n            if any(v>raw.get(k,0) for k,v in processed.items()):raise ValueError(\'O processamento adicionou alvos inesperados: \'+record[\'id\'])\n            row={\'id\':record[\'id\'],\'image_sha256\':record[\'sha256\'],\'label_sha256\':record[\'label_sha256\'],\n                 \'known_mask\':record[\'known_mask\'],\'raw_unique_counts\':raw,\'processed_counts\':processed}\n            rows.append(row)\n            if raw!=processed:\n                # O caso que disparou a auditoria deve ser reproduzível antes de aceitar.\n                for _ in range(2):\n                    again=dataset[index]\n                    if counts(again[\'cls\'].flatten().tolist())!=processed:\n                        raise ValueError(\'Contagem de alvos não determinística: \'+record[\'id\'])\n                differences.append({**row,\'discarded_by_class\':{k:v-processed.get(k,0) for k,v in raw.items() if v!=processed.get(k,0)}})\n            if (index+1)%250==0 or index+1==len(paths):\n                print(f\'Referência após loader: {index+1}/{len(paths)} imagens\',flush=True)\n        del dataset\n    summary={\'versao\':VERSION,\'imagens\':len(rows),\'imagens_com_alvos_filtrados\':len(differences),\n        \'alvos_filtrados_unicos\':sum(sum(r[\'discarded_by_class\'].values()) for r in differences),\n        \'ultralytics\':ultralytics.__version__,\'augment_source_sha256\':core.sha256_file(inspect.getfile(augment_source)),\n        \'reference_sha256\':protocol.digest(rows),\'argumentos\':arguments,\n        \'treino_alterado\':False,\'rotulos_alterados\':False,\'cobertura_alterada\':False,\n        \'explicacao\':\'A contagem é comparada após o mesmo pipeline TRAIN. O filtro padrão de caixas pode remover alvos pequenos/degenerados; a cobertura e os bytes originais continuam obrigatórios.\',\n        \'diferencas\':differences}\n    protocol.save_gz(output/\'referencia_alvos_pos_loader.json.gz\',rows)\n    core.save_json(output/\'diagnostico_alvos_pos_loader.json\',summary)\n    return {r[\'id\']:r for r in rows},summary\n\ndef audit_with_loader(results,plans,records,specs,data,arguments,output,frozen=None):\n    if frozen is not None:frozen_data.verify_frozen(frozen)\n    # Os registros do treino comum preservam os IDs, mas apontam para as cópias\n    # usadas de fato pelo loader. A auditoria usa essas cópias e confere os hashes.\n    common=list(data[\'records\'].values())\n    original={r[\'id\']:r for r in records}\n    if len(original)!=len(records) or len(common)!=len(records):raise ValueError(\'Manifesto de treino incompleto.\')\n    for r in common:\n        prior=original.get(r[\'id\'])\n        if prior is None or any(r[k]!=prior[k] for k in [\'sha256\',\'label_sha256\',\'known_mask\',\'source\']):\n            raise ValueError(\'Supervisão/cobertura comum alterada: \'+r[\'id\'])\n    reference,summary=loader_reference(common,data,arguments,output)\n    prior_counts=protocol.target_counts\n    try:\n        # Só muda a referência da auditoria. Nenhum loader de treino é executado\n        # pelo modelo e nenhuma checagem de máscara, gradiente ou orçamento é removida.\n        protocol.target_counts=lambda r:reference[r[\'id\']][\'processed_counts\']\n        result=protocol.audit_runs(results,plans,records,specs)\n    except Exception as exc:\n        divergences=[]\n        for arm,run in results.items():\n            for row in run[\'logs\']:\n                for j,iid in enumerate(row[\'ids\']):\n                    observed={str(k):v for k,v in row[\'gt_per_image_by_class\'][j].items()}\n                    expected=reference[iid]\n                    if observed!=expected[\'processed_counts\'] or row[\'known_masks\'][j]!=expected[\'known_mask\']:\n                        divergences.append({\'arm\':arm,\'step\':row[\'step\'],\'id\':iid,\'observed_counts\':observed,\n                            \'expected_counts\':expected[\'processed_counts\'],\'observed_mask\':row[\'known_masks\'][j],\n                            \'expected_mask\':expected[\'known_mask\']})\n        core.save_json(Path(output)/\'divergencias_nao_explicadas.json\',{\'erro\':str(exc),\'quantidade\':len(divergences),\'divergencias\':divergences})\n        raise ValueError(\'A divergência não foi explicada pelo processamento padrão; checagens preservadas. O diagnóstico foi salvo no pacote.\') from exc\n    finally:protocol.target_counts=prior_counts\n    result.update({\'auditoria_versao\':VERSION,\'alvos_comparados_pos_loader\':True,\n        \'treino_alterado_para_corrigir_auditoria\':False,\'imagens_com_alvos_filtrados\':summary[\'imagens_com_alvos_filtrados\'],\n        \'alvos_filtrados_unicos\':summary[\'alvos_filtrados_unicos\'],\'referencia_loader_sha256\':summary[\'reference_sha256\']})\n    core.save_json(Path(output)/\'auditoria_fontes.json\',result)\n    core.save_json(Path(output)/\'correcao_auditoria18.json\',{\'versao\':VERSION,\'concluida\':True,\n        \'checagens_mascara_gradiente_orcamento_preservadas\':True,\'treinamento_modificado\':False,\n        \'referencia\':\'referencia_alvos_pos_loader.json.gz\',\'diagnostico\':\'diagnostico_alvos_pos_loader.json\'})\n    print(\'Auditoria corrigida e conferida. Imagens com filtro padrão de alvos:\',summary[\'imagens_com_alvos_filtrados\'],flush=True)\n    return result\n\ndef recover(namespace,final_cell):\n    required=[\'PASTA_RESULTADOS\',\'PASTA_CODIGO\',\'BRACOS\',\'CRONOGRAMAS\',\'REGISTROS_TREINO\',\'SPECS\',\n        \'ARGUMENTOS_TREINO\',\'CONGELADOS\',\'CONTEXT_KEY\',\'INITIAL_STATE\',\'estado18\',\'registrar_falha\']\n    missing=[k for k in required if k not in namespace]\n    if missing:raise RuntimeError(\'Execute esta célula no notebook 18 original, na mesma sessão que terminou a etapa 7. Variáveis ausentes: \'+\', \'.join(missing))\n    output=Path(namespace[\'PASTA_RESULTADOS\'])\n    try:\n        results={}\n        for arm in protocol.ARMS:\n            result=namespace[\'estado18\'].load_completed(output,arm,namespace[\'CONTEXT_KEY\'],namespace[\'INITIAL_STATE\'],\n                namespace[\'SPECS\'][arm],namespace[\'CRONOGRAMAS\'][arm])\n            if result is None:raise RuntimeError(\'Não foi possível autenticar o braço completo \'+arm+\'. A recuperação foi interrompida sem iniciar outro treino.\')\n            results[arm]=result\n            print(\'Braço completo autenticado, sem repetir treino:\',arm,flush=True)\n        audit=audit_with_loader(results,namespace[\'CRONOGRAMAS\'],namespace[\'REGISTROS_TREINO\'],namespace[\'SPECS\'],\n            namespace[\'BRACOS\'][protocol.ARMS[0]],namespace[\'ARGUMENTOS_TREINO\'],output,namespace[\'CONGELADOS\'])\n        namespace[\'RESULTADOS_BRACOS\']=results;namespace[\'PAREAMENTO\']=audit\n    except Exception as exc:\n        namespace[\'registrar_falha\'](exc,\'correção da auditoria após loader\');raise\n    print(\'Gerando o relatório e o pacote final automaticamente...\',flush=True)\n    exec(compile(final_cell,\'etapa_8_original_preservada\',\'exec\'),namespace)\n'
_SHA_CORRECAO_18='bc7cd46e83265a3da81f4a35458c7ac825ee2d6a97efeca8f43938b9d7b648a6'
if hashlib.sha256(_FONTE_CORRECAO_18.encode()).hexdigest()!=_SHA_CORRECAO_18:
    raise RuntimeError('Código da correção da auditoria alterado.')
_destino_correcao=Path(PASTA_RESULTADOS)/'codigo/mix18_audit_fix.py'
_destino_correcao.parent.mkdir(parents=True,exist_ok=True)
_destino_correcao.write_text(_FONTE_CORRECAO_18,encoding='utf-8')
correcao18=types.ModuleType('mix18_audit_fix')
correcao18.__file__=str(_destino_correcao)
sys.modules['mix18_audit_fix']=correcao18
exec(compile(_FONTE_CORRECAO_18,str(_destino_correcao),'exec'),correcao18.__dict__)
core.save_json(Path(PASTA_RESULTADOS)/'codigo_correcao18.json',{
    'versao':correcao18.VERSION,'sha256':_SHA_CORRECAO_18,'contexto_treino_preservado':CONTEXT_KEY,
    'modulos_de_treino_modificados':False})

RESULTADOS_BRACOS={}
try:
    dados12.verify_frozen(CONGELADOS)
    for arm in protocolo18.ARMS:
        RESULTADOS_BRACOS[arm]=execucao18.run_arm(arm,BRACOS[arm],PESOS['candidato_10'],INITIAL_STATE,SPECS[arm],
            CRONOGRAMAS[arm],CONTEXT_KEY,COORTES,politica,avaliacao_base.corresponder_objetos,AMBIENTE,
            PASTA_RESULTADOS,PASTA_LOCAL,DEVICE,enviar_resultados,ARGUMENTOS_TREINO)
        progresso('Braço concluído: '+arm)
    PAREAMENTO=correcao18.audit_with_loader(RESULTADOS_BRACOS,CRONOGRAMAS,REGISTROS_TREINO,SPECS,BRACOS[protocolo18.ARMS[0]],ARGUMENTOS_TREINO,PASTA_RESULTADOS,CONGELADOS)
    core.save_json(PASTA_RESULTADOS/'auditoria_fontes.json',PAREAMENTO)
    print('Proporções e supervisão conferidas: 342 atualizações em cada braço.')
except Exception as exc:
    registrar_falha(exc,'treinos e avaliações');raise


## 8. Gerar a comparação e o pacote final automaticamente

Relata o efeito 8/8 menos 12/4 por coorte, classe e imagem. Julga o candidato
**E8_8_e03**, pré-fixado, pelos seis critérios históricos; avalia o controle
separadamente e mantém e01/e02 como diagnósticos. Cumprir os limites históricos
e melhorar sobre o controle são conclusões distintas.

Gera **pacote_experimento_18.zip**, relatório HTML, curvas e tempos por fase.
Os pesos padrão e caches completos ficam no Drive e na saída Kaggle, fora do ZIP
compacto. O modelo em uso não é substituído automaticamente. Resultados em o ZIP final.


In [ ]:
try:
    dados12.verify_frozen(CONGELADOS)
    if protocolo18.digest(json.loads(REGRA_PATH.read_text()))!=protocolo18.digest(REGRA):raise RuntimeError('Regra alterada em memória.')
    ENTRIES=[INICIAL];AVALIACOES={'candidato_10_inicial':REFERENCIAS['candidato_10']};ORCAMENTOS=[]
    for arm in protocolo18.ARMS:
        result=RESULTADOS_BRACOS[arm];ENTRIES+=result['entries'];AVALIACOES.update(result['evaluations']);ORCAMENTOS.append(result['budget'])
    if len(ENTRIES)!=7:raise RuntimeError('São necessários o inicial e os seis checkpoints completos.')
    core.save_json(PASTA_RESULTADOS/'checkpoints_avaliados.json',ENTRIES)
    core.save_json(PASTA_RESULTADOS/'orcamentos_realizados.json',ORCAMENTOS)
    RESUMO,SELECAO=relatorios18.report(PASTA_RESULTADOS,ENTRIES,AVALIACOES,REFERENCIAS,REGRA,PAREAMENTO,CONFIG,ORCAMENTOS)
    if SELECAO['eligible']:
        escolhido=SELECAO['eligible'];origem=PASTA_RESULTADOS/escolhido['weight_relative']
        destino=PASTA_RESULTADOS/'checkpoint_final_elegivel_validacao.pt';shutil.copy2(origem,destino)
        if core.sha256_file(destino)!=escolhido['weight_sha256']:raise RuntimeError('Cópia do peso elegível diverge.')
    else:
        (PASTA_RESULTADOS/'checkpoint_final_elegivel_validacao.pt').unlink(missing_ok=True)
    core.save_json(PASTA_RESULTADOS/'integridade_final.json',{'conferida':True,'auditoria_fontes_conferida':True,
        'supervisao_original_preservada':True,'projecao_treino_documentada':False,
        'validacao_rotulos_alterados':False,'test_home_inferido':False,'test_oi20k_inferido':False,
        'reserva_futura_extraida':False,'reserva_futura_inferida':False,'modelo_substituido':False})
    erro=PASTA_RESULTADOS/'execucao_interrompida.json'
    if erro.is_file():
        historico=PASTA_RESULTADOS/'historico_falhas';historico.mkdir(exist_ok=True)
        erro.replace(historico/('falha_resolvida_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')+'.json'))
    progresso('Relatório e integridade concluídos')
    PACOTE_FINAL=estado18.package(PASTA_RESULTADOS)
    enviar_resultados('resultado final do experimento 18')
    print('NOTEBOOK 18 CONCLUÍDO — dois braços pareados, avaliações e relatório salvos.')
    print(RESUMO['mensagem'])
    print('Arquivo de resultados:',PACOTE_FINAL)
    display(FileLink(str(PACOTE_FINAL)))
    display(FileLink(str(PASTA_RESULTADOS/'relatorio_experimento_18.html')))
except Exception as exc:
    registrar_falha(exc,'relatório e pacote final');raise
